# Block 14 → Block 18: Konvergenz und Dämpfung

**A100 → Alle ausführen.** Im kleinen Starter das bereitgestellte
**b14_b18_refine_paket.zip** auswählen. Kein Drive und kein weiterer Upload.

Vier festgelegte Fälle des vorigen Laufs werden mit genauer begrenzter
Konvergenz- und Dämpfungsprüfung untersucht. Originales Qwen3-1.7B, FP32,
ausgeschaltetes TF32, dieselben 286 Koordinaten, REF-W-Werte und Quellprompts
bleiben erhalten. Die alten Resultate werden geprüft und unverändert bewahrt.
Es gibt keine neuen Fragen, freien Antworten oder Anpassung der Gewichte.

Der Versuch misst lokalen, promptgebundenen Transport. Ein ausgeführter Solver
bedeutet keine bestätigte Konvergenz oder gelungene Rekonstruktion; diese Angaben
werden getrennt gespeichert. Es ist keine globale Jacobian-Lens und kein
Gedächtnisnachweis.

GPU-Torch bleibt installiert; Modellgewichte im vorhandenen Cache werden
wiederverwendet. Das Rechenbudget beträgt 20 Minuten mit sicheren Checkpoints;
Installation und Modellladen kommen hinzu. Das ist keine Laufzeitgarantie.
Ein Status `paused` bedeutet einen fortsetzbaren Teilstand.

Am Ende **b14_b18_refine_results.zip** als kompakte Auswertung herunterladen
und hier bereitstellen. **b14_b18_refine_results_RAW.zip** zusätzlich sichern;
nur die Rohdatei enthält vollständige Logits und dient zur Wiederaufnahme.
Bei Fehlern wird ein Diagnose-ZIP angeboten. Das private Paket nicht öffentlich
hochladen. Die GPU-Sitzung danach beenden, wenn sie nicht mehr gebraucht wird.


In [ ]:
# Private Nutzdatei: unverändert lassen und Alle ausführen.
# Öffentliches Template: optional vorhandene Originaldateien in diesem Verzeichnis.
REFERENCE_DIRECTORY = ""
RESUME_ZIP = ""  # Nur für einen vorhandenen Checkpoint von einer anderen Sitzung.
RUN_DIRECTORY = ""  # Leer: automatisch, wiederholbar in derselben Sitzung.
MIN_FREE_VRAM_GIB = 12.0


In [ ]:
# Eingebettete Quellen und optionale private Originalbytes; nicht manuell ändern.
TRANSFORMERS_VERSION = '5.22.2'
ACCELERATE_VERSION = '1.16.0'
REFERENCE_SHA256 = {'hn_set.json': '11390233b62c63b9d34805482ecbd3675eadd80570b9c01ef9e268bf565c98fd', 'htq_policy.json': '160c420603b719afba4b909d12d31316535290e13b00aea2a6dea88110684f7c', 'refw_h286.json': '68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc', 'b14_b18_items.json': 'cc482e87fc49dcea8fa77b0ccb4bf63ed8df41e5f75c5c47c434decd4e3d1a5e', 'b14_b18_plan.json': '76dcc81a06142b2fb662383ec82a6dc17834e9612543163036d2a7c278643bbf', 'b14_b18_refine_plan.json': 'ff4cd84d3ff1e01e87074d5c2938d79224c83634fa441a095a70c7d1d9b78afb', 'b14_b18_refine_anchors.json': 'd56873915a388bcf9c42e606a64f512d02098bf1747e45306196938e9284b2c7'}
REFERENCE_FILES_B64 = {}
MODULE_SOURCES = {'attention_probe/__init__.py': '"""Isolated, optional attention-pulse pilot; existing experiment code is unchanged."""\n', 'attention_probe/reference.py': '"""Bind a new attention pilot to the private, exact DSSE H_tq selection.\n\nThis module validates identities and provenance only. It imports no model runtime,\ncontains no private neuron coordinates, and does not map them to another model.\n\n    python -m attention_probe.reference --hn-set /private/hn_set.json \\\n        --policy /private/htq_policy.json --out /private/reference.json\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport tempfile\nfrom typing import Any\n\n\nHN_SET_SHA256 = "11390233b62c63b9d34805482ecbd3675eadd80570b9c01ef9e268bf565c98fd"\nPOLICY_SHA256 = "160c420603b719afba4b909d12d31316535290e13b00aea2a6dea88110684f7c"\nMODEL_ID = "qwen3-1.7b"\nMODEL_REPO = "Qwen/Qwen3-1.7B"\nMODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"\nN_COORDINATES, N_LAYERS, MLP_WIDTH = 286, 28, 6144\nH_POLICY = "htq_v2"\n\n\nclass ReferenceError(ValueError):\n    """The supplied files do not establish the required DSSE reference."""\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelPins:\n    repo: str\n    revision: str\n    sha256: tuple[tuple[str, str], ...]\n    model_id: str = MODEL_ID\n    n_layers: int = N_LAYERS\n    mlp_width: int = MLP_WIDTH\n\n    def to_dict(self) -> dict[str, Any]:\n        return dict(model_id=self.model_id, repo=self.repo, revision=self.revision,\n                    sha256=dict(self.sha256), n_layers=self.n_layers, mlp_width=self.mlp_width)\n\n\n@dataclass(frozen=True, slots=True)\nclass Reference:\n    coordinates: tuple[tuple[int, int], ...]\n    hn_sha256: str\n    policy_sha256: str\n    model: ModelPins\n    h_policy: str = H_POLICY\n    post_hoc: bool = True\n    amendment: int = 2\n\n    def to_dict(self) -> dict[str, Any]:\n        """Return a fresh JSON binding; its coordinates remain private run data."""\n        return dict(schema="dsse_h286_reference_v1", coordinates=[list(x) for x in self.coordinates],\n                    coordinate_order="hn_set.H_op (identical to H_tq)",\n                    n_coordinates=len(self.coordinates),\n                    source_sha256={"hn_set.json": self.hn_sha256, "htq_policy.json": self.policy_sha256},\n                    model=self.model.to_dict(), h_policy=self.h_policy, post_hoc=self.post_hoc,\n                    amendment=self.amendment, confirmatory_eligible=False,\n                    scope="Neuron identities only; no CETT equivalence or cross-model coordinate mapping.")\n\n\ndef _digest(raw: bytes) -> str:\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef _sha(value: Any, label: str) -> str:\n    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:\n        raise ReferenceError(f"{label} must be a lowercase SHA-256 digest.")\n    return value\n\n\ndef _object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:\n    out = {}\n    for key, value in pairs:\n        if key in out:\n            raise ReferenceError("Duplicate JSON object key.")\n        out[key] = value\n    return out\n\n\ndef _reject_constant(value: str) -> Any:\n    raise ReferenceError("Non-finite JSON constants are not allowed.")\n\n\ndef _document(raw: bytes, label: str) -> dict[str, Any]:\n    try:\n        value = json.loads(raw, object_pairs_hook=_object, parse_constant=_reject_constant)\n    except (UnicodeDecodeError, json.JSONDecodeError) as exc:\n        raise ReferenceError(f"{label} is not valid JSON.") from exc\n    if not isinstance(value, dict):\n        raise ReferenceError(f"{label} must be a JSON object.")\n    return value\n\n\ndef _coordinates(value: Any, label: str) -> tuple[tuple[int, int], ...]:\n    if not isinstance(value, list) or len(value) != N_COORDINATES:\n        raise ReferenceError(f"{label} must contain exactly {N_COORDINATES} coordinates.")\n    result = []\n    for i, coordinate in enumerate(value):\n        if not isinstance(coordinate, list) or len(coordinate) != 2 or any(type(x) is not int for x in coordinate):\n            raise ReferenceError(f"{label}[{i}] must be a pair of integer coordinates.")\n        layer, index = coordinate\n        if not (0 <= layer < N_LAYERS and 0 <= index < MLP_WIDTH):\n            raise ReferenceError(f"{label}[{i}] is outside the 28 by 6144 MLP architecture.")\n        result.append((layer, index))\n    if len(set(result)) != N_COORDINATES:\n        raise ReferenceError(f"{label} contains duplicate coordinates.")\n    return tuple(result)\n\n\ndef _policy(document: dict[str, Any], label: str) -> None:\n    if document.get("h_policy") != H_POLICY or document.get("post_hoc") is not True:\n        raise ReferenceError(f"{label} must identify the post-hoc htq_v2 policy.")\n    if type(document.get("amendment")) is not int or document["amendment"] != 2:\n        raise ReferenceError(f"{label} must identify amendment 2.")\n    if document.get("H_op_name") != "H_tq":\n        raise ReferenceError(f"{label} must use H_tq as the operative set.")\n\n\ndef _model(document: dict[str, Any], label: str) -> ModelPins:\n    provenance = document.get("provenance")\n    if not isinstance(provenance, dict) or not isinstance(provenance.get("models"), dict):\n        raise ReferenceError(f"{label} has no model provenance.")\n    for field, expected in (("h_policy", H_POLICY), ("post_hoc", True), ("amendment", 2), ("mock", False)):\n        if field in provenance and (type(provenance[field]) is not type(expected) or provenance[field] != expected):\n            raise ReferenceError(f"{label} has conflicting {field} provenance.")\n    model = provenance["models"].get(MODEL_ID)\n    if not isinstance(model, dict) or model.get("repo") != MODEL_REPO or model.get("revision") != MODEL_REVISION:\n        raise ReferenceError(f"{label} does not identify the pinned Qwen3-1.7B model.")\n    hashes = model.get("sha256")\n    if not isinstance(hashes, dict) or not hashes or any(not isinstance(k, str) or not k for k in hashes):\n        raise ReferenceError(f"{label} has no model-file SHA-256 pins.")\n    pins = tuple(sorted((name, _sha(digest, f"{label} model file")) for name, digest in hashes.items()))\n    return ModelPins(repo=model["repo"], revision=model["revision"], sha256=pins)\n\n\ndef verify_documents(hn_bytes: bytes, policy_bytes: bytes, *, expected_hn_sha256: str,\n                     expected_policy_sha256: str) -> Reference:\n    """Low-level verification with explicit trust anchors, including synthetic tests.\n\n    Normal pilot code must call load_reference(), whose anchors are the production\n    hashes. Checksums cover original bytes, so JSON reformatting is not equivalent.\n    """\n    expected_hn_sha256 = _sha(expected_hn_sha256, "Expected hn_set hash")\n    expected_policy_sha256 = _sha(expected_policy_sha256, "Expected policy hash")\n    if not isinstance(hn_bytes, bytes) or not isinstance(policy_bytes, bytes):\n        raise ReferenceError("Reference documents must be immutable bytes.")\n    hn_digest, policy_digest = _digest(hn_bytes), _digest(policy_bytes)\n    if hn_digest != expected_hn_sha256:\n        raise ReferenceError("hn_set.json SHA-256 mismatch.")\n    if policy_digest != expected_policy_sha256:\n        raise ReferenceError("htq_policy.json SHA-256 mismatch.")\n    hn, policy = _document(hn_bytes, "hn_set.json"), _document(policy_bytes, "htq_policy.json")\n    _policy(hn, "hn_set.json"); _policy(policy, "htq_policy.json")\n    coordinates = _coordinates(hn.get("H_op"), "H_op")\n    if coordinates != _coordinates(hn.get("H_tq"), "H_tq"):\n        raise ReferenceError("H_op and H_tq must be identical, including coordinate order.")\n    if type(hn.get("n_H")) is not int or hn["n_H"] != N_COORDINATES:\n        raise ReferenceError("hn_set n_H must equal 286.")\n    if type(policy.get("n_H_op")) is not int or policy["n_H_op"] != N_COORDINATES:\n        raise ReferenceError("Policy n_H_op must equal 286.")\n    hn_model, policy_model = _model(hn, "hn_set.json"), _model(policy, "htq_policy.json")\n    if hn_model != policy_model:\n        raise ReferenceError("The two files have different model-file pins.")\n    return Reference(coordinates=coordinates, hn_sha256=hn_digest,\n                     policy_sha256=policy_digest, model=hn_model)\n\n\ndef load_reference(hn_set_path: str | Path, policy_path: str | Path) -> Reference:\n    """Load only the exact DSSE production files; no caller-supplied hash bypass."""\n    return verify_documents(Path(hn_set_path).read_bytes(), Path(policy_path).read_bytes(),\n                            expected_hn_sha256=HN_SET_SHA256, expected_policy_sha256=POLICY_SHA256)\n\n\ndef write_reference(reference: Reference, out: str | Path) -> None:\n    """Atomically write a binding to the caller\'s explicit private output path."""\n    path = Path(out)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    payload = json.dumps(reference.to_dict(), indent=2, ensure_ascii=False, allow_nan=False) + "\\n"\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,\n                                         prefix="." + path.name + ".", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            handle.write(payload); handle.flush(); os.fsync(handle.fileno())\n        os.replace(temporary, path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--hn-set", required=True, type=Path)\n    parser.add_argument("--policy", required=True, type=Path)\n    parser.add_argument("--out", required=True, type=Path)\n    args = parser.parse_args(argv)\n    if args.out.resolve() in (args.hn_set.resolve(), args.policy.resolve()):\n        parser.error("--out must not overwrite a source document.")\n    try:\n        reference = load_reference(args.hn_set, args.policy)\n        write_reference(reference, args.out)\n    except (OSError, ReferenceError) as exc:\n        parser.error(str(exc))\n    print(f"Validated {len(reference.coordinates)} H_tq coordinates; private binding written to {args.out}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'attention_probe/fixed_refw.py': '"""Validate the private, fixed H286 REF-W extract without model imports.\n\nThe compact file contains existing population means/SDs, already cast to FP32\nas in DSSE\'s SetReader. This module neither fits nor recomputes normalization.\nCETT uses max(||down_proj(z)||, CETT_EPS); zCETT uses the saved c_mu and\nmax(c_sd, STANDARDIZATION_EPS). Zero saved SD is valid and stays zero here:\nthe consuming measurement applies the documented denominator floor.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nimport struct\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .reference import (\n    HN_SET_SHA256,\n    MLP_WIDTH,\n    N_COORDINATES,\n    N_LAYERS,\n    POLICY_SHA256,\n    Reference,\n    ReferenceError,\n)\n\nFIXED_REFW_SHA256 = "68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc"\nSOURCE_HN_META_SHA256 = "9c61bc34e804719e5bf5e72b644e8a38a94ad0f6c1708efaef75386acdae49ff"\nCETT_EPS = 1e-9\nSTANDARDIZATION_EPS = 1e-6\nSCHEMA = "dsse_h286_refw_v1"\n_VECTOR_NAMES = {"z_mu", "z_sd", "c_mu", "c_sd"}\n_FIELDS = {"schema", "source_hn_meta_sha256", "hn_sha256", "coordinates", "refw"}\n\n\nclass FixedRefWError(ReferenceError):\n    """The supplied extract does not establish the fixed normalization."""\n\n\ndef _sha(value: Any, label: str) -> str:\n    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:\n        raise FixedRefWError(f"{label} must be a lowercase SHA-256 digest.")\n    return value\n\n\ndef _object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:\n    result = {}\n    for key, value in pairs:\n        if key in result:\n            raise FixedRefWError("Duplicate JSON object key.")\n        result[key] = value\n    return result\n\n\ndef _reject_constant(value: str) -> Any:\n    raise FixedRefWError(f"Non-finite JSON constant: {value}")\n\n\ndef verify_fixed_refw(\n    raw: bytes,\n    reference: Reference,\n    *,\n    expected_sha256: str,\n    expected_source_hn_meta_sha256: str,\n    expected_hn_sha256: str,\n) -> dict[str, Any]:\n    """Low-level validation with explicit trust anchors for synthetic tests.\n\n    Production callers must use load_fixed_refw(), which fixes all three\n    anchors. The source hash binds the previously verified full hn_meta file;\n    this compact loader does not claim to inspect that large file again.\n    """\n    expected_sha256 = _sha(expected_sha256, "Expected compact hash")\n    expected_source_hn_meta_sha256 = _sha(expected_source_hn_meta_sha256, "Expected source hash")\n    expected_hn_sha256 = _sha(expected_hn_sha256, "Expected H selection hash")\n    if not isinstance(raw, bytes):\n        raise FixedRefWError("The compact document must be immutable bytes.")\n    if hashlib.sha256(raw).hexdigest() != expected_sha256:\n        raise FixedRefWError("Fixed REF-W SHA-256 mismatch.")\n    if not isinstance(reference, Reference) or reference.hn_sha256 != expected_hn_sha256:\n        raise FixedRefWError("Reference has a different H selection hash.")\n    try:\n        document = json.loads(raw, object_pairs_hook=_object, parse_constant=_reject_constant)\n    except (UnicodeDecodeError, json.JSONDecodeError) as exc:\n        raise FixedRefWError("Fixed REF-W is not valid JSON.") from exc\n    if not isinstance(document, dict) or set(document) != _FIELDS:\n        raise FixedRefWError("Unexpected fixed REF-W document fields.")\n    if document["schema"] != SCHEMA:\n        raise FixedRefWError("Unsupported fixed REF-W schema.")\n    if document["source_hn_meta_sha256"] != expected_source_hn_meta_sha256:\n        raise FixedRefWError("Full hn_meta source hash mismatch.")\n    if document["hn_sha256"] != expected_hn_sha256:\n        raise FixedRefWError("Compact H selection hash mismatch.")\n\n    coordinates = document["coordinates"]\n    if not isinstance(coordinates, list) or len(coordinates) != N_COORDINATES:\n        raise FixedRefWError("Expected exactly 286 ordered coordinates.")\n    checked = []\n    for coordinate in coordinates:\n        if (not isinstance(coordinate, list) or len(coordinate) != 2\n                or any(type(value) is not int for value in coordinate)):\n            raise FixedRefWError("Coordinates must be integer pairs.")\n        layer, neuron = coordinate\n        if not (0 <= layer < N_LAYERS and 0 <= neuron < MLP_WIDTH):\n            raise FixedRefWError("Coordinate outside the Qwen3-1.7B MLP architecture.")\n        checked.append((layer, neuron))\n    if len(set(checked)) != N_COORDINATES or tuple(checked) != reference.coordinates:\n        raise FixedRefWError("Coordinates differ from the ordered H286 reference.")\n\n    vectors = document["refw"]\n    if not isinstance(vectors, dict) or set(vectors) != _VECTOR_NAMES:\n        raise FixedRefWError("Expected exactly z_mu, z_sd, c_mu and c_sd.")\n    for name, vector in vectors.items():\n        if not isinstance(vector, list) or len(vector) != N_COORDINATES:\n            raise FixedRefWError(f"{name} must have exactly 286 values.")\n        for value in vector:\n            if type(value) not in (int, float):\n                raise FixedRefWError(f"{name} must contain finite FP32 numbers.")\n            try:\n                fp32 = struct.unpack("<f", struct.pack("<f", value))[0]\n            except (OverflowError, struct.error) as exc:\n                raise FixedRefWError(f"{name} contains a value outside FP32.") from exc\n            if not math.isfinite(fp32) or fp32 != value:\n                raise FixedRefWError(f"{name} contains a non-finite or non-FP32 value.")\n            if name.endswith("_sd") and value < 0:\n                raise FixedRefWError(f"{name} cannot contain negative standard deviations.")\n    return document\n\n\ndef load_fixed_refw(path: str | Path, reference: Reference) -> dict[str, Any]:\n    """Load only the exact production extract bound to the original reference."""\n    if not isinstance(reference, Reference) or reference.policy_sha256 != POLICY_SHA256:\n        raise FixedRefWError("Expected the original production reference policy.")\n    return verify_fixed_refw(\n        Path(path).read_bytes(), reference,\n        expected_sha256=FIXED_REFW_SHA256,\n        expected_source_hn_meta_sha256=SOURCE_HN_META_SHA256,\n        expected_hn_sha256=HN_SET_SHA256,\n    )\n', 'attention_probe/interventions.py': '"""One-pass attention pulses and read-only activation capture for Qwen3.\n\nPositions are zero-based columns of the complete input sequence, not relative\nindices of a cached decode chunk. Selected positions must be real tokens in\nevery batch row. Qwen3 queries are scaled *after* q_norm and before RoPE, so\nnormalization cannot cancel the pulse. Scaling queries scales their attention\nlogits; it makes no claim about FLOPs, energy, or a computational bottleneck.\n\nCoordinateCapture stores raw gated-MLP z at down_proj input, not CETT.\n"""\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nimport math\nimport inspect\nfrom numbers import Integral\nfrom typing import Iterable, Sequence\n\nimport torch\n\n\ndef _integers(values: Iterable[int], name: str, *, nonempty: bool = True) -> tuple[int, ...]:\n    result = tuple(values)\n    if nonempty and not result:\n        raise ValueError(f"{name} must not be empty")\n    if any(isinstance(x, bool) or not isinstance(x, Integral) or x < 0 for x in result):\n        raise ValueError(f"{name} must contain nonnegative integers")\n    if len(set(result)) != len(result):\n        raise ValueError(f"{name} must not contain duplicates")\n    return tuple(int(x) for x in result)\n\n\ndef _layers(model):\n    base = getattr(model, "model", model)\n    layers = getattr(base, "layers", None)\n    if layers is None or not len(layers):\n        raise ValueError("Expected Hugging Face decoder model.model.layers (or model.layers)")\n    return layers\n\n\ndef _cache_length(cache) -> int | None:\n    if cache is None:\n        return 0\n    getter = getattr(cache, "get_seq_length", None)\n    if getter is not None:\n        try:\n            return int(getter())\n        except (TypeError, ValueError, RuntimeError):\n            return None\n    if isinstance(cache, (tuple, list)):\n        if not cache:\n            return 0\n        try:\n            return int(cache[0][0].shape[-2])\n        except (IndexError, AttributeError, TypeError):\n            return None\n    return None\n\n\nclass _OneFullPass:\n    """Bind hooks to one complete, uncached outer model forward and clean them up."""\n\n    def __init__(self, model, positions: Sequence[int]):\n        self.model = model\n        self.positions = _integers(positions, "positions")\n        self._handles = []\n        self._entered = False\n        self._used = False\n        self._claimed = False\n        self._window = None\n        self.full_passes = 0\n        self.skipped_cached_passes = 0\n\n    def _model_pre(self, module, args, kwargs):\n        self._window = None\n        if self._claimed:\n            return\n        # HF forwards allow both positional and keyword cache arguments.\n        try:\n            named = inspect.signature(module.forward).bind_partial(*args, **kwargs).arguments\n        except (TypeError, ValueError):\n            named = kwargs\n        fields = dict(kwargs)\n        fields.update({name: value for name, value in named.items() if name not in ("args", "kwargs")})\n        cache = fields.get("past_key_values", fields.get("past_key_value"))\n        length = _cache_length(cache)\n        cache_position = fields.get("cache_position")\n        nonzero_offset = (cache_position is not None and torch.as_tensor(cache_position).numel()\n                          and int(torch.as_tensor(cache_position).reshape(-1)[0]) != 0)\n        if length is None or length > 0 or nonzero_offset:\n            self.skipped_cached_passes += 1\n            return\n        ids = fields.get("input_ids")\n        embeds = fields.get("inputs_embeds")\n        value = ids if ids is not None else embeds\n        if value is None and args:\n            value = args[0]\n        if not isinstance(value, torch.Tensor) or value.ndim not in (2, 3):\n            raise ValueError("Cannot identify full input sequence for absolute token positions")\n        batch, tokens = value.shape[:2]\n        if max(self.positions) >= tokens:\n            raise ValueError(f"position {max(self.positions)} is outside the full sequence of {tokens} tokens")\n        mask = fields.get("attention_mask")\n        if mask is not None:\n            if tuple(mask.shape) != (batch, tokens):\n                raise ValueError("Expected a two-dimensional outer attention mask")\n            if not bool(mask[:, list(self.positions)].bool().all()):\n                raise ValueError("Selected absolute position is padding in at least one batch row")\n        self._claimed = True\n        self.full_passes += 1\n        self._window = (int(batch), int(tokens))\n\n    def _model_post(self, module, args, kwargs, output):\n        self._window = None\n\n    def _install(self):\n        raise NotImplementedError\n\n    def validate_fired(self):\n        raise NotImplementedError\n\n    def __enter__(self):\n        if self._used:\n            raise RuntimeError("Use a fresh hook context for each attempt")\n        if self.model.training:\n            raise ValueError("Attention pilot requires model.eval(); model mode is never changed by hooks")\n        self._used = self._entered = True\n        try:\n            self._handles.append(self.model.register_forward_pre_hook(self._model_pre, with_kwargs=True))\n            self._handles.append(self.model.register_forward_hook(self._model_post, with_kwargs=True, always_call=True))\n            self._install()\n        except BaseException:\n            self.close()\n            raise\n        return self\n\n    def close(self):\n        for handle in reversed(self._handles):\n            handle.remove()\n        self._handles.clear()\n        self._window = None\n        self._entered = False\n\n    def __exit__(self, kind, error, tb):\n        self.close()\n        if kind is None:\n            self.validate_fired()\n        return False\n\n\nclass AttentionPulse(_OneFullPass):\n    """Scale post-q_norm queries, or add a per-head matched random delta, once.\n\n    Primary architecture is config.model_type=\'qwen3\'. Qwen2\'s unnormalized\n    q_proj output is supported only with explicit architecture=\'qwen2\'. Random\n    deltas match ||(beta-1) q|| independently for each selected token/query head\n    before output-dtype rounding. actual_delta_norms reports rounding effects.\n    A local CPU generator never reads or changes PyTorch\'s global RNG state.\n    """\n\n    def __init__(self, model, block: int, positions: Sequence[int], beta: float = 1.0,\n                 mode: str = "scale", seed: int = 0, *, architecture: str = "qwen3"):\n        super().__init__(model, positions)\n        if not isinstance(beta, (int, float)) or not math.isfinite(beta) or not 0 <= beta <= 1:\n            raise ValueError("beta must be finite and between zero and one")\n        if mode not in ("scale", "random"):\n            raise ValueError("mode must be \'scale\' or \'random\'")\n        if isinstance(seed, bool) or not isinstance(seed, Integral) or seed < 0:\n            raise ValueError("seed must be a nonnegative integer")\n        self.layers = _layers(model)\n        self.block = _integers((block,), "block")[0]\n        if self.block >= len(self.layers):\n            raise ValueError("block is outside the decoder")\n        if architecture not in ("qwen3", "qwen2") or getattr(model.config, "model_type", None) != architecture:\n            raise ValueError("Unsupported architecture: Qwen3 is primary; Qwen2 requires explicit opt-in")\n        self.architecture, self.mode, self.beta = architecture, mode, float(beta)\n        self.seed = int(seed)\n        attention = self.layers[self.block].self_attn\n        self.heads = int(model.config.num_attention_heads)\n        self.head_dim = int(getattr(attention, "head_dim", getattr(model.config, "head_dim", 0)))\n        if self.heads <= 0 or self.head_dim <= 0:\n            raise ValueError("Query-head shape is not declared")\n        if architecture == "qwen3":\n            self.hook_module = getattr(attention, "q_norm", None)\n            self.hook_location = "self_attn.q_norm.output (B,T,H,D), before RoPE"\n            if self.hook_module is None:\n                raise ValueError("Qwen3 requires post-q_norm intervention; q_proj fallback is forbidden")\n        else:\n            if getattr(attention, "q_norm", None) is not None:\n                raise ValueError("Qwen2 fallback expects no query normalization")\n            self.hook_module = getattr(attention, "q_proj", None)\n            self.hook_location = "self_attn.q_proj.output, explicit Qwen2 fallback"\n            if self.hook_module is None:\n                raise ValueError("Qwen2 query projection is absent")\n        self.fired_count = 0\n        self.requested_delta_norms = None\n        self.actual_delta_norms = None\n        self._generator = torch.Generator(device="cpu").manual_seed(self.seed)\n\n    def _install(self):\n        self._handles.append(self.hook_module.register_forward_hook(self._pulse))\n\n    def _pulse(self, module, args, output):\n        if self._window is None or self.fired_count:\n            return None\n        batch, tokens = self._window\n        if not isinstance(output, torch.Tensor):\n            raise ValueError("Query hook did not return a tensor")\n        shape = (batch, tokens, self.heads, self.head_dim)\n        if self.architecture == "qwen3":\n            if tuple(output.shape) != shape:\n                raise ValueError(f"Unsupported Qwen3 post-q_norm layout: expected {shape}, got {tuple(output.shape)}")\n            query = output\n        else:\n            if tuple(output.shape) != (batch, tokens, self.heads * self.head_dim):\n                raise ValueError("Unsupported Qwen2 q_proj output shape")\n            query = output.reshape(shape)\n        selected = query[:, list(self.positions), :, :]\n        if not bool(torch.isfinite(selected).all()):\n            raise ValueError("Nonfinite query in selected pulse positions")\n        self.fired_count += 1\n        norms = (1-self.beta) * selected.detach().double().cpu().norm(dim=-1)\n        self.requested_delta_norms = norms.clone()\n        if self.beta == 1:\n            self.actual_delta_norms = torch.zeros_like(norms)\n            return None  # Exact sham: preserve the original tensor, no multiplication or RNG draw.\n        if self.mode == "scale":\n            changed = selected * self.beta\n        else:\n            random = torch.randn(selected.shape, generator=self._generator, dtype=torch.float64, device="cpu")\n            random = random / random.norm(dim=-1, keepdim=True).clamp_min(torch.finfo(torch.float64).tiny)\n            delta = (random * norms[..., None]).to(device=selected.device, dtype=selected.dtype)\n            changed = selected + delta\n        self.actual_delta_norms = (changed.detach().double().cpu()-selected.detach().double().cpu()).norm(dim=-1)\n        result = query.clone()\n        result[:, list(self.positions), :, :] = changed\n        return result if self.architecture == "qwen3" else result.reshape(output.shape)\n\n    def validate_fired(self):\n        if self.fired_count != 1:\n            raise RuntimeError(f"Attention pulse must fire once on a full/prefill pass; fired {self.fired_count}")\n        return self.fired_count\n\n\nclass CoordinateCapture(_OneFullPass):\n    """Read exact ordered down_proj-input coordinates and decoder residuals.\n\n    values is CPU float32 [batch, positions, coordinates]; coordinate order is\n    exactly the supplied order. residuals maps each requested decoder layer to\n    CPU float32 [batch, positions, hidden_size]. No input/output tensor changes.\n    This is raw z capture, not CETT or a causal-neuron importance estimate.\n    """\n\n    feature_kind = "raw gated-MLP activation z at down_proj input; not CETT"\n\n    def __init__(self, model, coordinates: Sequence[tuple[int, int]], positions: Sequence[int],\n                 residual_layers: Sequence[int] | None = None, *, expected_count: int = 286):\n        super().__init__(model, positions)\n        self.layers = _layers(model)\n        if getattr(model.config, "model_type", None) != "qwen3":\n            raise ValueError("Original H coordinates require the Qwen3 decoder architecture")\n        if isinstance(expected_count, bool) or not isinstance(expected_count, Integral) or expected_count < 1:\n            raise ValueError("expected_count must be a positive integer")\n        self.coordinates = tuple(tuple(pair) for pair in coordinates)\n        if len(self.coordinates) != expected_count or not self.coordinates:\n            raise ValueError(f"Expected exactly {expected_count} ordered H coordinates")\n        if len(set(self.coordinates)) != len(self.coordinates):\n            raise ValueError("Duplicate H coordinates")\n        self._by_layer = defaultdict(list)\n        for output_index, pair in enumerate(self.coordinates):\n            if len(pair) != 2:\n                raise ValueError("Each coordinate must be (block, down_proj_input_index)")\n            # A coordinate may have equal block/channel numbers; duplicates are\n            # forbidden across coordinate pairs, not within a pair.\n            if any(isinstance(x, bool) or not isinstance(x, Integral) or x < 0 for x in pair):\n                raise ValueError("Coordinate indices must be nonnegative integers")\n            block, neuron = (int(x) for x in pair)\n            if block >= len(self.layers):\n                raise ValueError("Coordinate block is outside the decoder")\n            down = getattr(getattr(self.layers[block], "mlp", None), "down_proj", None)\n            if down is None or not hasattr(down, "in_features") or neuron >= down.in_features:\n                raise ValueError("Coordinate channel is outside mlp.down_proj input")\n            self._by_layer[block].append((output_index, neuron))\n        self.residual_layers = _integers(residual_layers if residual_layers is not None else sorted(self._by_layer),\n                                        "residual_layers", nonempty=False)\n        if any(block >= len(self.layers) for block in self.residual_layers):\n            raise ValueError("Residual layer is outside the decoder")\n        self._values = None\n        self.residuals = {}\n        self._seen = set()\n        self.fired_count = 0\n\n    def _install(self):\n        for block in self._by_layer:\n            self._handles.append(self.layers[block].mlp.down_proj.register_forward_pre_hook(self._coordinate_hook(block)))\n        for block in self.residual_layers:\n            self._handles.append(self.layers[block].register_forward_hook(self._residual_hook(block)))\n\n    def _coordinate_hook(self, block):\n        def hook(module, args):\n            if self._window is None:\n                return None\n            if block in self._seen:\n                raise RuntimeError("down_proj was called twice in the selected full pass")\n            z = args[0]\n            batch, tokens = self._window\n            if not isinstance(z, torch.Tensor) or z.ndim != 3 or tuple(z.shape[:2]) != (batch, tokens):\n                raise ValueError("Unsupported down_proj input shape")\n            pairs = self._by_layer[block]\n            observed = z[:, list(self.positions), :][:, :, [channel for _, channel in pairs]].detach().float().cpu().clone()\n            if not bool(torch.isfinite(observed).all()):\n                raise ValueError("Nonfinite selected raw activation")\n            if self._values is None:\n                self._values = torch.empty((batch, len(self.positions), len(self.coordinates)), dtype=torch.float32)\n            self._values[:, :, [index for index, _ in pairs]] = observed\n            self._seen.add(block)\n            self.fired_count = int(self._seen == set(self._by_layer))\n            return None\n        return hook\n\n    def _residual_hook(self, block):\n        def hook(module, args, output):\n            if self._window is None:\n                return None\n            if block in self.residuals:\n                raise RuntimeError("Decoder layer was called twice in the selected full pass")\n            value = output[0] if isinstance(output, (tuple, list)) else output\n            if not isinstance(value, torch.Tensor) or value.ndim != 3 or tuple(value.shape[:2]) != self._window:\n                raise ValueError("Unsupported decoder residual output shape")\n            observed = value[:, list(self.positions)].detach().float().cpu().clone()\n            if not bool(torch.isfinite(observed).all()):\n                raise ValueError("Nonfinite selected decoder residual")\n            self.residuals[block] = observed\n            return None\n        return hook\n\n    @property\n    def values(self):\n        self.validate_fired()\n        return self._values\n\n    def validate_fired(self):\n        if self.fired_count != 1 or self._seen != set(self._by_layer) or set(self.residuals) != set(self.residual_layers):\n            raise RuntimeError("Raw-coordinate capture did not complete exactly one full/prefill pass")\n        return self.fired_count\n', 'attention_probe/pilot_items.json': '[\n  {\n    "id": "p01_correct_draft",\n    "pair_id": "p01",\n    "prompt": "7 + 8 = ?. Draft: 15. Give only the correct number.",\n    "target": "15",\n    "category": "arithmetic_add"\n  },\n  {\n    "id": "p01_wrong_draft",\n    "pair_id": "p01",\n    "prompt": "7 + 8 = ?. Draft: 14. Give only the correct number.",\n    "target": "15",\n    "category": "arithmetic_add"\n  },\n  {\n    "id": "p02_correct_draft",\n    "pair_id": "p02",\n    "prompt": "9 - 4 = ?. Draft: 5. Give only the correct number.",\n    "target": "5",\n    "category": "arithmetic_subtract"\n  },\n  {\n    "id": "p02_wrong_draft",\n    "pair_id": "p02",\n    "prompt": "9 - 4 = ?. Draft: 6. Give only the correct number.",\n    "target": "5",\n    "category": "arithmetic_subtract"\n  },\n  {\n    "id": "p03_correct_draft",\n    "pair_id": "p03",\n    "prompt": "6 * 3 = ?. Draft: 18. Give only the correct number.",\n    "target": "18",\n    "category": "arithmetic_multiply"\n  },\n  {\n    "id": "p03_wrong_draft",\n    "pair_id": "p03",\n    "prompt": "6 * 3 = ?. Draft: 19. Give only the correct number.",\n    "target": "18",\n    "category": "arithmetic_multiply"\n  },\n  {\n    "id": "p04_correct_draft",\n    "pair_id": "p04",\n    "prompt": "20 / 5 = ?. Draft: 4. Give only the correct number.",\n    "target": "4",\n    "category": "arithmetic_divide"\n  },\n  {\n    "id": "p04_wrong_draft",\n    "pair_id": "p04",\n    "prompt": "20 / 5 = ?. Draft: 3. Give only the correct number.",\n    "target": "4",\n    "category": "arithmetic_divide"\n  },\n  {\n    "id": "p05_correct_draft",\n    "pair_id": "p05",\n    "prompt": "The key is in the blue box. Draft: blue. Give the box color only.",\n    "target": "blue",\n    "category": "context_fact"\n  },\n  {\n    "id": "p05_wrong_draft",\n    "pair_id": "p05",\n    "prompt": "The key is in the blue box. Draft: pink. Give the box color only.",\n    "target": "blue",\n    "category": "context_fact"\n  },\n  {\n    "id": "p06_correct_draft",\n    "pair_id": "p06",\n    "prompt": "The card was blue; now it is pink. Draft: pink. Give its current color only.",\n    "target": "pink",\n    "category": "context_update"\n  },\n  {\n    "id": "p06_wrong_draft",\n    "pair_id": "p06",\n    "prompt": "The card was blue; now it is pink. Draft: blue. Give its current color only.",\n    "target": "pink",\n    "category": "context_update"\n  },\n  {\n    "id": "p07_correct_draft",\n    "pair_id": "p07",\n    "prompt": "Every wug is green. Pip is a wug. Draft: green. Give Pip\'s color only.",\n    "target": "green",\n    "category": "rule_application"\n  },\n  {\n    "id": "p07_wrong_draft",\n    "pair_id": "p07",\n    "prompt": "Every wug is green. Pip is a wug. Draft: brown. Give Pip\'s color only.",\n    "target": "green",\n    "category": "rule_application"\n  },\n  {\n    "id": "p08_correct_draft",\n    "pair_id": "p08",\n    "prompt": "Exactly one of A and B is on. A is off. Draft: B. Which is on? Reply A or B.",\n    "target": "B",\n    "category": "exclusive_logic"\n  },\n  {\n    "id": "p08_wrong_draft",\n    "pair_id": "p08",\n    "prompt": "Exactly one of A and B is on. A is off. Draft: A. Which is on? Reply A or B.",\n    "target": "B",\n    "category": "exclusive_logic"\n  }\n]\n', 'qwen3_h286_runtime.py': '"""Short exploratory ORIGINAL Qwen3/H286 pilot; embedded notebook entry point.\n\nNo re-selection, fitting, generation, training, or cross-model coordinate mapping.\nOriginal FP32, TF32-off, fixed REF-W and efficient SDPA are retained. The two\ntechnical checks precede a fixed 16-request/five-arm exploratory run. Results do\nnot constitute a held-out confirmatory test or a proven attention bottleneck.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport datetime as dt\nimport hashlib\nimport json\nimport os\nimport platform\nimport random\nimport shutil\nimport tempfile\nimport time\nimport traceback\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\n\nMODEL = \'Qwen/Qwen3-1.7B\'\nREVISION = \'70d244cc86ccca08cf5af4e1e306ecf908b1ad5e\'\nTRANSFORMERS = \'5.22.2\'\nACCELERATE = \'1.16.0\'\nREFW_SHA256 = \'68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc\'\nMETA_SHA256 = \'9c61bc34e804719e5bf5e72b644e8a38a94ad0f6c1708efaef75386acdae49ff\'\nSEED = 20261117\nARMS = (\n    {\'name\': \'sham\', \'block\': 14, \'beta\': 1., \'mode\': \'scale\'},\n    {\'name\': \'soft_103\', \'block\': 14, \'beta\': 1 / 1.03, \'mode\': \'scale\'},\n    {\'name\': \'soft_108\', \'block\': 14, \'beta\': 1 / 1.08, \'mode\': \'scale\'},\n    {\'name\': \'random_matched_108\', \'block\': 14, \'beta\': 1 / 1.08, \'mode\': \'random\'},\n    {\'name\': \'other_block_108\', \'block\': 7, \'beta\': 1 / 1.08, \'mode\': \'scale\'},\n)\n# Declared before any new model result. These are technical/exploratory items,\n# not the frozen DSSE evaluation sample or a benchmark accuracy test.\nITEMS_SHA256 = \'12a3cba862185b9f25ca39ea3ea49bd3bb6fc5f248c12dbbe02bcc8836663fb3\'\n\n\ndef canonical(value):\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'), ensure_ascii=False, allow_nan=False).encode()\n\n\ndef sha_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 << 20), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=path.parent, prefix=\'.\' + path.name, delete=False) as handle:\n        temporary = Path(handle.name)\n        handle.write(canonical(value) + b\'\\n\')\n        handle.flush()\n        os.fsync(handle.fileno())\n    try:\n        os.replace(temporary, path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef archive(output):\n    target = output / \'qwen3_h286_results.zip\'\n    temporary = output / \'.qwen3_h286_results.zip.tmp\'\n    with zipfile.ZipFile(temporary, \'w\', zipfile.ZIP_DEFLATED) as bundle:\n        for item in sorted(output.rglob(\'*\')):\n            if item.is_file() and item not in (target, temporary) and not any(part.startswith(\'.\') for part in item.relative_to(output).parts):\n                bundle.write(item, item.relative_to(output))\n    os.replace(temporary, target)\n    return target\n\n\ndef load_inputs(hn_path, policy_path, refw_path, items_path):\n    from attention_probe.fixed_refw import load_fixed_refw\n    from attention_probe.reference import load_reference\n    reference = load_reference(hn_path, policy_path)\n    stats = load_fixed_refw(refw_path, reference)\n    raw = Path(items_path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != ITEMS_SHA256:\n        raise ValueError(\'The exact prespecified 16-item file is required; no replacement or post-result selection\')\n    items = json.loads(raw)\n    if len(items) != 16 or len({item[\'id\'] for item in items}) != 16:\n        raise ValueError(\'Exactly 16 unique fixed requests are required\')\n    return reference, stats, items\n\n\ndef hardware_preflight(torch, cache_dir):\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA GPU erforderlich; keine Warteschleife. A100 empfohlen, L4 ausreichend.\')\n    free, total = torch.cuda.mem_get_info(0)\n    cache_dir = Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    disk = shutil.disk_usage(cache_dir).free\n    if free < 10 * (1 << 30):\n        raise RuntimeError(\'Mindestens 10 GiB freie GPU-VRAM für unverändertes FP32 erforderlich.\')\n    if disk < 8 * (1 << 30):\n        raise RuntimeError(\'Mindestens 8 GiB freier Cache-Speicher erforderlich; keine Gewichte heruntergeladen.\')\n    available = None\n    meminfo = Path(\'/proc/meminfo\')\n    if meminfo.exists():\n        fields = dict(line.split(\':\', 1) for line in meminfo.read_text().splitlines())\n        available = int(fields[\'MemAvailable\'].split()[0]) * 1024\n        if available < 10 * (1 << 30):\n            raise RuntimeError(\'Mindestens 10 GiB verfügbarer Host-RAM erforderlich.\')\n    return {\'gpu\': torch.cuda.get_device_name(0), \'free_vram\': free, \'total_vram\': total,\n            \'free_disk\': disk, \'available_host_ram\': available, \'device\': \'cuda:0\'}\n\n\ndef model_load_kwargs(torch):\n    return {\'revision\': REVISION, \'dtype\': torch.float32, \'device_map\': {\'\': 0},\n            \'attn_implementation\': \'sdpa\', \'trust_remote_code\': False, \'use_safetensors\': True,\n            \'local_files_only\': True}\n\n\ndef load_model(reference, cache_dir):\n    import torch\n    from huggingface_hub import snapshot_download\n    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer\n    snapshot = Path(snapshot_download(MODEL, revision=REVISION, cache_dir=str(cache_dir),\n                                     allow_patterns=[\'*.json\', \'*.safetensors\', \'*.jinja\']))\n    if snapshot.name != REVISION:\n        raise ValueError(\'HF snapshot path does not identify the exact pinned revision\')\n    for filename, expected in reference.model.sha256:\n        if sha_file(snapshot / filename) != expected:\n            raise ValueError(\'Original model/tokenizer SHA mismatch: \' + filename)\n    config = AutoConfig.from_pretrained(str(snapshot), revision=REVISION, trust_remote_code=False,\n                                       local_files_only=True)\n    if config.model_type != \'qwen3\' or config.num_hidden_layers != 28 or config.intermediate_size != 6144:\n        raise ValueError(\'Original Qwen3-1.7B architecture mismatch\')\n    tokenizer = AutoTokenizer.from_pretrained(str(snapshot), revision=REVISION, trust_remote_code=False,\n                                             local_files_only=True)\n    tokenizer.padding_side = \'left\'\n    model = AutoModelForCausalLM.from_pretrained(str(snapshot), config=config, **model_load_kwargs(torch))\n    model.eval().requires_grad_(False)\n    if any(parameter.device.type != \'cuda\' or parameter.device.index != 0 or parameter.dtype != torch.float32\n           for parameter in model.parameters()):\n        raise ValueError(\'All model parameters must remain full FP32 on cuda:0; no offload/quantization\')\n    return tokenizer, model\n\n\ndef prepare_request(tokenizer, index, item):\n    import torch\n    category, question, answer = item[\'category\'], item[\'prompt\'], item[\'target\']\n    formatted = tokenizer.apply_chat_template([{\'role\': \'user\', \'content\': question}], tokenize=False,\n                                              add_generation_prompt=True, enable_thinking=False)\n    prompt = tokenizer(formatted, add_special_tokens=False)[\'input_ids\']\n    answer_ids = tokenizer(answer, add_special_tokens=False)[\'input_ids\']\n    if not prompt or not answer_ids or len(prompt) + len(answer_ids) > 128:\n        raise ValueError(\'Unexpected empty or long fixed teacher-forcing request; no silent truncation\')\n    # Token at p predicts token p+1. The last answer activation is recorded but\n    # cannot cause its own prediction. Answer likelihood uses preceding positions.\n    ids = prompt + answer_ids\n    positions = list(range(max(0, len(ids) - 4), len(ids)))\n    return {\'request_id\': f\'r{index:03d}\', \'item_id\': item[\'id\'], \'pair_id\': item[\'pair_id\'], \'category\': category, \'question\': question, \'answer\': answer,\n            \'formatted_prompt\': formatted, \'formatted_prompt_sha256\': hashlib.sha256(formatted.encode()).hexdigest(), \'prompt_ids\': prompt, \'answer_ids\': answer_ids,\n            \'input_ids\': ids, \'capture_positions\': positions, \'pulse_positions\': [len(prompt) - 1],\n            \'answer_logit_positions\': list(range(len(prompt) - 1, len(ids) - 1)),\n            \'thinking_enabled\': False, \'ids\': torch.tensor([ids], dtype=torch.long)}\n\n\n@contextlib.contextmanager\ndef cett_tap(model, coordinates, positions, weight_norms=None):\n    import torch\n    groups = defaultdict(list)\n    for order, (layer, neuron) in enumerate(coordinates):\n        groups[layer].append((order, neuron))\n    result = {\'cett\': torch.empty((1, len(positions), len(coordinates)), dtype=torch.float32),\n              \'denominator\': torch.empty((1, len(positions), len(coordinates)), dtype=torch.float32)}\n    handles, seen = [], set()\n    try:\n        for layer, pairs in groups.items():\n            down = model.model.layers[layer].mlp.down_proj\n            columns = [neuron for _, neuron in pairs]\n            weight_norm = (weight_norms[layer] if weight_norms is not None\n                           else down.weight.float().norm(dim=0)[columns])\n\n            def hook(_module, args, output, layer=layer, pairs=pairs, columns=columns, weight_norm=weight_norm):\n                if layer in seen:\n                    raise ValueError(\'CETT tap fired repeatedly\')\n                z = args[0][:, positions][:, :, columns].abs().float()\n                den = output.norm(dim=-1)[:, positions].float()\n                contribution = z * weight_norm / den.clamp_min(1e-9)[..., None]\n                order = [index for index, _ in pairs]\n                result[\'cett\'][:, :, order] = contribution.detach().cpu()\n                result[\'denominator\'][:, :, order] = den[..., None].expand_as(contribution).detach().cpu()\n                seen.add(layer)\n            handles.append(down.register_forward_hook(hook))\n        yield result\n        if seen != set(groups):\n            raise ValueError(\'CETT capture incomplete\')\n    finally:\n        for handle in reversed(handles):\n            handle.remove()\n\n\ndef parameter_evidence(model):\n    return tuple((name, id(value), value.data_ptr(), value._version, str(value.dtype), str(value.device))\n                 for name, value in model.named_parameters())\n\n\ndef run_arm(model, request, coordinates, stats, arm, *, efficient_gpu=True, include_logits=False, weight_norms=None):\n    import torch\n\n    from attention_probe.interventions import AttentionPulse, CoordinateCapture\n    device = next(model.parameters()).device\n    ids = request[\'ids\'].to(device)\n    attention = (torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION)\n                 if efficient_gpu else contextlib.nullcontext())\n    started = time.monotonic()\n    with (torch.inference_mode(), attention,\n          CoordinateCapture(model, coordinates, request[\'capture_positions\'],\n                            residual_layers=tuple(layer for layer in (3, 7, 14, 27) if layer < len(model.model.layers)),\n                            expected_count=len(coordinates)) as capture,\n          cett_tap(model, coordinates, request[\'capture_positions\'], weight_norms) as contributions,\n          AttentionPulse(model, arm[\'block\'], request[\'pulse_positions\'], arm[\'beta\'], arm[\'mode\'],\n                         seed=SEED + int(request[\'request_id\'][1:])) as pulse):\n        output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)\n        if not bool(torch.isfinite(output.logits).all()):\n            raise ValueError(\'Nonfinite model logits\')\n        selected_logits = output.logits[:, request[\'answer_logit_positions\']].float()\n        targets = torch.tensor(request[\'answer_ids\'], device=device)[None, :, None]\n        logprob = selected_logits.log_softmax(-1).gather(-1, targets).squeeze(-1).cpu()\n        raw = capture.values\n        cett = contributions[\'cett\']\n        mu = torch.tensor(stats[\'refw\'][\'c_mu\'], dtype=torch.float32)\n        sd = torch.tensor(stats[\'refw\'][\'c_sd\'], dtype=torch.float32).clamp_min(1e-6)\n        zcett = (cett - mu) / sd\n        result = {\'arm\': arm, \'answer_token_logprobs\': logprob.flatten().tolist(),\n                  \'answer_logprob_sum\': float(logprob.sum()), \'answer_logprob_mean\': float(logprob.mean()),\n                  \'pulse_fired\': pulse.fired_count, \'capture_fired\': capture.fired_count,\n                  \'requested_delta_norms\': pulse.requested_delta_norms.tolist(),\n                  \'actual_delta_norms\': pulse.actual_delta_norms.tolist(),\n                  \'elapsed_seconds\': time.monotonic() - started}\n        arrays = {\'raw_z\': raw.numpy(), \'cett\': cett.numpy(), \'zcett\': zcett.numpy(),\n                  \'down_output_norm_raw\': contributions[\'denominator\'].numpy(),\n                  \'gold_token_logprob\': logprob.numpy()}\n        for layer, value in capture.residuals.items():\n            arrays[f\'residual_layer_{layer}\'] = value.numpy()\n        if include_logits:\n            arrays[\'technical_logits\'] = selected_logits.detach().cpu().numpy()\n    return result, arrays\n\n\ndef plan_document(reference, stats, items):\n    import attention_probe.reference as reference_module\n    module_root = Path(reference_module.__file__).parent\n    module_hashes = {name: sha_file(module_root / name) for name in (\'reference.py\', \'interventions.py\', \'fixed_refw.py\')}\n    return {\'schema\': \'qwen3_h286_short_pilot_v1\', \'model\': reference.model.to_dict(),\n            \'hn_sha256\': reference.hn_sha256, \'policy_sha256\': reference.policy_sha256,\n            \'refw_sha256\': REFW_SHA256, \'refw_source_sha256\': stats[\'source_hn_meta_sha256\'],\n            \'coordinates\': [list(pair) for pair in reference.coordinates], \'seed\': SEED,\n            \'arms\': list(ARMS), \'items\': items, \'items_sha256\': ITEMS_SHA256,\n            \'helper_sha256\': sha_file(__file__), \'module_sha256\': module_hashes,\n            \'dtype\': \'float32\', \'tf32\': False, \'attention\': \'sdpa/EFFICIENT_ATTENTION\',\n            \'transformers\': TRANSFORMERS, \'thinking_enabled\': False,\n            \'claim\': \'externally supplied draft-checking proxy; not own prior output or proof of self-monitoring; no held-out confirmatory test\',\n            \'candidate_block\': \'14 zero-based, prespecified candidate; not a proven bottleneck\',\n            \'cett\': \'abs(z)*norm(Wdown_col)/max(norm(down(z)),1e-9)\',\n            \'zcett\': \'(CETT-fixed_REF_W_c_mu)/max(fixed_REF_W_c_sd,1e-6)\'}\n\n\ndef verified_request(output, request_id, plan_hash):\n    directory = output / \'requests\' / request_id\n    if not directory.exists():\n        return None\n    record = json.loads((directory / \'result.json\').read_text())\n    if (record.get(\'state\') != \'complete\' or record.get(\'plan_sha256\') != plan_hash\n            or record.get(\'request_id\') != request_id or len(record.get(\'arms\', [])) != 5\n            or [row[\'arm\'] for row in record[\'arms\']] != list(ARMS)):\n        raise ValueError(\'Existing request does not match fixed all-five-arm plan: \' + request_id)\n    proof = record.get(\'technical_checks\')\n    if not isinstance(proof, dict) or proof.get(\'state\') != \'passed\' or proof.get(\'plan_sha256\') != plan_hash:\n        raise ValueError(\'Request lacks bound technical-check evidence: \' + request_id)\n    if sha_file(directory / \'arrays.npz\') != record[\'npz_sha256\']:\n        raise ValueError(\'Existing request array SHA mismatch: \' + request_id)\n    return record\n\n\ndef restore_resume(resume_zip, output, plan_hash):\n    if not resume_zip:\n        return\n    with zipfile.ZipFile(resume_zip) as bundle:\n        source_plan = json.loads(bundle.read(\'run_plan.json\'))\n        if hashlib.sha256(canonical(source_plan)).hexdigest() != plan_hash:\n            raise ValueError(\'Resume ZIP belongs to a different immutable plan/reference\')\n        for index in range(16):\n            request_id = f\'r{index:03d}\'\n            prefix = f\'requests/{request_id}/\'\n            names = [prefix + \'result.json\', prefix + \'arrays.npz\']\n            present = [name in bundle.namelist() for name in names]\n            if not any(present):\n                continue\n            if not all(present):\n                raise ValueError(\'Resume ZIP contains an incomplete request chunk\')\n            destination = output / \'requests\' / request_id\n            if destination.exists():\n                verified_request(output, request_id, plan_hash)\n                if any((destination / Path(name).name).read_bytes() != bundle.read(name) for name in names):\n                    raise ValueError(\'Resume ZIP conflicts with existing complete request; no overwrite\')\n                continue\n            parent = destination.parent\n            parent.mkdir(parents=True, exist_ok=True)\n            temporary = Path(tempfile.mkdtemp(prefix=\'.\' + request_id + \'_\', dir=parent))\n            try:\n                for name in names:\n                    if bundle.getinfo(name).file_size > 128 * (1 << 20):\n                        raise ValueError(\'Oversized request chunk in resume ZIP\')\n                    (temporary / Path(name).name).write_bytes(bundle.read(name))\n                record = json.loads((temporary / \'result.json\').read_text())\n                if record.get(\'plan_sha256\') != plan_hash or sha_file(temporary / \'arrays.npz\') != record.get(\'npz_sha256\'):\n                    raise ValueError(\'Resume request failed hash/plan verification\')\n                os.rename(temporary, destination)\n                verified_request(output, request_id, plan_hash)\n            finally:\n                if temporary.exists():\n                    shutil.rmtree(temporary)\n\n\ndef write_request(output, request, results, arrays, plan_hash, hardware, technical_checks=None):\n    import numpy as np\n    destination = output / \'requests\' / request[\'request_id\']\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if destination.exists():\n        raise ValueError(\'Refusing to overwrite existing request outputs\')\n    temporary = Path(tempfile.mkdtemp(dir=destination.parent, prefix=\'.\' + destination.name + \'_\'))\n    try:\n        np.savez_compressed(temporary / \'arrays.npz\',\n                            **{key: np.stack([row[key] for row in arrays]) for key in arrays[0]})\n        with (temporary / \'arrays.npz\').open(\'rb\') as stream:\n            os.fsync(stream.fileno())\n        public_request = {key: value for key, value in request.items() if key != \'ids\'}\n        record = {\'state\': \'complete\', \'request_id\': request[\'request_id\'], \'request\': public_request,\n                  \'plan_sha256\': plan_hash, \'npz_sha256\': sha_file(temporary / \'arrays.npz\'), \'arms\': results,\n                  \'hardware\': hardware, \'technical_checks\': technical_checks, \'completed_utc\': dt.datetime.now(dt.UTC).isoformat()}\n        atomic_json(temporary / \'result.json\', record)\n        os.rename(temporary, destination)  # Both files become visible as one paired-request checkpoint.\n        return record\n    finally:\n        if temporary.exists():\n            shutil.rmtree(temporary)\n\n\ndef run(hn_set, policy, refw, items_path, out_dir, cache_dir=None, resume_zip=None):\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    status = {\'state\': \'running\', \'stage\': \'reference\', \'completed_requests\': 0,\n              \'confirmatory_eligible\': False, \'scope\': \'original_fixed_H286_exploratory_pilot\'}\n    metrics = {\'pretrained_model_run\': False, \'checks\': {}, \'no_training\': True}\n    started = time.monotonic()\n    def save():\n        atomic_json(output / \'status.json\', status)\n        atomic_json(output / \'metrics.json\', metrics)\n    save()\n    try:\n        reference, stats, items = load_inputs(hn_set, policy, refw, items_path)\n        plan = plan_document(reference, stats, items)\n        plan_hash = hashlib.sha256(canonical(plan)).hexdigest()\n        existing = output / \'run_plan.json\'\n        if existing.exists() and existing.read_bytes() != canonical(plan) + b\'\\n\':\n            raise ValueError(\'Existing output uses a different plan; choose a fresh directory\')\n        if not existing.exists():\n            atomic_json(existing, plan)\n        restore_resume(resume_zip, output, plan_hash)\n        done = [verified_request(output, f\'r{index:03d}\', plan_hash) for index in range(len(items))]\n        status[\'completed_requests\'] = sum(item is not None for item in done)\n        if all(item is not None for item in done):\n            metrics[\'technical_checks_from_completed_requests\'] = [row.get(\'technical_checks\') for row in done]\n            metrics[\'checks_executed_this_attempt\'] = False\n            if any(not isinstance(row.get(\'technical_checks\'), dict) or row[\'technical_checks\'].get(\'state\') != \'passed\'\n                   or row[\'technical_checks\'].get(\'plan_sha256\') != plan_hash for row in done):\n                raise ValueError(\'Completed resume lacks the bound technical-check evidence\')\n            status.update(state=\'complete\', stage=\'complete\', resumed_without_gpu=True)\n            return status\n        from importlib.metadata import version\n\n        import torch\n        import transformers\n        if transformers.__version__ != TRANSFORMERS or version(\'accelerate\') != ACCELERATE:\n            raise ValueError(\'Exact transformers5.22.2 and accelerate1.16.0 are required\')\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n        torch.set_float32_matmul_precision(\'highest\')\n        torch.manual_seed(SEED)\n        disable_bar = getattr(transformers.utils.logging, \'disable_progress_bar\', None)\n        if disable_bar is not None:\n            disable_bar()\n        metrics.update(torch=torch.__version__, transformers=transformers.__version__, python=platform.python_version())\n        status[\'stage\'] = \'hardware_preflight\'\n        cache = Path(cache_dir or Path.home() / \'.cache\' / \'h286_pilot\')\n        hardware = hardware_preflight(torch, cache)\n        metrics[\'hardware\'] = hardware\n        status[\'stage\'] = \'pinned_download_and_load\'\n        save()\n        os.environ[\'HF_DEACTIVATE_ASYNC_LOAD\'] = \'1\'\n        print(\'Status: Original Qwen3-1.7B laden, exakte Dateien prüfen; FP32/SDPA.\', flush=True)\n        tokenizer, model = load_model(reference, cache)\n        metrics[\'weights_loaded\'] = True\n        hardware.update(torch=torch.__version__, cuda=torch.version.cuda, python=platform.python_version(), compute_capability=list(torch.cuda.get_device_capability(0)), transformers=transformers.__version__, attention=\'sdpa/EFFICIENT_ATTENTION\', dtype=\'float32\', tf32=False)\n        groups = defaultdict(list)\n        for layer, neuron in reference.coordinates:\n            groups[layer].append(neuron)\n        with torch.inference_mode():\n            weight_norms = {layer: model.model.layers[layer].mlp.down_proj.weight.float().norm(dim=0)[columns]\n                            for layer, columns in groups.items()}\n        evidence = parameter_evidence(model)\n        first_index = next(index for index, result in enumerate(done) if result is None)\n        warm = prepare_request(tokenizer, first_index, items[first_index])\n        ids = warm[\'ids\'].to(\'cuda:0\')\n        status[\'stage\'] = \'warmup\'\n        save()\n        with torch.inference_mode(), torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION):\n            model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)  # untimed warmup\n            baseline = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False).logits.detach().cpu()\n        metrics[\'pretrained_model_run\'] = True\n        status[\'stage\'] = \'technical_checks\'\n        print(\'Status: Zwei technische Prüfungen vor der festen 16-Request-Auswertung.\', flush=True)\n        _sham, sham_arrays = run_arm(model, warm, reference.coordinates, stats, ARMS[0], include_logits=True, weight_norms=weight_norms)\n        selected_baseline = baseline[:, warm[\'answer_logit_positions\']].float()\n        observed = torch.from_numpy(sham_arrays[\'technical_logits\'])\n        delta = (selected_baseline - observed).abs()\n        metrics[\'checks\'][\'sham\'] = {\'bitwise_equal\': torch.equal(selected_baseline, observed),\n                                     \'max_abs_logit_difference\': float(delta.max()), \'rtol\': 1e-5, \'atol\': 1e-6}\n        if not torch.allclose(selected_baseline, observed, rtol=1e-5, atol=1e-6):\n            raise ValueError(\'Technical sham logit check failed; exploratory queue not started\')\n        pulse_result, pulse_arrays = run_arm(model, warm, reference.coordinates, stats, ARMS[1], weight_norms=weight_norms)\n        import numpy as np\n        upstream = [index for index, (layer, _) in enumerate(reference.coordinates) if layer < 14]\n        earlier = [index for index, position in enumerate(warm[\'capture_positions\']) if position < warm[\'pulse_positions\'][0]]\n        upstream_equal = np.array_equal(sham_arrays[\'raw_z\'][..., upstream], pulse_arrays[\'raw_z\'][..., upstream])\n        earlier_equal = np.array_equal(sham_arrays[\'raw_z\'][:, earlier], pulse_arrays[\'raw_z\'][:, earlier])\n        metrics[\'checks\'][\'causality\'] = {\'upstream_coordinates\': len(upstream), \'upstream_raw_equal\': upstream_equal,\n                                           \'earlier_positions\': len(earlier), \'earlier_raw_equal\': earlier_equal}\n        if not upstream_equal or not earlier_equal:\n            raise ValueError(\'Technical causal guard failed; exploratory queue not started\')\n        metrics[\'checks\'][\'pulse_and_capture\'] = {\'pulse_once\': pulse_result[\'pulse_fired\'] == 1,\n                                                  \'capture_once\': pulse_result[\'capture_fired\'] == 1,\n                                                  \'frozen_parameter_evidence\': parameter_evidence(model) == evidence}\n        if parameter_evidence(model) != evidence:\n            raise ValueError(\'Model parameter identity/storage/version counters changed\')\n        metrics[\'checks_executed_this_attempt\'] = True\n        technical_checks = {\'state\': \'passed\', \'plan_sha256\': plan_hash, \'checks\': metrics[\'checks\'],\n                            \'hardware\': hardware, \'executed_utc\': dt.datetime.now(dt.UTC).isoformat()}\n        measured = []\n        for index, item in enumerate(items):\n            if done[index] is not None:\n                continue\n            request = prepare_request(tokenizer, index, item)\n            status.update(stage=\'exploratory_requests\', current_request=request[\'request_id\'])\n            save()\n            print(f"Request {index + 1}/16: {request[\'request_id\']} — fünf feste Arme", flush=True)\n            t0 = time.monotonic()\n            order = list(range(len(ARMS)))\n            random.Random(SEED + index).shuffle(order)\n            request[\'execution_order\'] = [ARMS[arm_index][\'name\'] for arm_index in order]\n            results, array_rows = [None] * len(ARMS), [None] * len(ARMS)\n            cpu_rng, cuda_rng = torch.random.get_rng_state().clone(), torch.cuda.get_rng_state().clone()\n            for arm_index in order:\n                result, arrays = run_arm(model, request, reference.coordinates, stats, ARMS[arm_index], weight_norms=weight_norms)\n                results[arm_index], array_rows[arm_index] = result, arrays\n            if not torch.equal(cpu_rng, torch.random.get_rng_state()) or not torch.equal(cuda_rng, torch.cuda.get_rng_state()):\n                raise ValueError(\'Global RNG changed during fixed five-arm request\')\n            if parameter_evidence(model) != evidence:\n                raise ValueError(\'Frozen parameter evidence changed; request is not published complete\')\n            record = write_request(output, request, results, array_rows, plan_hash, hardware, technical_checks)\n            done[index] = record\n            measured.append(time.monotonic() - t0)\n            status[\'completed_requests\'] = sum(row is not None for row in done)\n            remaining = len(items) - status[\'completed_requests\']\n            eta_seconds = sum(measured) / len(measured) * remaining\n            finish = dt.datetime.now(ZoneInfo(\'Europe/Berlin\')) + dt.timedelta(seconds=eta_seconds)\n            status.update(estimated_remaining_seconds=eta_seconds, estimated_finish_berlin=finish.isoformat())\n            save()\n            print(f"Gespeichert {status[\'completed_requests\']}/16; Restschätzung {eta_seconds / 60:.1f} min; Berlin {finish:%H:%M}", flush=True)\n            if remaining and (eta_seconds > 3600 or finish.date() > dt.datetime.now(ZoneInfo(\'Europe/Berlin\')).date()):\n                status.update(state=\'paused\', stage=\'safe_request_checkpoint\',\n                              reason=\'Runtime estimate exceeds one hour or today; resume ZIP keeps paired requests together\')\n                break\n        else:\n            status.update(state=\'complete\', stage=\'complete\')\n        metrics[\'weights_unchanged_evidence\'] = \'same parameter identities, storage pointers, version counters; no training; no full tensor rehash\'\n        metrics[\'hardware_comparison_warning\'] = \'Each complete paired request stays on one GPU; resumed chunks can differ by GPU/software.\'\n    except BaseException as exc:  # noqa: BLE001 — preserve diagnostics also on interruption\n        status.update(state=\'failed\', error_type=type(exc).__name__, error=str(exc))\n        (output / \'traceback.txt\').write_text(traceback.format_exc())\n        print(\'HARD FAILURE: \' + str(exc), flush=True)\n    finally:\n        metrics[\'elapsed_seconds\'] = time.monotonic() - started\n        save()\n        archive(output)\n    return status\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--hn-set\', required=True)\n    parser.add_argument(\'--policy\', required=True)\n    parser.add_argument(\'--refw\', required=True)\n    parser.add_argument(\'--items\', required=True)\n    parser.add_argument(\'--out-dir\', required=True)\n    parser.add_argument(\'--cache-dir\')\n    parser.add_argument(\'--resume-zip\')\n    args = parser.parse_args()\n    result = run(args.hn_set, args.policy, args.refw, args.items, args.out_dir, args.cache_dir, args.resume_zip)\n    print(json.dumps(result), flush=True)\n    raise SystemExit(0 if result[\'state\'] in (\'complete\', \'paused\') else 1)\n', 'attention_probe/answer_validation.py': '"""Prompt-only greedy answer validation; the attention pulse ends after prefill.\n\nThe old teacher-forcing runner is unchanged. This module retains its fixed REF-W\nCETT definition but never supplies request[\'ids\'], input_ids or answer_ids to\ngeneration. All saved activation arrays contain prompt positions only.\n"""\nfrom __future__ import annotations\n\nimport contextlib\nimport hashlib\nimport re\nimport time\nfrom numbers import Integral\n\nimport torch\n\nfrom .interventions import AttentionPulse, CoordinateCapture, _cache_length\n\nSEED = 20261117\nMAX_NEW_TOKENS = 48\nTERMINAL_PUNCTUATION = ".!?,;:"\n\n\ndef normalize_answer(text: str) -> str:\n    """Only case, outer whitespace and one simple terminal punctuation mark."""\n    if not isinstance(text, str):\n        raise TypeError(\'Answer text must be a string\')\n    result = text.strip().casefold()\n    if result and result[-1] in TERMINAL_PUNCTUATION:\n        result = result[:-1].rstrip()\n    return result\n\n\ndef classify_answer(generated: str, expected: str, *, truncated: bool = False) -> dict:\n    """Strict surface comparison, never extract an answer from an explanation.\n\n    Numeric values are not parsed or rounded: \'15.0\', \'+15\' and \'015\' do not\n    match \'15\'. A capped output is always classified as truncated, even if its\n    current surface matches. Noncanonical is not a semantic-incorrect verdict.\n    """\n    normalized, gold = normalize_answer(generated), normalize_answer(expected)\n    if not gold:\n        raise ValueError(\'Expected answer must not be empty\')\n    surface = (\'exact\' if generated == expected else\n               \'format_equivalent\' if normalized == gold else \'noncanonical\')\n    return {"classification": \'truncated\' if truncated else surface,\n                "surface_classification": surface, "normalized_generated": normalized,\n                "normalized_expected": gold, "truncated": bool(truncated),\n                "normalization": \'casefold; outer whitespace; at most one final . ! ? , ; :\',\n                "interpretation": \'Surface comparison only; noncanonical is not a semantic-wrong label.\'}\n\n\ndef _decode(tokenizer, ids):\n    return tokenizer.decode(list(ids), skip_special_tokens=False, clean_up_tokenization_spaces=False)\n\n\ndef _eos_ids(model, vocab_size):\n    configured = getattr(getattr(model, \'generation_config\', None), \'eos_token_id\', None)\n    source = \'generation_config\'\n    if configured is None:\n        configured = getattr(model.config, \'eos_token_id\', None)\n        source = \'config\'\n    if configured is None:\n        return (), \'none\'\n    values = [configured] if isinstance(configured, Integral) else list(configured)\n    if any(isinstance(v, bool) or not isinstance(v, Integral) or not 0 <= v < vocab_size for v in values):\n        raise ValueError(\'EOS IDs must be integer vocabulary IDs\')\n    return tuple(sorted({int(v) for v in values})), source\n\n\ndef _cett_context(model, coordinates, positions, weight_norms):\n    # The source bundle installs the old runner at its root. A repository checkout\n    # can instead resolve it as the notebooks namespace; neither path changes it.\n    try:\n        from qwen3_h286_runtime import cett_tap\n    except ModuleNotFoundError as error:\n        if error.name != \'qwen3_h286_runtime\':\n            raise\n        from notebooks.qwen3_h286_runtime import cett_tap\n    return cett_tap(model, coordinates, positions, weight_norms)\n\n\ndef _sync(device):\n    if device.type == \'cuda\':\n        torch.cuda.synchronize(device)\n\n\ndef run_generation_arm(model, tokenizer, request, coordinates, stats, arm, *,\n                       efficient_gpu=True, weight_norms=None, max_new_tokens=48,\n                       top_k=10):\n    """Return a JSON-compatible record and fixed-shape prefill numpy arrays.\n\n    arm=None disables the pulse; passive prefill capture still runs. Only that\n    prefill has hooks. Decode uses its returned cache and newly selected tokens.\n    Selected EOS is logged in steps/selected_token_ids but excluded from emitted\n    generated_token_ids and generated_text. The 48-step maximum includes EOS.\n    """\n    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, Integral) or not 1 <= max_new_tokens <= MAX_NEW_TOKENS:\n        raise ValueError(\'max_new_tokens must be an integer from 1 through 48\')\n    if isinstance(top_k, bool) or not isinstance(top_k, Integral) or not 1 <= top_k <= 10:\n        raise ValueError(\'top_k must be an integer from 1 through 10\')\n    prompt = list(request[\'prompt_ids\'])  # Sole source of generation input IDs.\n    vocab_size = int(model.config.vocab_size)\n    if not prompt or any(isinstance(v, bool) or not isinstance(v, Integral) or not 0 <= v < vocab_size for v in prompt):\n        raise ValueError(\'prompt_ids must be a nonempty list of integer vocabulary IDs\')\n    request_id = request[\'request_id\']\n    if not isinstance(request_id, str) or re.fullmatch(r\'r[0-9]+\', request_id) is None:\n        raise ValueError(\'request_id must have the original rNNN form\')\n    seed = SEED + int(request_id[1:])\n    coordinates = tuple(tuple(pair) for pair in coordinates)\n    device = next(model.parameters()).device\n    if any(parameter.is_floating_point() and (parameter.dtype != torch.float32 or parameter.device != device)\n           for parameter in model.parameters()):\n        raise ValueError(\'Generation validation requires all floating parameters in FP32 on one device\')\n    if efficient_gpu:\n        if device.type != \'cuda\':\n            raise ValueError(\'efficient_gpu=True requires a CUDA model\')\n        if getattr(model.config, \'_attn_implementation\', None) != \'sdpa\':\n            raise ValueError(\'The GPU validation requires the sdpa attention implementation\')\n        if torch.backends.cuda.matmul.allow_tf32:\n            raise ValueError(\'Disable TF32 before full-FP32 generation validation\')\n    model.eval()\n    prompt_length = len(prompt)\n    positions = list(range(max(0, prompt_length-4), prompt_length))\n    pulse_positions = [prompt_length-1]\n    mu = torch.as_tensor(stats[\'refw\'][\'c_mu\'], dtype=torch.float32)\n    sd = torch.as_tensor(stats[\'refw\'][\'c_sd\'], dtype=torch.float32)\n    if mu.shape != (len(coordinates),) or sd.shape != mu.shape or not bool(torch.isfinite(mu).all() and torch.isfinite(sd).all()) or bool((sd < 0).any()):\n        raise ValueError(\'Fixed REF-W means/SDs must match the finite ordered coordinates\')\n    sd = sd.clamp_min(1e-6)\n    eos_ids, eos_source = _eos_ids(model, vocab_size)\n    ids = torch.tensor([prompt], dtype=torch.long, device=device)\n    attention = (torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION)\n                 if efficient_gpu else contextlib.nullcontext())\n    pulse = None\n    steps, emitted, selected_ids = [], [], []\n    decode_forwards = 0\n    _sync(device)\n    started = time.monotonic()\n    with torch.no_grad(), torch.autocast(device_type=device.type, enabled=False), attention:\n        with contextlib.ExitStack() as hooks:\n            capture = hooks.enter_context(CoordinateCapture(model, coordinates, positions,\n                residual_layers=tuple(l for l in (3, 7, 14, 27) if l < len(model.model.layers)),\n                expected_count=len(coordinates)))\n            contributions = hooks.enter_context(_cett_context(model, coordinates, positions, weight_norms))\n            if arm is not None:\n                pulse = hooks.enter_context(AttentionPulse(model, arm[\'block\'], pulse_positions,\n                    arm[\'beta\'], arm[\'mode\'], seed=seed))\n            output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=True)\n            cache = output.past_key_values\n            if cache is None:\n                raise RuntimeError(\'Prefill did not return a cache\')\n            # Capture arrays are copied to CPU by the existing read-only hooks.\n            raw = capture.values\n            cett = contributions[\'cett\']\n            zcett = (cett-mu)/sd\n            if not all(bool(torch.isfinite(v).all()) for v in (raw, cett, zcett, contributions[\'denominator\'])):\n                raise ValueError(\'Nonfinite prefill raw/CETT/reference-normalized capture\')\n            arrays = {"raw_z": raw.numpy(), "cett": cett.numpy(), "zcett": zcett.numpy(),\n                          "down_output_norm_raw": contributions[\'denominator\'].numpy()}\n            arrays.update({f\'residual_layer_{l}\': value.numpy() for l, value in capture.residuals.items()})\n            logits = output.logits[0, -1].float()\n        # ExitStack has removed every pulse/capture hook before any decode call.\n        _sync(device)\n        prefill_seconds = time.monotonic()-started\n        if not bool(torch.isfinite(logits).all()):\n            raise ValueError(\'Nonfinite prefill logits\')\n        first_logits = logits.detach().cpu().clone()\n        first_logits_sha = hashlib.sha256(first_logits.numpy().tobytes()).hexdigest()\n        stop_reason = \'max_new_tokens\'\n        for step in range(int(max_new_tokens)):\n            if not bool(torch.isfinite(logits).all()):\n                raise ValueError(\'Nonfinite generation logits\')\n            logprobs = logits.log_softmax(-1)\n            chosen = int(torch.argmax(logits))\n            values, indices = torch.topk(logprobs, k=min(int(top_k), vocab_size))\n            top_tokens = [{"token_id": int(t), "text": _decode(tokenizer, [int(t)]), "logprob": float(v)}\n                          for t, v in zip(indices.detach().cpu().tolist(), values.detach().cpu().tolist())]\n            is_eos = chosen in eos_ids\n            steps.append({"step": step, "phase": \'prefill\' if step == 0 else \'decode\',\n                "logit_position": prompt_length-1+step, "selected_token_id": chosen,\n                "selected_token_text": _decode(tokenizer, [chosen]), "selected_token_logprob": float(logprobs[chosen]),\n                "is_eos": is_eos, "top_tokens": top_tokens})\n            selected_ids.append(chosen)\n            if is_eos:\n                stop_reason = \'eos\'\n                break\n            emitted.append(chosen)\n            if step+1 == max_new_tokens:\n                break\n            # Only the model\'s own last selected token is passed to cached decode.\n            current = torch.tensor([[chosen]], dtype=torch.long, device=device)\n            mask = torch.ones((1, prompt_length+len(emitted)), dtype=torch.long, device=device)\n            output = model(input_ids=current, attention_mask=mask, past_key_values=cache, use_cache=True)\n            cache = output.past_key_values\n            if cache is None:\n                raise RuntimeError(\'Decode lost its cache\')\n            logits = output.logits[0, -1].float()\n            decode_forwards += 1\n        _sync(device)\n        elapsed = time.monotonic()-started\n    generated_text = _decode(tokenizer, emitted)\n    truncated = stop_reason == \'max_new_tokens\'\n    # Gold is consulted only after generation has ended, solely for diagnostics.\n    answer = request[\'answer\']\n    classification = classify_answer(generated_text, answer, truncated=truncated)\n    gold_ids = tokenizer(answer, add_special_tokens=False)[\'input_ids\']\n    if not gold_ids or not isinstance(gold_ids[0], Integral) or not 0 <= gold_ids[0] < vocab_size:\n        raise ValueError(\'Expected answer must tokenize to a valid first vocabulary ID\')\n    gold_first = int(gold_ids[0])\n    first_logprobs = first_logits.log_softmax(-1)\n    first_gold = {"token_id": gold_first, "text": _decode(tokenizer, [gold_first]),\n        "rank": int((first_logits > first_logits[gold_first]).sum())+1,\n        "rank_rule": \'one plus number of strictly higher logits; ties can share rank\',\n        "logprob": float(first_logprobs[gold_first]), "is_top1": int(torch.argmax(first_logits)) == gold_first}\n    record = {"schema": \'h286_prompt_only_generation_v1\', "request_id": request_id,\n        "arm": dict(arm) if arm is not None else None, "seed": seed, "prompt_length": prompt_length,\n        "generation_input_source": \'request.prompt_ids only; no teacher-forced answer input\',\n        "prefill_capture_positions": positions, "pulse_positions": pulse_positions if pulse is not None else [],\n        "max_new_tokens": int(max_new_tokens), "top_k": int(top_k), "eos_token_ids": list(eos_ids), "eos_source": eos_source,\n        "generated_token_ids": emitted, "generated_text": generated_text, "selected_token_ids": selected_ids,\n        "eos_convention": \'selected EOS is recorded in steps/selected_token_ids but not emitted text/IDs\',\n        "steps": steps, "num_steps": len(steps), "num_generated_tokens": len(emitted), "stop_reason": stop_reason,\n        "truncated": truncated, "classification": classification[\'classification\'], "classification_details": classification,\n        "first_token_gold": first_gold, "first_step_logits_sha256": first_logits_sha,\n        "pulse_fired": 0 if pulse is None else pulse.fired_count, "capture_fired": capture.fired_count,\n        "prefill_forward_count": 1, "decode_forward_count": decode_forwards, "final_cache_length": _cache_length(cache),\n        "requested_delta_norms": None if pulse is None else pulse.requested_delta_norms.tolist(),\n        "actual_delta_norms": None if pulse is None else pulse.actual_delta_norms.tolist(),\n        "elapsed_seconds": elapsed, "prefill_seconds": prefill_seconds, "decode_and_logging_seconds": elapsed-prefill_seconds,\n        "timing_note": \'Synchronized wall time includes capture and top-k logging; not energy telemetry.\',\n        "dtype": \'float32\', "attention": \'sdpa/EFFICIENT_ATTENTION\' if efficient_gpu else str(getattr(model.config, \'_attn_implementation\', None)),\n        "capture_scope": \'last min(4,prompt_length) PROMPT positions; no generated-token activation capture\'}\n    record[\'baseline_kind\'] = \'no_pulse_with_passive_capture\' if arm is None else \'prefill_pulse_with_passive_capture\'\n    return record, arrays\n', 'attention_probe/native_jacobian.py': '"""Native Qwen3 residual suffix maps and matrix-free local derivatives.\n\nA map is resid_post(source_block, p) -> resid_post(target_block, p), where p is\nONLY the last column of the original fixed prompt. Block numbers are zero-based;\noutputs precede the model\'s final RMSNorm. Other source-token rows stay fixed.\nThis is an offset-zero, prompt-local Jacobian, not a corpus-averaged J-lens or a\ncross-token/time transport. No model weights are loaded or changed here.\n\nOne no-cache HF prefill records the source residual and the exact native suffix\narguments (causal masks, position IDs, rotary cos/sin). Subsequent calls execute\nonly the original decoder layers source+1..target, with those arguments. Native\nMATH SDPA, FP32 and frozen weights are required. No TransformerLens conversion,\nsequence flattening, dense Jacobian, cache, optimizer, or inferred bottleneck.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom numbers import Integral, Real\n\nimport torch\nimport transformers\nfrom torch.nn.attention import SDPBackend, sdpa_kernel\n\nTRANSFORMERS_VERSION = \'5.22.2\'\n\n\nclass _TargetReached(Exception):\n    """Private early stop after the baseline target decoder block."""\n\n\ndef _integer(value, label):\n    if isinstance(value, bool) or not isinstance(value, Integral) or value < 0:\n        raise ValueError(f\'{label} must be a nonnegative integer\')\n    return int(value)\n\n\ndef _hooks(model):\n    return sum(len(module._forward_hooks)+len(module._forward_pre_hooks) for module in model.modules())\n\n\ndef _parameters(model):\n    return tuple((id(p), p.data_ptr(), p._version, p.dtype, p.device, p.requires_grad)\n                 for p in model.parameters())\n\n\ndef _model_device(model):\n    cls = type(model)\n    if (cls.__name__ != \'Qwen3ForCausalLM\'\n            or cls.__module__ != \'transformers.models.qwen3.modeling_qwen3\'\n            or getattr(model.config, \'model_type\', None) != \'qwen3\'):\n        raise ValueError(\'Expected native Hugging Face Qwen3ForCausalLM\')\n    if transformers.__version__ != TRANSFORMERS_VERSION:\n        raise ValueError(f\'Expected transformers=={TRANSFORMERS_VERSION}\')\n    if any(module.training for module in model.modules()):\n        raise ValueError(\'Expected model.eval(); the adapter never changes model mode\')\n    if getattr(model.config, \'_attn_implementation\', None) != \'sdpa\':\n        raise ValueError(\'Expected native SDPA implementation for the MATH diagnostic\')\n    params = tuple(model.parameters())\n    if not params or any(p.requires_grad for p in params):\n        raise ValueError(\'Model parameters must already be frozen\')\n    if any(p.is_floating_point() and p.dtype != torch.float32 for p in params):\n        raise ValueError(\'Native FP32 parameters required; no implicit dtype conversion\')\n    devices = {p.device for p in params}\n    if len(devices) != 1 or next(iter(devices)).type not in (\'cpu\', \'cuda\'):\n        raise ValueError(\'Expected parameters on one CPU/CUDA device, without meta/offload\')\n    return next(iter(devices))\n\n\ndef _execution_guard(model, device, expected_parameters=None):\n    if torch.is_inference_mode_enabled():\n        raise ValueError(\'Disable inference_mode: local derivatives require ordinary autograd tensors\')\n    if _model_device(model) != device:\n        raise ValueError(\'Model device changed after constructing the residual map\')\n    if _hooks(model):\n        raise ValueError(\'Residual map requires no pre-existing model hooks\')\n    if torch.is_autocast_enabled(device.type):\n        raise ValueError(\'Autocast must be disabled for native FP32 derivatives\')\n    if device.type == \'cuda\' and (torch.backends.cuda.matmul.allow_tf32 or torch.backends.cudnn.allow_tf32):\n        raise ValueError(\'CUDA matmul and cuDNN TF32 must both be disabled\')\n    if expected_parameters is not None and _parameters(model) != expected_parameters:\n        raise ValueError(\'Model parameters changed after constructing the residual map\')\n\n\ndef _constant(value):\n    """Private copies of native mask/RoPE arguments; never model/cache objects."""\n    if isinstance(value, torch.Tensor):\n        return value.detach().clone()\n    if isinstance(value, tuple):\n        return tuple(_constant(item) for item in value)\n    if isinstance(value, list):\n        return [_constant(item) for item in value]\n    if isinstance(value, dict):\n        return {key: _constant(item) for key, item in value.items()}\n    if value is None or isinstance(value, (str, bool, int, float)):\n        return value\n    raise ValueError(f\'Unexpected native suffix argument type: {type(value).__name__}\')\n\n\ndef _hidden(output, batch, tokens, width):\n    value = output[0] if isinstance(output, (tuple, list)) else output\n    if (not isinstance(value, torch.Tensor) or tuple(value.shape) != (batch, tokens, width)\n            or value.dtype != torch.float32):\n        raise ValueError(\'Unexpected native FP32 decoder residual shape\')\n    return value\n\n\nclass ResidualMap:\n    """Fixed-prompt native suffix map with [hidden_size] input/output vectors.\n\n    __call__(vector) returns the nonlinear target vector with an autograd graph\n    when vector requires gradients. jvp/vjp return detached local products at\n    baseline_source by default; `at` can specify another source vector. Both use\n    the SAME primal map and no parameter gradients. central_fd(v, step) computes\n    [F(x+step*v)-F(x-step*v)]/(2*step), so step is relative to v\'s native scale.\n    """\n\n    def __init__(self, model, input_ids, source_block, target_block, *, attention_mask=None):\n        self.model = model\n        self.device = _model_device(model)\n        _execution_guard(model, self.device)\n        self.source_block = _integer(source_block, \'source_block\')\n        self.target_block = _integer(target_block, \'target_block\')\n        self.layers = model.model.layers\n        if not self.source_block < self.target_block < len(self.layers):\n            raise ValueError(\'Require 0 <= source_block < target_block < native layer count\')\n        if (not isinstance(input_ids, torch.Tensor) or input_ids.dtype != torch.long\n                or input_ids.device != self.device or input_ids.ndim != 2\n                or input_ids.shape[0] != 1 or input_ids.shape[1] == 0):\n            raise ValueError(\'input_ids must be native-device int64 [1,T] with T>0\')\n        if not bool(((input_ids >= 0) & (input_ids < model.config.vocab_size)).all()):\n            raise ValueError(\'input_ids contain out-of-vocabulary indices\')\n        self.input_ids = input_ids.detach().clone()\n        self.tokens = int(input_ids.shape[1])\n        self.position = self.tokens-1\n        self.width = int(model.config.hidden_size)\n        if attention_mask is None:\n            attention_mask = torch.ones_like(input_ids)\n        if (not isinstance(attention_mask, torch.Tensor) or attention_mask.device != self.device\n                or tuple(attention_mask.shape) != tuple(input_ids.shape)\n                or attention_mask.requires_grad or not bool(((attention_mask == 0) | (attention_mask == 1)).all())\n                or not bool((attention_mask[:, -1] == 1).all())):\n            raise ValueError(\'attention_mask must be native-device binary [1,T], with the last token unmasked\')\n        self.attention_mask = attention_mask.detach().clone()\n        self._fingerprint = _parameters(model)\n        self._source_rows = None\n        self._target = None\n        self._calls = {}\n        self._busy = False\n        self._capture_baseline()\n\n    @property\n    def baseline_source(self):\n        return self._source_rows[0, -1, :].detach().clone()\n\n    @property\n    def baseline_target(self):\n        return self._target.detach().clone()\n\n    def _capture_baseline(self):\n        handles = []\n        seen = set()\n\n        def source_hook(module, args, output):\n            if self._source_rows is not None:\n                raise RuntimeError(\'Source decoder block fired twice\')\n            value = _hidden(output, 1, self.tokens, self.width)\n            if not bool(torch.isfinite(value).all()):\n                raise ValueError(\'Nonfinite native source residual\')\n            self._source_rows = value.detach().clone()\n\n        def capture_call(index):\n            def hook(module, args, kwargs):\n                if index in seen or len(args) != 1:\n                    raise ValueError(\'Unexpected repeated/positional native decoder call\')\n                if kwargs.get(\'past_key_values\') is not None or kwargs.get(\'use_cache\') is not False:\n                    raise ValueError(\'Suffix capture must use_cache=False and past_key_values=None\')\n                if kwargs.get(\'position_embeddings\') is None or kwargs.get(\'position_ids\') is None:\n                    raise ValueError(\'Native rotary/position arguments were not provided\')\n                self._calls[index] = _constant(kwargs)\n                seen.add(index)\n            return hook\n\n        def target_hook(module, args, output):\n            value = _hidden(output, 1, self.tokens, self.width)[0, -1, :]\n            if not bool(torch.isfinite(value).all()):\n                raise ValueError(\'Nonfinite native target residual\')\n            self._target = value.detach().clone()\n            raise _TargetReached\n\n        try:\n            handles.append(self.layers[self.source_block].register_forward_hook(source_hook))\n            for index in range(self.source_block+1, self.target_block+1):\n                handles.append(self.layers[index].register_forward_pre_hook(capture_call(index), with_kwargs=True))\n            handles.append(self.layers[self.target_block].register_forward_hook(target_hook))\n            try:\n                with torch.no_grad(), sdpa_kernel(SDPBackend.MATH):\n                    self.model(input_ids=self.input_ids, attention_mask=self.attention_mask, use_cache=False)\n            except _TargetReached:\n                pass\n        finally:\n            for handle in reversed(handles):\n                handle.remove()\n        expected = set(range(self.source_block+1, self.target_block+1))\n        if self._source_rows is None or self._target is None or seen != expected:\n            raise RuntimeError(\'Native baseline did not capture the complete requested suffix\')\n        _execution_guard(self.model, self.device, self._fingerprint)\n\n    def _vector(self, value, label):\n        if (not isinstance(value, torch.Tensor) or tuple(value.shape) != (self.width,)\n                or value.device != self.device or value.dtype != torch.float32):\n            raise ValueError(f\'{label} must be native-device FP32 [{self.width}]; no implicit casts/broadcasting\')\n        if value.is_inference():\n            raise ValueError(f\'{label} must not be an inference-mode tensor\')\n        if not bool(torch.isfinite(value).all()):\n            raise ValueError(f\'{label} must be finite\')\n        return value\n\n    def __call__(self, vector):\n        _execution_guard(self.model, self.device, self._fingerprint)\n        vector = self._vector(vector, \'source vector\')\n        if self._busy:\n            raise RuntimeError(\'ResidualMap calls must not overlap\')\n        self._busy = True\n        try:\n            with torch.enable_grad(), sdpa_kernel(SDPBackend.MATH):\n                hidden = self._source_rows.clone()\n                hidden[0, -1, :] = vector\n                for index in range(self.source_block+1, self.target_block+1):\n                    hidden = self.layers[index](hidden, **self._calls[index])\n                    hidden = _hidden(hidden, 1, self.tokens, self.width)\n                target = hidden[0, -1, :]\n                if not bool(torch.isfinite(target).all()):\n                    raise ValueError(\'Nonfinite target residual\')\n                return target\n        finally:\n            self._busy = False\n\n    def _point(self, at):\n        return self.baseline_source if at is None else self._vector(at, \'linearization point\').detach().clone()\n\n    def jvp(self, direction, *, at=None):\n        """Reverse-over-reverse Jv; no dense Jacobian and no parameter gradients."""\n        _execution_guard(self.model, self.device, self._fingerprint)\n        direction = self._vector(direction, \'direction\').detach()\n        with torch.enable_grad():\n            point = self._point(at).requires_grad_(True)\n            target = self(point)\n            cotangent = torch.zeros_like(target, requires_grad=True)\n            reverse = torch.autograd.grad(target, point, cotangent, create_graph=True)[0]\n            result = torch.autograd.grad(reverse, cotangent, direction)[0]\n        return result.detach()\n\n    def vjp(self, cotangent, *, at=None):\n        """J^T u at the same fixed-prompt primal map used by jvp."""\n        _execution_guard(self.model, self.device, self._fingerprint)\n        cotangent = self._vector(cotangent, \'cotangent\').detach()\n        with torch.enable_grad():\n            point = self._point(at).requires_grad_(True)\n            result = torch.autograd.grad(self(point), point, cotangent)[0]\n        return result.detach()\n\n    def central_fd(self, direction, step, *, at=None):\n        """Central finite difference, with an explicit positive finite step."""\n        _execution_guard(self.model, self.device, self._fingerprint)\n        direction = self._vector(direction, \'direction\').detach()\n        if isinstance(step, bool) or not isinstance(step, Real) or not math.isfinite(step) or step <= 0:\n            raise ValueError(\'Finite-difference step must be finite and positive\')\n        point = self._point(at)\n        plus = self(point+float(step)*direction).detach()\n        minus = self(point-float(step)*direction).detach()\n        return (plus-minus)/(2*float(step))\n\n    def fd(self, direction, step, *, at=None):\n        """Alias for central_fd; step multiplies the complete supplied direction."""\n        return self.central_fd(direction, step, at=at)\n\n    def validate_jvp(self, direction, step, *, at=None, rtol=1e-2, atol=1e-5):\n        """Explicit diagnostic at a supplied step; no adaptive epsilon search."""\n        for value in (rtol, atol):\n            if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value) or value < 0:\n                raise ValueError(\'Validation tolerances must be finite and nonnegative\')\n        product = self.jvp(direction, at=at)\n        finite_difference = self.central_fd(direction, step, at=at)\n        error = (product-finite_difference).double().norm()\n        scale = max(float(product.double().norm()), float(finite_difference.double().norm()), 1e-12)\n        return {\'passed\': bool(torch.allclose(product, finite_difference, rtol=rtol, atol=atol)),\n                \'absolute_l2_error\': float(error), \'relative_l2_error\': float(error)/scale,\n                \'jvp_norm\': float(product.double().norm()), \'fd_norm\': float(finite_difference.double().norm()),\n                \'step\': float(step), \'rtol\': float(rtol), \'atol\': float(atol),\n                \'source_block\': self.source_block, \'target_block\': self.target_block,\n                \'position\': self.position, \'scope\': \'original-prompt last-position to same-position residual derivative\'}\n\n\ndef build_residual_map(model, input_ids, source_block, target_block, *, attention_mask=None):\n    return ResidualMap(model, input_ids, source_block, target_block, attention_mask=attention_mask)\n', 'attention_probe/local_transport.py': '"""Matrix-free Tikhonov transport for one fixed local residual-map Jacobian.\n\nFor J = dF(x0)/dx, solve (J J^T + lambda^2 I) u = delta_target with\nbounded conjugate gradients, then return delta_source = J^T u. JVP/VJP must\nrefer to the SAME native map and linearization point (e.g. context-local\npost-block14 -> post-block18), not an averaged/global Jacobian lens.\n\nThe regularization principle matches Burny\'s jtransport/transport.py::ridge_inverse\nin jacobian-mem2gen. Reference: P. C. Hansen, *Rank-Deficient and Discrete\nIll-Posed Problems*, SIAM, 1998, ch.4; https://doi.org/10.1137/1.9780898719697.\nUnlike that SVD implementation this module forms no full Jacobian or inverse,\nestimates no singular values, and makes no sigma_min/conditioning guarantee.\n\nlambda_ is an explicit positive absolute value. A caller may prespecify\nlambda_ = c * estimated_gain and must record c and the gain-estimation rule;\nthis solver never chooses damping from answers or convergence outcomes.\nConvergence refers to the damped dual equation, not exact target reconstruction\nor nonlinear endpoint agreement. The latter require separate measurements.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom collections.abc import Callable\nfrom dataclasses import dataclass\nfrom numbers import Integral, Real\nfrom typing import Any\n\nimport torch\n\nMAX_CG_ITERATIONS = 16\n\n\n@dataclass(frozen=True)\nclass TransportResult:\n    """Detached native-dtype source displacement plus finite JSON diagnostics."""\n\n    delta_source: torch.Tensor\n    diagnostics: dict[str, Any]\n\n\ndef _positive_real(value: Real, name: str, *, below_one: bool = False) -> float:\n    if isinstance(value, bool) or not isinstance(value, Real):\n        raise TypeError(f"{name} must be an explicit real scalar")\n    result = float(value)\n    if not math.isfinite(result) or result <= 0 or (below_one and result >= 1):\n        raise ValueError(\n            f"{name} must be finite, positive" + (" and below one" if below_one else "")\n        )\n    return result\n\n\ndef _norm64(value: torch.Tensor) -> float:\n    """Scale first, so tiny finite FP32 displacements do not square to zero."""\n    flat = value.detach().reshape(-1).to(torch.float64)\n    scale = float(flat.abs().max().item())\n    if scale == 0:\n        return 0.0\n    result = float(torch.linalg.vector_norm(flat / scale).item()) * scale\n    if not math.isfinite(result):\n        raise FloatingPointError("Vector norm exceeds finite FP64 diagnostic range")\n    return result\n\n\ndef ridge_transport(\n    jvp: Callable[[torch.Tensor], torch.Tensor],\n    vjp: Callable[[torch.Tensor], torch.Tensor],\n    delta_target: torch.Tensor,\n    lambda_: float,\n    *,\n    max_iter: int = MAX_CG_ITERATIONS,\n    rtol: float = 1e-5,\n) -> TransportResult:\n    """Return J^T (J J^T + lambda_^2 I)^-1 delta_target approximately.\n\n    The right-hand side is internally normalized, then the solution is rescaled;\n    this changes no mathematical objective and avoids tiny-RHS CG underflow.\n    Operator vectors retain the target\'s FP32/FP64 dtype/device and exact shape;\n    scalar reductions use FP64. Source and target may have different shapes.\n    Callables receive detached copies and must return finite native tensors.\n\n    max_iter is in [1,16]; there are at most max_iter dual CG products plus one\n    final VJP and one reconstruction JVP. JVP(0) is skipped analytically. A zero\n    target performs no CG/JVP and one VJP(0) to discover the source shape.\n    Nonconvergence/breakdown returns the last finite iterate, explicitly flagged.\n    Invalid/nonfinite callback data raises instead of publishing NaN tensors.\n    """\n    if not callable(jvp) or not callable(vjp):\n        raise TypeError("jvp and vjp must be callables bound to one local Jacobian")\n    if not isinstance(delta_target, torch.Tensor):\n        raise TypeError("delta_target must be a torch.Tensor")\n    if delta_target.dtype not in (torch.float32, torch.float64):\n        raise ValueError("delta_target must use native FP32 or FP64 real arithmetic")\n    if delta_target.numel() == 0 or not bool(torch.isfinite(delta_target).all().item()):\n        raise ValueError("delta_target must be nonempty and finite")\n    if isinstance(max_iter, bool) or not isinstance(max_iter, Integral):\n        raise TypeError("max_iter must be an integer")\n    if not 1 <= max_iter <= MAX_CG_ITERATIONS:\n        raise ValueError("max_iter must be between 1 and16")\n    lam = _positive_real(lambda_, "lambda_")\n    tolerance = _positive_real(rtol, "rtol", below_one=True)\n    squared = lam * lam\n    if not math.isfinite(squared):\n        raise ValueError("lambda_^2 must be finite")\n    y = delta_target.detach().clone()\n    damping = torch.tensor(squared, dtype=y.dtype, device=y.device)\n    if not bool(torch.isfinite(damping).item()) or float(damping.item()) <= 0:\n        raise ValueError("lambda_^2 overflows or underflows the native dtype")\n    damping_value = float(damping.item())\n    target_norm = _norm64(y)\n    source_shape: tuple[int, ...] | None = None\n    counts = {\n        "jvp_calls": 0,\n        "vjp_calls": 0,\n        "cg_operator_calls": 0,\n        "zero_jvp_shortcuts": 0,\n    }\n\n    def checked(value, name, shape=None):\n        if not isinstance(value, torch.Tensor):\n            raise TypeError(f"{name} must return a torch.Tensor")\n        if value.dtype != y.dtype or value.device != y.device:\n            raise ValueError(f"{name} must preserve native dtype/device")\n        if value.numel() == 0 or (shape is not None and tuple(value.shape) != shape):\n            raise ValueError(f"{name} returned an unexpected shape")\n        if not bool(torch.isfinite(value).all().item()):\n            raise ValueError(f"{name} returned nonfinite values")\n        return value.detach().clone()\n\n    def transpose(vector):\n        nonlocal source_shape\n        counts["vjp_calls"] += 1\n        value = checked(vjp(vector.detach().clone()), "vjp", source_shape)\n        if source_shape is None:\n            source_shape = tuple(value.shape)\n        return value\n\n    def forward(vector):\n        if not bool(torch.count_nonzero(vector).item()):\n            counts["zero_jvp_shortcuts"] += 1\n            return torch.zeros_like(y)\n        counts["jvp_calls"] += 1\n        return checked(jvp(vector.detach().clone()), "jvp", tuple(y.shape))\n\n    def dual_product(vector):\n        counts["cg_operator_calls"] += 1\n        return checked(\n            forward(transpose(vector)) + damping * vector,\n            "dual operator",\n            tuple(y.shape),\n        )\n\n    def inner(left, right):\n        value = float(\n            torch.dot(left.reshape(-1).double(), right.reshape(-1).double()).item()\n        )\n        if not math.isfinite(value):\n            raise FloatingPointError("Nonfinite FP64 CG reduction")\n        return value\n\n    def finish(solution, iterations, termination, history, *, zero_target=False):\n        if zero_target:\n            source = transpose(torch.zeros_like(y))\n            if bool(torch.count_nonzero(source).item()):\n                raise ValueError("vjp(0) must be zero for a fixed linearized operator")\n            delta = torch.zeros_like(source)\n            reconstruction = torch.zeros_like(y)\n            dual64 = torch.zeros_like(y, dtype=torch.float64)\n        else:\n            source = transpose(solution)\n            delta = checked(\n                (source.double() * target_norm).to(y.dtype), "rescaled source"\n            )\n            reconstruction = forward(delta)\n            dual64 = solution.double() * target_norm\n        reconstruction_error = reconstruction.double() - y.double()\n        dual_residual = y.double() - reconstruction.double() - damping_value * dual64\n        residual_norm, reconstruction_norm = (\n            _norm64(dual_residual),\n            _norm64(reconstruction_error),\n        )\n        residual_relative = residual_norm / target_norm if target_norm else 0.0\n        reconstruction_relative = (\n            reconstruction_norm / target_norm if target_norm else 0.0\n        )\n        source_norm = _norm64(delta)\n        normalized_objective = (\n            (\n                reconstruction_relative**2\n                + damping_value * (source_norm / target_norm) ** 2\n            )\n            if target_norm\n            else 0.0\n        )\n        if not all(\n            math.isfinite(v)\n            for v in (residual_relative, reconstruction_relative, normalized_objective)\n        ):\n            raise FloatingPointError("Nonfinite final transport diagnostics")\n        converged = residual_relative <= tolerance\n        breakdown = (\n            termination\n            if termination\n            in ("nonpositive_curvature", "nonfinite_step", "nonfinite_iterate")\n            else None\n        )\n        if breakdown:\n            status = "breakdown"\n            converged = False\n        elif converged:\n            status = "converged"\n        else:\n            status = "not_converged"\n            if termination == "recursive_tolerance":\n                termination = "recursive_tolerance_not_confirmed"\n        diagnostics = {\n            "schema": "local_tikhonov_transport_v1",\n            "method": "matrix_free_dual_conjugate_gradients",\n            "status": status,\n            "converged": converged,\n            "termination": termination,\n            "breakdown": breakdown,\n            "iterations": iterations,\n            "max_iter": int(max_iter),\n            "rtol": tolerance,\n            "lambda": lam,\n            "lambda_squared_native": damping_value,\n            "target_norm": target_norm,\n            "source_norm": source_norm,\n            "dual_solution_norm": _norm64(dual64),\n            "dual_residual_norm": residual_norm,\n            "dual_relative_residual": residual_relative,\n            "linear_reconstruction_error_norm": reconstruction_norm,\n            "linear_reconstruction_relative_error": reconstruction_relative,\n            "normalized_tikhonov_objective": normalized_objective,\n            "recursive_relative_residual_history": history,\n            "rhs_normalization": "unit L2 internally, final solution rescaled",\n            "zero_target": zero_target,\n            "target_shape": list(y.shape),\n            "source_shape": list(delta.shape),\n            "dtype": str(y.dtype),\n            "device": str(y.device),\n            **counts,\n            "full_jacobian_formed": False,\n            "singular_values_computed": False,\n            "convergence_scope": "damped local linear dual equation; not nonlinear endpoint agreement",\n        }\n        return TransportResult(delta.detach(), diagnostics)\n\n    if target_norm == 0:\n        return finish(torch.zeros_like(y), 0, "zero_target", [0.0], zero_target=True)\n    rhs = (y.double() / target_norm).to(y.dtype)\n    rhs_norm = _norm64(rhs)\n    solution, residual = torch.zeros_like(rhs), rhs.clone()\n    direction = residual.clone()\n    rr = inner(residual, residual)\n    history = [1.0]\n    iterations, termination = 0, "iteration_limit"\n    for _ in range(int(max_iter)):\n        product = dual_product(direction)\n        curvature = inner(direction, product)\n        if curvature <= 0:\n            termination = "nonpositive_curvature"\n            break\n        alpha = rr / curvature\n        if not math.isfinite(alpha):\n            termination = "nonfinite_step"\n            break\n        next_solution = solution + alpha * direction\n        next_residual = residual - alpha * product\n        if not bool(torch.isfinite(next_solution).all().item()) or not bool(\n            torch.isfinite(next_residual).all().item()\n        ):\n            termination = "nonfinite_iterate"\n            break\n        next_rr = inner(next_residual, next_residual)\n        solution, residual = next_solution, next_residual\n        iterations += 1\n        relative = math.sqrt(next_rr) / rhs_norm\n        history.append(relative)\n        if relative <= tolerance:\n            termination = "recursive_tolerance"\n            break\n        beta = next_rr / rr\n        direction = residual + beta * direction\n        if not bool(torch.isfinite(direction).all().item()):\n            termination = "nonfinite_iterate"\n            break\n        rr = next_rr\n    return finish(solution, iterations, termination, history)\n', 'attention_probe/result_bundle.py': '"""Lossless state summary of a completed/diagnostic ZIP, never a resume archive.\n\nOriginal JSON/log bytes stay unchanged. Their NPZ/logit hashes still name the\nfull archive; transformed state NPZs receive different filenames and their own\nhashes. Full-vocabulary KL remains a recorded metric once logits are omitted.\nOnly the existing local ZIP is read; this module uses no model or network.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport io\nimport json\nimport os\nimport re\nimport stat\nimport tempfile\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\n\nARCHIVE_KIND = "analysis_summary_not_resume"\nPACKAGE_FILE = "COMPACT_PACKAGE.json"\nMANIFEST_FILE = "COMPACT_MANIFEST.json"\nFULL_MANIFEST_FILE = "full_archive_manifest.json"\n\n\ndef _sha(raw: bytes) -> str:\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef _file_sha(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _json(value) -> bytes:\n    return (\n        json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False)\n        + "\\n"\n    ).encode()\n\n\ndef _vocab_sizes(document):\n    sizes = set()\n    if isinstance(document, dict):\n        for key, value in document.items():\n            if (\n                key == "vocab_size"\n                and isinstance(value, int)\n                and not isinstance(value, bool)\n                and value > 0\n            ):\n                sizes.add(value)\n            sizes.update(_vocab_sizes(value))\n    elif isinstance(document, list):\n        for value in document:\n            sizes.update(_vocab_sizes(value))\n    return sizes\n\n\ndef _array_metadata(array: np.ndarray) -> dict:\n    return {\n        "dtype": str(array.dtype),\n        "shape": list(array.shape),\n        "nbytes": int(array.nbytes),\n        "sha256": _sha(array.tobytes(order="C")),\n        "hash_order": "C contiguous raw bytes",\n    }\n\n\ndef _filtered_npz(raw: bytes, vocab_sizes: set[int]):\n    included, omitted, states = {}, {}, {}\n    with np.load(io.BytesIO(raw), allow_pickle=False) as archive:\n        if len(archive.files) != len(set(archive.files)):\n            raise ValueError("Duplicate NPZ array keys")\n        for key in archive.files:\n            array = archive[key]  # Object arrays are rejected by allow_pickle=False.\n            metadata = _array_metadata(array)\n            name = key.lower()\n            named_logits = re.search(r"(?:^|[/._])(?:full_)?logits$", name) is not None\n            vocabulary_vector = (\n                "logits" in name and array.ndim == 1 and array.shape[0] in vocab_sizes\n            )\n            if named_logits or vocabulary_vector:\n                omitted[key] = metadata\n            else:\n                included[key] = metadata\n                states[key] = array\n    output = io.BytesIO()\n    np.savez_compressed(output, **states)\n    return output.getvalue(), included, omitted\n\n\ndef build_compact_result(full_zip: Path, compact_zip: Path) -> dict:\n    """Atomically create a separately hashed analysis ZIP, preserving the full ZIP.\n\n    A malformed input raises and leaves an existing compact destination intact.\n    The caller can then offer the full archive instead. All arrays retained in\n    state NPZs keep their exact dtype/shape/values; there is no FP16 conversion.\n    Excluded logits have raw-C-byte SHA256, dtype, shape and nbytes receipts.\n    """\n    full, compact = Path(full_zip), Path(compact_zip)\n    if full.resolve() == compact.resolve() or (\n        compact.exists() and os.path.samefile(full, compact)\n    ):\n        raise ValueError("Compact output must not overwrite the full result archive")\n    full_sha, full_bytes = _file_sha(full), full.stat().st_size\n    payload, transforms, vocab_sizes = {}, [], set()\n    plan_sha = plan_file_sha = None\n    with zipfile.ZipFile(full) as archive:\n        members = archive.infolist()\n        names = [member.filename for member in members]\n        if len(names) != len(set(names)):\n            raise ValueError("Duplicate ZIP members")\n        if any(\n            name in names for name in (PACKAGE_FILE, MANIFEST_FILE, FULL_MANIFEST_FILE)\n        ):\n            raise ValueError(\n                "Input is already a compact analysis summary or uses reserved names"\n            )\n        for member in members:\n            path = PurePosixPath(member.filename)\n            if (\n                path.is_absolute()\n                or member.filename.rstrip("/") != str(path)\n                or ".." in path.parts\n                or "\\\\" in member.filename\n                or member.flag_bits & 1\n                or stat.S_ISLNK(member.external_attr >> 16)\n            ):\n                raise ValueError("Unsafe, encrypted or noncanonical ZIP member")\n        if archive.testzip() is not None:\n            raise ValueError("Full ZIP CRC verification failed")\n        # Inspect metadata, preserving the exact original bytes when copied later.\n        for member in members:\n            if member.is_dir() or not member.filename.endswith(".json"):\n                continue\n            raw = archive.read(member)\n            document = json.loads(raw)\n            vocab_sizes.update(_vocab_sizes(document))\n            if member.filename in ("run_plan.json", "plan.json") and plan_sha is None:\n                canonical = json.dumps(\n                    document,\n                    sort_keys=True,\n                    separators=(",", ":"),\n                    ensure_ascii=False,\n                    allow_nan=False,\n                ).encode()\n                plan_sha, plan_file_sha = _sha(canonical), _sha(raw)\n        for member in members:\n            if member.is_dir():\n                continue\n            name, raw = member.filename, archive.read(member)\n            transformed = name\n            if name.endswith(".npz"):\n                original_sha = _sha(raw)\n                original_bytes = len(raw)\n                raw, included, excluded = _filtered_npz(raw, vocab_sizes)\n                path = PurePosixPath(name)\n                basename = (\n                    "state_arrays.npz"\n                    if path.name == "arrays.npz"\n                    else path.stem + "_state_arrays.npz"\n                )\n                transformed = str(path.with_name(basename))\n                transforms.append(\n                    {\n                        "original_path": name,\n                        "original_sha256": original_sha,\n                        "original_bytes": original_bytes,\n                        "transformed_path": transformed,\n                        "transformed_sha256": _sha(raw),\n                        "transformed_bytes": len(raw),\n                        "included_keys": list(included),\n                        "excluded_keys": list(excluded),\n                        "included_arrays": included,\n                        "excluded_arrays": excluded,\n                    }\n                )\n            elif name == "WRAPPER_MANIFEST.json":\n                transformed = FULL_MANIFEST_FILE\n            if transformed in payload:\n                raise ValueError("Transformed ZIP member collision")\n            payload[transformed] = raw\n    included_keys = [key for row in transforms for key in row["included_keys"]]\n    available = {\n        "state_geometry": bool(included_keys),\n        "jvp_error": any(\n            any(\n                word in key.lower()\n                for word in ("jvp", "fd", "transport", "jacobian", "jac_")\n            )\n            for key in included_keys\n        ),\n        "full_vocab_KL": False,\n    }\n    metadata = {\n        "schema": "compact_result_v1",\n        "archive_kind": ARCHIVE_KIND,\n        "resume_eligible": False,\n        "full_archive": {\n            "filename": full.name,\n            "sha256": full_sha,\n            "bytes": full_bytes,\n        },\n        "full_archive_payload_included": False,\n        "npz_transforms": transforms,\n        "plan_sha256": plan_sha,\n        "plan_file_sha256": plan_file_sha,\n        "plan_canonicalization": "sorted compact UTF8 JSON, ensure_ascii=False",\n        "metrics_source": "runtime_recorded_fp64",\n        "independent_recompute_available": available,\n        "availability_scope": "Only corresponding preserved measurements are available; no new measurement is implied.",\n        "original_record_hash_scope": "Unchanged JSON NPZ/logit hashes and full_archive_manifest.json refer to the full archive, not transformed state files.",\n        "not_resume_reason": "Full arrays.npz members are replaced by separately named state-only NPZs; omitted full-vocabulary values are not embedded.",\n    }\n    payload[PACKAGE_FILE] = _json(metadata)\n    manifest = {\n        name: {"sha256": _sha(raw), "bytes": len(raw)}\n        for name, raw in sorted(payload.items())\n    }\n    payload[MANIFEST_FILE] = _json(manifest)\n    compact.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(\n        dir=compact.parent, prefix="." + compact.name + ".", suffix=".tmp", delete=False\n    ) as stream:\n        temporary = Path(stream.name)\n    try:\n        with zipfile.ZipFile(\n            temporary, "w", zipfile.ZIP_DEFLATED, compresslevel=6\n        ) as archive:\n            for name, raw in sorted(payload.items()):\n                info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))\n                info.external_attr = 0o600 << 16\n                archive.writestr(\n                    info, raw, compress_type=zipfile.ZIP_DEFLATED, compresslevel=6\n                )\n        with zipfile.ZipFile(temporary) as archive:\n            if archive.testzip() is not None:\n                raise ValueError("Compact ZIP CRC verification failed")\n        os.replace(temporary, compact)\n    finally:\n        temporary.unlink(missing_ok=True)\n    return {\n        "archive_kind": ARCHIVE_KIND,\n        "resume_eligible": False,\n        "compact_path": str(compact),\n        "compact_sha256": _file_sha(compact),\n        "compact_bytes": compact.stat().st_size,\n        "full_archive_filename": full.name,\n        "full_archive_sha256": full_sha,\n        "full_archive_bytes": full_bytes,\n        "npz_count": len(transforms),\n        "included_array_count": len(included_keys),\n        "excluded_array_count": sum(len(row["excluded_keys"]) for row in transforms),\n        "plan_sha256": plan_sha,\n        "metrics_source": "runtime_recorded_fp64",\n        "independent_recompute_available": available,\n    }\n', 'qwen3_b14_b18_runtime.py': '"""Bounded native Qwen3 B14→B18 local-Jacobian/transport pilot.\n\nOriginal H286/REF-W and weights are read-only. This is a local, prompt-specific\nJacobian experiment, not a fitted/global JLens or a confirmatory benchmark.\nSource-bound prefixes are teacher forced; separate short free answers are saved\nfor later blind interpretation, with no online semantic correctness evaluator.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport datetime as dt\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport shutil\nimport tempfile\nimport time\nimport traceback\nimport types\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport qwen3_h286_runtime as base\n\nSCHEMA = \'qwen3_b14_b18_local_transport_v1\'\nITEMS_SHA256 = \'cc482e87fc49dcea8fa77b0ccb4bf63ed8df41e5f75c5c47c434decd4e3d1a5e\'\nINPUT_PLAN_SHA256 = \'76dcc81a06142b2fb662383ec82a6dc17834e9612543163036d2a7c278643bbf\'\nOFFSETS = (0, 1, 4, 8)\nRESIDUAL_BLOCKS = (13, 14, 15, 17, 18, 19)\nCALIBRATION_INDICES = (0, 12)\nTRANSPORT_INDICES = (0, 1, 12, 13)\nFD_STEPS = (0.25, 0.5)\nRIDGE_RELATIVE = 0.1\nCG_MAX_ITER = 16\nCG_RTOL = 1e-4\nREQUEST_LIMIT_SECONDS = 45 * 60\nSOURCE_NAMES = (\'qwen3_h286_runtime.py\', \'attention_probe/reference.py\',\n                \'attention_probe/fixed_refw.py\', \'attention_probe/interventions.py\',\n                \'attention_probe/answer_validation.py\', \'attention_probe/native_jacobian.py\',\n                \'attention_probe/local_transport.py\')\nARMS = ({\'name\': \'clean\', \'block\': 14, \'beta\': 1., \'mode\': \'scale\'},) + tuple(\n    {\'name\': f\'b{block}_soft_{level}\', \'block\': block, \'beta\': 1 / divisor, \'mode\': \'scale\'}\n    for block in (13, 14, 15) for level, divisor in ((\'103\', 1.03), (\'108\', 1.08)))\nPROOF_KEYS = (\'sham_exact\', \'pulse_once\', \'captures_complete\', \'upstream_causal\',\n              \'frozen_parameters\', \'global_rng\', \'hook_cleanup\', \'jacobian_finite\',\n              \'generation_prefill_exact\')\nGENERATION_ARMS = tuple(arm for arm in ARMS if arm[\'name\'] == \'clean\' or arm[\'name\'].endswith(\'108\'))\nGENERATION_CAP = 64\n\n\ndef source_path(name):\n    folder = Path(__file__).parent\n    candidate = folder / name\n    return candidate if candidate.exists() else folder.parent / name\n\n\ndef load_inputs(hn, policy, refw, items, plan_path):\n    from attention_probe.fixed_refw import load_fixed_refw\n    from attention_probe.reference import load_reference\n    reference = load_reference(hn, policy)\n    stats = load_fixed_refw(refw, reference)\n    raw = Path(items).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != ITEMS_SHA256:\n        raise ValueError(\'Exact source-bound sixteen-item private input file required\')\n    selected = json.loads(raw)\n    if (not isinstance(selected, list) or len(selected) != 16\n            or len({row[\'id\'] for row in selected}) != 16\n            or [row[\'family\'] for row in selected] != [\'s3\'] * 12 + [\'adet\'] * 4):\n        raise ValueError(\'Expected fixed 12 S3 plus 4 A-det source-bound inputs\')\n    for row in selected:\n        ids = row.get(\'prompt_ids\')\n        if (not isinstance(ids, list) or not 1 <= len(ids) <= 64\n                or any(type(token) is not int or token < 0 for token in ids)\n                or not isinstance(row.get(\'fixed_continuation_text\'), str)\n                or not row[\'fixed_continuation_text\']):\n            raise ValueError(\'Malformed source prompt or fixed continuation; no truncation/padding\')\n    raw_plan = Path(plan_path).read_bytes()\n    if hashlib.sha256(raw_plan).hexdigest() != INPUT_PLAN_SHA256:\n        raise ValueError(\'Exact prespecified private scientific plan required\')\n    scientific_plan = json.loads(raw_plan)\n    return reference, stats, selected, scientific_plan\n\n\ndef plan_document(reference, stats, items, scientific_plan):\n    return {\'schema\': SCHEMA, \'helper_sha256\': base.sha_file(__file__),\n        \'source_sha256\': {name: base.sha_file(source_path(name)) for name in SOURCE_NAMES},\n        \'items_sha256\': ITEMS_SHA256, \'items\': items, \'model\': reference.model.to_dict(),\n        \'input_plan_sha256\': INPUT_PLAN_SHA256, \'scientific_plan\': scientific_plan,\n        \'hn_sha256\': reference.hn_sha256, \'policy_sha256\': reference.policy_sha256,\n        \'refw_sha256\': base.REFW_SHA256, \'refw_source_sha256\': stats[\'source_hn_meta_sha256\'],\n        \'coordinates\': [list(pair) for pair in reference.coordinates], \'seed\': base.SEED,\n        \'dtype\': \'float32\', \'tf32\': False, \'transformers\': base.TRANSFORMERS,\n        \'accelerate\': base.ACCELERATE, \'measurement_attention\': \'sdpa/EFFICIENT_ATTENTION\',\n        \'derivative_attention\': \'sdpa/MATH\', \'arms\': list(ARMS), \'offsets\': list(OFFSETS),\n        \'prefix\': \'same source-bound GOLD continuation per arm; unavailable offsets not_run; no padding\',\n        \'pulse_position\': \'original final prompt token only, even on extended prefixes\',\n        \'capture_position\': \'final token of each matched prompt+gold-prefix\',\n        \'residuals\': {\'kind\': \'native decoder resid_post, zero-based\', \'blocks\': list(RESIDUAL_BLOCKS)},\n        \'jacobian\': {\'source\': 14, \'target\': 18, \'position\': \'final original prompt token\',\n                     \'arm\': \'b14_soft_108\', \'indices\': list(range(16)), \'full_fitted_jlens\': False},\n        \'bounded_counts\': {\'matched_prefix_measurement_slots_upper_bound\':448,\n                           \'secondary_free_generations\':64,\'jvp_requests\':16,\'transport_requests\':4,\n                           \'derivative_calls\':\'native suffix evaluations; not counted as full model prefills\'},\n        \'calibration\': {\'indices\': list(CALIBRATION_INDICES), \'fd_relative_steps\': list(FD_STEPS),\n                        \'math_efficient_rtol\': 1e-3, \'math_efficient_atol\': 1e-4},\n        \'transport\': {\'indices\': list(TRANSPORT_INDICES), \'lambda_relative_to_power_gain_estimate\': RIDGE_RELATIVE,\n                      \'gain_estimator\': \'four fixed-seed J^T J power iterations\',\n                      \'max_iter\': CG_MAX_ITER, \'rtol\': CG_RTOL,\n                      \'equation\': \'delta14=J^T (J J^T + lambda^2 I)^-1 delta18\',\n                      \'controls\': [\'direct_delta18_at14\', \'source_norm_matched_fixed_noise_at14\']},\n        \'b18_controls\': [\'jvp_direction_projection_replay\', \'jvp_direction_projection_removal\',\n                         \'projection_norm_matched_fixed_noise\', \'full_delta18_replay_cut_control\',\n                         \'full_delta18_removal_cut_control\'],\n        \'free_generation\': {\'arm_names\': [arm[\'name\'] for arm in GENERATION_ARMS],\n                            \'max_selected_tokens_including_EOS\': GENERATION_CAP, \'planned_count\': 64,\n                            \'scoring\': \'blind later; no online semantic evaluator\',\n                            \'pulse\': \'original prompt prefill only; fresh native KV cache per arm\'},\n        \'budget\': {\'seconds_after_model_load\': REQUEST_LIMIT_SECONDS,\n                   \'soft_boundary\': \'between complete input checkpoints; atomic input may overrun\',\n                   \'pause_if_next_input_estimate_exceeds_remaining_budget\': True,\n                   \'next_input_estimate\':\'maximum of last three complete-input durations; not total remaining ETA\'},\n        \'claim\': \'exploratory native local delta transport; no training, no observer proof, no global JLens\'}\n\n\ndef hook_count(model):\n    return sum(len(module._forward_hooks) + len(module._forward_pre_hooks) for module in model.modules())\n\n\ndef attention_context(torch, backend):\n    return torch.nn.attention.sdpa_kernel(getattr(torch.nn.attention.SDPBackend, backend))\n\n\ndef digest_tensor(tensor):\n    return hashlib.sha256(tensor.detach().float().cpu().contiguous().numpy().tobytes()).hexdigest()\n\n\ndef prepare_request(tokenizer, index, item, vocab_size):\n    prompt = list(item[\'prompt_ids\'])\n    continuation = tokenizer(item[\'fixed_continuation_text\'], add_special_tokens=False)[\'input_ids\']\n    if not continuation or len(continuation) > 32:\n        raise ValueError(\'Fixed continuation must have 1–32 tokens; no silent truncation\')\n    if any(type(token) is not int or not 0 <= token < vocab_size for token in prompt + continuation):\n        raise ValueError(\'Source/continuation token outside the pinned vocabulary\')\n    return {\'request_id\': f\'b14b18_{index:03d}\', \'index\': index, \'item\': item,\n        \'prompt_ids\': prompt, \'continuation_ids\': continuation,\n        \'continuation_text\': item[\'fixed_continuation_text\'],\n        \'pulse_positions\': [len(prompt) - 1],\n        \'available_offsets\': [offset for offset in OFFSETS if offset <= len(continuation)],\n        \'missing_offsets\': [offset for offset in OFFSETS if offset > len(continuation)],\n        \'thinking_enabled\': False, \'tokenization\': \'original source prompt IDs unchanged; fixed gold continuation only\'}\n\n\n@contextlib.contextmanager\ndef residual_patch(model, block, position, delta):\n    """One full-forward additive resid_post patch; other tokens remain exact."""\n    import torch\n    if not isinstance(delta, torch.Tensor) or tuple(delta.shape) != (model.config.hidden_size,):\n        raise ValueError(\'Residual delta must have the exact [hidden_size] shape\')\n    if not bool(torch.isfinite(delta).all()):\n        raise ValueError(\'Nonfinite residual delta\')\n    state = {\'fired\': 0}\n    def patch(_module, _args, output):\n        value = output[0] if isinstance(output, tuple) else output\n        if state[\'fired\'] or value.ndim != 3 or value.shape[0] != 1 or not 0 <= position < value.shape[1]:\n            raise ValueError(\'Residual patch must fire exactly once on one complete sequence\')\n        before = value[0, position].detach().clone()\n        changed = value.clone()\n        changed[0, position] = value[0, position] + delta.to(device=value.device, dtype=value.dtype)\n        state.update(fired=1, requested_delta_norm=float(delta.detach().double().norm()),\n                     actual_delta_norm=float((changed[0, position] - before).detach().double().norm()))\n        return (changed, *output[1:]) if isinstance(output, tuple) else changed\n    handle = model.model.layers[block].register_forward_hook(patch)\n    try:\n        yield state\n        if state[\'fired\'] != 1:\n            raise ValueError(\'Residual patch did not fire\')\n    finally:\n        handle.remove()\n\n\ndef vector_metrics(actual, predicted):\n    import torch\n    a, p = actual.detach().double().cpu(), predicted.detach().double().cpu()\n    an, pn = float(a.norm()), float(p.norm())\n    return {\'actual_norm\': an, \'predicted_norm\': pn, \'error_norm\': float((a-p).norm()),\n            \'relative_error\': float((a-p).norm()) / an if an else None,\n            \'cosine\': float(torch.dot(a, p) / (an*pn)) if an and pn else None}\n\n\ndef fixed_noise_like(vector, seed):\n    import torch\n    norm = float(vector.detach().double().norm())\n    generator = torch.Generator(device=\'cpu\').manual_seed(seed)\n    direction = torch.randn(vector.shape, dtype=torch.float64, generator=generator)\n    direction = direction / direction.norm() * norm if norm else torch.zeros_like(direction)\n    return direction.to(device=vector.device, dtype=vector.dtype)\n\n\ndef power_gain(mapping, width, device, seed):\n    """Fixed four-step directional spectral estimate, independent of outcomes."""\n    import torch\n    direction = fixed_noise_like(torch.ones(width, device=device, dtype=torch.float32), seed)\n    direction /= direction.norm()\n    iterations = []\n    for _ in range(4):\n        image = mapping.jvp(direction)\n        transpose = mapping.vjp(image)\n        gain, norm = float(image.double().norm()), float(transpose.double().norm())\n        iterations.append({\'gain\': gain, \'transpose_norm\': norm})\n        if not math.isfinite(gain) or not math.isfinite(norm) or gain == 0 or norm == 0:\n            return 0., iterations\n        direction = transpose / norm\n    return gain, iterations\n\n\ndef generation_runner():\n    # Isolated globals retain the old decoder byte-for-byte. Only its explicit\n    # cap guard changes; the original module still rejects >48 tokens.\n    from attention_probe import answer_validation\n    namespace = dict(vars(answer_validation))\n    namespace[\'MAX_NEW_TOKENS\'] = GENERATION_CAP\n    original = answer_validation.run_generation_arm\n    isolated = types.FunctionType(original.__code__, namespace, original.__name__, original.__defaults__, original.__closure__)\n    isolated.__kwdefaults__ = dict(original.__kwdefaults__ or {})\n    return isolated\n\n\ndef logits_metrics(logits, baseline, next_token=None):\n    current, clean = logits.detach().double().cpu().log_softmax(-1), baseline.detach().double().cpu().log_softmax(-1)\n    values, indices = current.topk(min(10, current.numel()))\n    return {\'logits_sha256\': digest_tensor(logits), \'metric_dtype\': \'float64\',\n        \'top10\': [{\'token_id\': int(i), \'logprob\': float(v)} for i, v in zip(indices, values, strict=True)],\n        \'clean_to_arm_KL\': float((clean.exp()*(clean-current)).sum()),\n        \'arm_to_clean_KL\': float((current.exp()*(current-clean)).sum()),\n        \'next_fixed_token_id\': next_token,\n        \'next_fixed_token_logprob\': float(current[next_token]) if next_token is not None else None,\n        \'next_fixed_token_scope\': \'source-bound gold next token; unavailable after final supplied token\'}\n\n\ndef forward_probe(model, request, offset, coordinates, stats, arm, *, backend=\'EFFICIENT_ATTENTION\',\n                  weight_norms=None, patch=None, capture_blocks=RESIDUAL_BLOCKS):\n    import torch\n\n    from attention_probe.interventions import AttentionPulse, CoordinateCapture\n    prompt = request[\'prompt_ids\'] + request[\'continuation_ids\'][:offset]\n    if patch is not None and offset != 0:\n        raise ValueError(\'Local residual-map patches are defined at original prompt offset zero only\')\n    position = len(prompt)-1\n    ids = torch.tensor([prompt], dtype=torch.long, device=next(model.parameters()).device)\n    before = hook_count(model)\n    with (torch.no_grad(), attention_context(torch, backend), contextlib.ExitStack() as contexts):\n        patched = contexts.enter_context(residual_patch(model, patch[\'block\'], position, patch[\'delta\'])) if patch else None\n        capture = contexts.enter_context(CoordinateCapture(model, coordinates, [position],\n            residual_layers=tuple(capture_blocks), expected_count=len(coordinates)))\n        cett = contexts.enter_context(base.cett_tap(model, coordinates, [position], weight_norms))\n        pulse = contexts.enter_context(AttentionPulse(model, arm[\'block\'], request[\'pulse_positions\'], arm[\'beta\'],\n            arm[\'mode\'], seed=base.SEED+request[\'index\']))\n        output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)\n        logits = output.logits[0, -1].detach().float().cpu()\n        mu = torch.tensor(stats[\'refw\'][\'c_mu\'], dtype=torch.float32)\n        sd = torch.tensor(stats[\'refw\'][\'c_sd\'], dtype=torch.float32).clamp_min(1e-6)\n        arrays = {\'raw_z\': capture.values.numpy(), \'cett\': cett[\'cett\'].numpy(),\n                  \'zcett\': ((cett[\'cett\']-mu)/sd).numpy(), \'down_output_norm_raw\': cett[\'denominator\'].numpy()}\n        arrays.update({f\'residual_{block}\': value.numpy() for block,value in capture.residuals.items()})\n        evidence = {\'pulse_fired\': pulse.fired_count, \'capture_fired\': capture.fired_count,\n            \'query_requested_delta_norms\': pulse.requested_delta_norms.tolist(),\n            \'query_actual_delta_norms\': pulse.actual_delta_norms.tolist(),\n            \'pulse_positions\': request[\'pulse_positions\'], \'capture_positions\': [position],\n            \'input_ids_sha256\': hashlib.sha256(base.canonical(prompt)).hexdigest(), \'backend\': backend,\n            \'patch\': dict(patched) if patched else None}\n    if hook_count(model) != before:\n        raise ValueError(\'Forward hook cleanup failed\')\n    if not bool(torch.isfinite(logits).all()) or evidence[\'pulse_fired\'] != 1 or evidence[\'capture_fired\'] != 1:\n        raise ValueError(\'Incomplete/nonfinite pulse or capture\')\n    import numpy as np\n    if any(not np.isfinite(value).all() for value in arrays.values()):\n        raise ValueError(\'Nonfinite fixed-coordinate or residual capture\')\n    return {\'logits\': logits, \'arrays\': arrays, \'evidence\': evidence}\n\n\ndef tensor_residual(probe, block, device):\n    import torch\n    return torch.from_numpy(probe[\'arrays\'][f\'residual_{block}\'][0,0]).to(device)\n\n\ndef save_probe(arrays, prefix, probe):\n    arrays.update({prefix+\'_\'+key: value for key,value in probe[\'arrays\'].items()})\n\n\ndef compute_request(engine, reference, stats, request, *, backend=\'EFFICIENT_ATTENTION\'):\n    import numpy as np\n    import torch\n\n    from attention_probe.local_transport import ridge_transport\n    from attention_probe.native_jacobian import build_residual_map\n    model = engine[\'model\']; device = next(model.parameters()).device\n    started = time.monotonic()\n    norms = engine.get(\'weight_norms\')\n    parameters, hooks = base.parameter_evidence(model), hook_count(model)\n    cpu_rng = torch.random.get_rng_state().clone()\n    cuda_rng = torch.cuda.get_rng_state().clone() if device.type == \'cuda\' else None\n    arrays, measurements, probes = {}, [], {}\n    upstream_equal = True\n    for offset in OFFSETS:\n        if offset not in request[\'available_offsets\']:\n            measurements.append({\'offset\': offset, \'state\': \'not_run\', \'reason\': \'source continuation shorter than offset\'})\n            continue\n        baseline = None\n        for arm in ARMS:\n            probe = forward_probe(model, request, offset, reference.coordinates, stats, arm,\n                                  backend=backend, weight_norms=norms)\n            baseline = probe if baseline is None else baseline\n            if arm[\'name\'] != \'clean\':\n                upstream = [i for i,(block,_) in enumerate(reference.coordinates) if block < arm[\'block\']]\n                upstream_equal &= np.array_equal(probe[\'arrays\'][\'raw_z\'][...,upstream], baseline[\'arrays\'][\'raw_z\'][...,upstream])\n            prefix = f\'o{offset}_{arm["name"]}\'\n            save_probe(arrays, prefix, probe)\n            arrays[prefix+\'_logits\'] = probe[\'logits\'].numpy()\n            next_id = request[\'continuation_ids\'][offset] if offset < len(request[\'continuation_ids\']) else None\n            measurements.append({\'offset\': offset, \'arm\': arm[\'name\'], \'state\': \'complete\',\n                **logits_metrics(probe[\'logits\'], baseline[\'logits\'], next_id), **probe[\'evidence\']})\n            if offset == 0:\n                probes[arm[\'name\']] = probe\n    # A genuine uninstrumented forward checks that passive captures + beta=1\n    # preserve the original native forward, rather than comparing two hook runs.\n    ids = torch.tensor([request[\'prompt_ids\']], dtype=torch.long, device=device)\n    with torch.no_grad(), attention_context(torch, backend):\n        native = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False).logits[0,-1].detach().float().cpu()\n    if not torch.equal(native, probes[\'clean\'][\'logits\']):\n        raise ValueError(\'Uninstrumented/native clean and sham differ\')\n    primary_elapsed = time.monotonic()-started\n    print(f"Stage {request[\'index\']+1}/16: lokaler JVP 14→18", flush=True)\n    # Map internals always use MATH; calculate its actual pulse in MATH too.\n    math_clean = forward_probe(model, request, 0, reference.coordinates, stats, ARMS[0], backend=\'MATH\', weight_norms=norms)\n    math_pulse = forward_probe(model, request, 0, reference.coordinates, stats, ARMS[4], backend=\'MATH\', weight_norms=norms)\n    arrays[\'math_clean_logits\'] = math_clean[\'logits\'].numpy()\n    arrays[\'math_pulse_logits\'] = math_pulse[\'logits\'].numpy()\n    save_probe(arrays,\'math_clean\',math_clean)\n    save_probe(arrays,\'math_pulse\',math_pulse)\n    mapping = build_residual_map(model, ids, 14, 18, attention_mask=torch.ones_like(ids))\n    if not (torch.equal(mapping.baseline_source,tensor_residual(math_clean,14,device))\n            and torch.equal(mapping.baseline_target,tensor_residual(math_clean,18,device))):\n        raise ValueError(\'Residual-map baseline differs from the captured native MATH clean measurement\')\n    recovered = mapping(mapping.baseline_source).detach()\n    if not torch.allclose(recovered,mapping.baseline_target,rtol=1e-5,atol=1e-5):\n        raise ValueError(\'Native suffix primal baseline parity failed\')\n    source_delta = tensor_residual(math_pulse,14,device)-mapping.baseline_source\n    actual_delta = tensor_residual(math_pulse,18,device)-mapping.baseline_target\n    predicted = mapping.jvp(source_delta)\n    nonlinear = mapping(tensor_residual(math_pulse,14,device)).detach()-mapping.baseline_target\n    if not torch.allclose(nonlinear,actual_delta,rtol=1e-4,atol=1e-5):\n        raise ValueError(\'Native suffix actual-pulse primal parity failed\')\n    jacobian = {\'source\':14, \'target\':18, \'backend\':\'MATH\', \'arm\':\'b14_soft_108\',\n        \'prediction\':vector_metrics(actual_delta,predicted),\n        \'last_prompt_only\':True, \'map_source_shape\':list(mapping.baseline_source.shape),\n        \'primal_baseline_parity\':vector_metrics(mapping.baseline_target,recovered),\n        \'primal_baseline_allclose\':True,\n        \'nonlinear_actual_pulse_parity\':vector_metrics(actual_delta,nonlinear),\n        \'b15_source14_unchanged\':bool(np.array_equal(probes[\'clean\'][\'arrays\'][\'residual_14\'],\n                                                   probes[\'b15_soft_108\'][\'arrays\'][\'residual_14\']))}\n    arrays.update(jac_source_delta=source_delta.detach().cpu().numpy(),\n                  jac_target_actual=actual_delta.detach().cpu().numpy(), jac_target_predicted=predicted.detach().cpu().numpy(),\n                  jac_baseline_source=mapping.baseline_source.cpu().numpy(), jac_baseline_target=mapping.baseline_target.cpu().numpy(),\n                  jac_target_nonlinear=nonlinear.cpu().numpy())\n    calibration = {\'state\':\'not_run\',\'reason\':\'fixed calibration indices only\'}\n    if request[\'index\'] in CALIBRATION_INDICES:\n        efficient_clean = probes[\'clean\'][\'logits\']\n        delta = efficient_clean-math_clean[\'logits\']\n        parity = bool(torch.allclose(efficient_clean,math_clean[\'logits\'],rtol=1e-3,atol=1e-4))\n        calibration = {\'state\':\'complete\',\'math_efficient_logits_allclose\':parity,\n            \'math_efficient_max_abs\':float(delta.abs().max()),\n            \'math_efficient_resid18\':vector_metrics(tensor_residual(probes[\'clean\'],18,device),tensor_residual(math_clean,18,device)),\n            \'finite_differences\':[], \'fd_gate\':{\'minimum_cosine\':.99,\'maximum_relative_error\':.05,\n                \'noise_floor_factor\':64,\'minimum_absolute_floor\':1e-7}}\n        if not parity:\n            raise ValueError(\'MATH/EFFICIENT baseline parity failed before local transport\')\n        for step in FD_STEPS:\n            fd = mapping.central_fd(source_delta,step)\n            comparison = vector_metrics(fd,predicted)\n            floor = max(1e-7,64*torch.finfo(torch.float32).eps*float(mapping.baseline_target.double().norm())/step)\n            informative = comparison[\'actual_norm\'] > floor\n            passed = informative and comparison[\'relative_error\'] <= .05 and comparison[\'cosine\'] is not None and comparison[\'cosine\'] >= .99\n            calibration[\'finite_differences\'].append({\'relative_step\':step, **comparison,\n                \'estimated_FP32_noise_floor\':floor,\'informative\':informative,\'agreement_passed\':passed})\n            arrays[f\'jac_fd_{str(step).replace(".","_")}\'] = fd.detach().cpu().numpy()\n        informative = [row for row in calibration[\'finite_differences\'] if row[\'informative\']]\n        calibration[\'fd_gate_state\'] = (\'passed\' if any(row[\'agreement_passed\'] for row in informative)\n                                        else \'failed\' if informative else \'degraded_below_noise_floor\')\n        if engine.get(\'diagnostics_dir\') is not None:\n            base.atomic_json(Path(engine[\'diagnostics_dir\'])/(request[\'request_id\']+\'_calibration.json\'),calibration)\n        if calibration[\'fd_gate_state\'] != \'passed\':\n            raise ValueError(\'Local JVP finite-difference calibration \'+calibration[\'fd_gate_state\'])\n    derivative_elapsed = time.monotonic()-started-primary_elapsed\n    # Directional projection is the mechanistic diagnostic. The complete delta\n    # replay/removal is retained only as a trivial downstream cut control.\n    direction_norm = float(predicted.detach().double().norm())\n    projection = predicted * (float(torch.dot(actual_delta.double(),predicted.double()))/(direction_norm**2)) if direction_norm else torch.zeros_like(predicted)\n    noise = fixed_noise_like(projection,base.SEED+request[\'index\']+18000)\n    patch_specs = [(\'projection_replay\',ARMS[0],projection),(\'projection_removal\',ARMS[4],-projection),\n                   (\'projection_noise\',ARMS[0],noise),(\'full_replay_cut_control\',ARMS[0],actual_delta),\n                   (\'full_removal_cut_control\',ARMS[4],-actual_delta)]\n    patch_rows = []\n    for name,receiver,delta in patch_specs:\n        probe = forward_probe(model,request,0,reference.coordinates,stats,receiver,backend=\'MATH\',weight_norms=norms,\n                              patch={\'block\':18,\'delta\':delta})\n        target = tensor_residual(probe,18,device)-mapping.baseline_target\n        patch_rows.append({\'name\':name,\'receiver\':receiver[\'name\'],**probe[\'evidence\'],\n            \'target_vs_actual\':vector_metrics(actual_delta,target),\n            \'logit_references\':{\'math_clean\':logits_metrics(probe[\'logits\'],math_clean[\'logits\']),\n                                \'math_pulse\':logits_metrics(probe[\'logits\'],math_pulse[\'logits\'])}})\n        save_probe(arrays,\'patch18_\'+name,probe)\n        arrays[\'patch18_\'+name+\'_logits\'] = probe[\'logits\'].numpy()\n    controls_elapsed = time.monotonic()-started-primary_elapsed-derivative_elapsed\n    transport = {\'state\':\'not_run\',\'reason\':\'fixed transport indices only\'}\n    if request[\'index\'] in TRANSPORT_INDICES:\n        gain, gain_iterations = power_gain(mapping,model.config.hidden_size,device,base.SEED+request[\'index\']+40000)\n        if gain <= 0 or not math.isfinite(gain):\n            transport = {\'state\':\'invalid\',\'reason\':\'zero/nonfinite fixed directional gain; no resampling\'}\n        else:\n            regularizer = RIDGE_RELATIVE*gain\n            solved = ridge_transport(mapping.jvp,mapping.vjp,actual_delta,regularizer,max_iter=CG_MAX_ITER,rtol=CG_RTOL)\n            transported = solved.delta_source\n            transport = {\'state\':\'complete\',\'lambda\':regularizer,\'lambda_relative\':RIDGE_RELATIVE,\n                         \'power_gain_estimate\':gain,\'power_iterations\':gain_iterations,\n                         \'power_seed\':base.SEED+request[\'index\']+40000,\'solver\':solved.diagnostics,\'arms\':[]}\n            controls = [(\'ridge_transport\',transported),(\'direct_delta18_at14\',actual_delta),\n                        (\'source_norm_matched_noise\',fixed_noise_like(transported,base.SEED+request[\'index\']+14000))]\n            for name,delta in controls:\n                probe = forward_probe(model,request,0,reference.coordinates,stats,ARMS[0],backend=\'MATH\',weight_norms=norms,\n                                      patch={\'block\':14,\'delta\':delta})\n                achieved = tensor_residual(probe,18,device)-mapping.baseline_target\n                transport[\'arms\'].append({\'name\':name,**probe[\'evidence\'],\n                    \'amplitude_control\':(\'same raw source L2 as ridge\' if name==\'source_norm_matched_noise\'\n                                         else \'direct target delta copied without source norm matching\' if name==\'direct_delta18_at14\'\n                                         else \'regularized inverse solution\'),\n                    \'target_prediction\':vector_metrics(actual_delta,achieved),\n                    \'logit_references\':{\'math_clean\':logits_metrics(probe[\'logits\'],math_clean[\'logits\']),\n                                        \'math_pulse\':logits_metrics(probe[\'logits\'],math_pulse[\'logits\'])}})\n                save_probe(arrays,\'transport_\'+name,probe)\n                arrays[\'transport_\'+name+\'_logits\'] = probe[\'logits\'].numpy()\n                arrays[\'transport_delta_\'+name] = delta.detach().cpu().numpy()\n    transport_elapsed = time.monotonic()-started-primary_elapsed-derivative_elapsed-controls_elapsed\n    generation_started = time.monotonic()\n    generations = []\n    generation_prefill_exact = True\n    generator = generation_runner()\n    generation_request = {\'request_id\':f\'r{request["index"]:03d}\',\n                          \'prompt_ids\':request[\'prompt_ids\'], \'answer\':request[\'continuation_text\']}\n    for arm in GENERATION_ARMS:\n        with attention_context(torch,backend):\n            record, captured = generator(model,engine[\'tokenizer\'],generation_request,reference.coordinates,stats,arm,\n                efficient_gpu=device.type==\'cuda\' and backend==\'EFFICIENT_ATTENTION\',weight_norms=norms,\n                max_new_tokens=GENERATION_CAP,top_k=10)\n        exact = record[\'first_step_logits_sha256\'] == digest_tensor(probes[arm[\'name\']][\'logits\'])\n        generation_prefill_exact &= exact\n        generations.append({\'arm\':arm[\'name\'],\'prefill_equals_primary_measurement\':exact,\'record\':record,\n                            \'evaluation\':\'unscored free text for later blind analysis\'})\n        arrays.update({f\'generation_{arm["name"]}_{key}\':value for key,value in captured.items()})\n    checks = {\'sham_exact\':True,\'pulse_once\':all(row.get(\'pulse_fired\')==1 for row in measurements if row[\'state\']==\'complete\'),\n        \'captures_complete\':all(row.get(\'capture_fired\')==1 for row in measurements if row[\'state\']==\'complete\'),\n        \'upstream_causal\':bool(upstream_equal),\'frozen_parameters\':base.parameter_evidence(model)==parameters,\n        \'global_rng\':bool(torch.equal(cpu_rng,torch.random.get_rng_state()) and\n                           (cuda_rng is None or torch.equal(cuda_rng,torch.cuda.get_rng_state()))),\n        \'hook_cleanup\':hook_count(model)==hooks,\'jacobian_finite\':bool(torch.isfinite(predicted).all()),\n        \'generation_prefill_exact\':bool(generation_prefill_exact)}\n    if not all(checks.values()):\n        raise ValueError(\'Bound native/causal/RNG/parameter technical checks failed: \'+str(checks))\n    return {\'request\':request,\'measurements\':measurements,\'jacobian\':jacobian,\'calibration\':calibration,\n            \'b18_controls\':patch_rows,\'transport\':transport,\'generations\':generations,\'checks\':checks,\n            \'timing\':{\'primary_measurements_seconds\':primary_elapsed,\n                      \'jacobian_seconds\':derivative_elapsed,\'b18_controls_seconds\':controls_elapsed,\n                      \'transport_seconds\':transport_elapsed,\'generation_seconds\':time.monotonic()-generation_started,\n                      \'total_seconds\':time.monotonic()-started,\n                      \'scope\':\'synchronized generation timers plus CPU capture/solver/logging; not energy telemetry\'},\n            \'array_contract\':{\'coordinates\':len(reference.coordinates),\'hidden_size\':model.config.hidden_size,\n                              \'vocab_size\':model.config.vocab_size}},arrays\n\n\ndef array_shapes(record):\n    """Derive the entire checkpoint contract without trusting uploaded shapes."""\n    n, hidden = record[\'array_contract\'][\'coordinates\'], record[\'array_contract\'][\'hidden_size\']\n    expected = {}\n    def probe(prefix):\n        expected.update({prefix+\'_\'+key:(1,1,n) for key in (\'raw_z\',\'cett\',\'zcett\',\'down_output_norm_raw\')})\n        expected.update({prefix+f\'_residual_{block}\':(1,1,hidden) for block in RESIDUAL_BLOCKS})\n    for offset in record[\'request\'][\'available_offsets\']:\n        for arm in ARMS:\n            probe(f\'o{offset}_{arm["name"]}\')\n            expected[f\'o{offset}_{arm["name"]}_logits\'] = (record[\'array_contract\'][\'vocab_size\'],)\n    expected.update({name:(record[\'array_contract\'][\'vocab_size\'],) for name in (\'math_clean_logits\',\'math_pulse_logits\')})\n    probe(\'math_clean\'); probe(\'math_pulse\')\n    for key in (\'jac_source_delta\',\'jac_target_actual\',\'jac_target_predicted\',\'jac_baseline_source\',\'jac_baseline_target\',\'jac_target_nonlinear\'):\n        expected[key] = (hidden,)\n    if record[\'request\'][\'index\'] in CALIBRATION_INDICES:\n        expected.update({f\'jac_fd_{str(step).replace(".","_")}\':(hidden,) for step in FD_STEPS})\n    for name in (\'projection_replay\',\'projection_removal\',\'projection_noise\',\'full_replay_cut_control\',\'full_removal_cut_control\'):\n        probe(\'patch18_\'+name)\n        expected[\'patch18_\'+name+\'_logits\'] = (record[\'array_contract\'][\'vocab_size\'],)\n    if record[\'transport\'][\'state\'] == \'complete\':\n        for name in (\'ridge_transport\',\'direct_delta18_at14\',\'source_norm_matched_noise\'):\n            probe(\'transport_\'+name)\n            expected[\'transport_\'+name+\'_logits\'] = (record[\'array_contract\'][\'vocab_size\'],)\n            expected[\'transport_delta_\'+name] = (hidden,)\n    positions = min(4,len(record[\'request\'][\'prompt_ids\']))\n    for arm in GENERATION_ARMS:\n        prefix = f\'generation_{arm["name"]}_\'\n        expected.update({prefix+key:(1,positions,n) for key in (\'raw_z\',\'cett\',\'zcett\',\'down_output_norm_raw\')})\n        expected.update({prefix+f\'residual_layer_{block}\':(1,positions,hidden) for block in (3,7,14,27)})\n    return expected\n\n\ndef validate_record(record, plan_hash, item=None):\n    request = record.get(\'request\', {})\n    if (record.get(\'state\') != \'complete\' or record.get(\'schema\') != SCHEMA\n            or record.get(\'plan_sha256\') != plan_hash or type(request.get(\'index\')) is not int\n            or request.get(\'request_id\') != f\'b14b18_{request["index"]:03d}\'\n            or not 0 <= request[\'index\'] < 16 or (item is not None and request.get(\'item\') != item)):\n        raise ValueError(\'Complete checkpoint has a different bound request/plan/schema\')\n    if (request.get(\'prompt_ids\') != request[\'item\'][\'prompt_ids\']\n            or request.get(\'pulse_positions\') != [len(request[\'prompt_ids\'])-1]\n            or request.get(\'available_offsets\') != [v for v in OFFSETS if v <= len(request[\'continuation_ids\'])]\n            or request.get(\'missing_offsets\') != [v for v in OFFSETS if v > len(request[\'continuation_ids\'])]):\n        raise ValueError(\'Checkpoint changes source tokens, pulse position or continuation offsets\')\n    proof = record.get(\'technical_checks\', {})\n    if (proof.get(\'plan_sha256\') != plan_hash or proof.get(\'state\') != \'passed\'\n            or set(proof.get(\'checks\', {})) != set(PROOF_KEYS)\n            or not all(value is True for value in proof[\'checks\'].values())\n            or record.get(\'checks\') != proof[\'checks\']):\n        raise ValueError(\'Bound complete technical proof is required\')\n    wanted = [(offset,arm[\'name\']) for offset in request[\'available_offsets\'] for arm in ARMS]\n    rows = [row for row in record.get(\'measurements\',[]) if row.get(\'state\') == \'complete\']\n    missing = [row for row in record.get(\'measurements\',[]) if row.get(\'state\') == \'not_run\']\n    if ([(row.get(\'offset\'),row.get(\'arm\')) for row in rows] != wanted\n            or [row.get(\'offset\') for row in missing] != request[\'missing_offsets\']\n            or len(record[\'measurements\']) != len(rows)+len(missing)):\n        raise ValueError(\'Missing, duplicated or altered fixed measurement slots\')\n    for row in rows:\n        offset = row[\'offset\']\n        if (row.get(\'pulse_positions\') != request[\'pulse_positions\']\n                or row.get(\'capture_positions\') != [len(request[\'prompt_ids\'])+offset-1]\n                or row.get(\'pulse_fired\') != 1 or row.get(\'capture_fired\') != 1\n                or not math.isfinite(row.get(\'clean_to_arm_KL\',math.nan))\n                or not math.isfinite(row.get(\'arm_to_clean_KL\',math.nan))\n                or len(row.get(\'top10\',[])) != min(10,record[\'array_contract\'][\'vocab_size\'])):\n            raise ValueError(\'Incomplete finite full-KL/readout/pulse measurement\')\n    jacobian = record.get(\'jacobian\',{})\n    if (jacobian.get(\'source\') != 14 or jacobian.get(\'target\') != 18 or jacobian.get(\'arm\') != \'b14_soft_108\'\n            or jacobian.get(\'backend\') != \'MATH\' or jacobian.get(\'last_prompt_only\') is not True\n            or jacobian.get(\'primal_baseline_allclose\') is not True or jacobian.get(\'b15_source14_unchanged\') is not True):\n        raise ValueError(\'Missing fixed native local Jacobian identity/parity\')\n    calibration = record.get(\'calibration\',{})\n    if request[\'index\'] in CALIBRATION_INDICES:\n        if (calibration.get(\'state\') != \'complete\' or calibration.get(\'fd_gate_state\') != \'passed\'\n                or calibration.get(\'math_efficient_logits_allclose\') is not True\n                or [row.get(\'relative_step\') for row in calibration.get(\'finite_differences\',[])] != list(FD_STEPS)\n                or not any(row.get(\'agreement_passed\') is True for row in calibration[\'finite_differences\'])):\n            raise ValueError(\'Calibration input requires passed FD and MATH/EFFICIENT gates\')\n    elif calibration.get(\'state\') != \'not_run\':\n        raise ValueError(\'Calibration differs from its fixed subset\')\n    names = [\'projection_replay\',\'projection_removal\',\'projection_noise\',\'full_replay_cut_control\',\'full_removal_cut_control\']\n    controls = record.get(\'b18_controls\',[])\n    if ([row.get(\'name\') for row in controls] != names\n            or [row.get(\'receiver\') for row in controls] != [\'clean\',\'b14_soft_108\',\'clean\',\'clean\',\'b14_soft_108\']):\n        raise ValueError(\'Exactly five fixed B18 controls and receivers required\')\n    for row in controls+record.get(\'transport\',{}).get(\'arms\',[]):\n        if (row.get(\'pulse_positions\') != request[\'pulse_positions\'] or row.get(\'capture_positions\') != request[\'pulse_positions\']\n                or row.get(\'pulse_fired\') != 1 or row.get(\'capture_fired\') != 1\n                or row.get(\'patch\',{}).get(\'fired\') != 1 or set(row.get(\'logit_references\',{})) != {\'math_clean\',\'math_pulse\'}):\n            raise ValueError(\'Fixed offset-zero residual patch/evidence/references required\')\n    if record.get(\'transport\',{}).get(\'state\') not in (\'complete\',\'invalid\',\'not_run\'):\n        raise ValueError(\'Missing explicit local transport status\')\n    if (request[\'index\'] in TRANSPORT_INDICES) == (record[\'transport\'][\'state\']==\'not_run\'):\n        raise ValueError(\'Transport execution differs from the fixed subset\')\n    if record[\'transport\'][\'state\']==\'complete\':\n        transport=record[\'transport\']\n        if ([row.get(\'name\') for row in transport.get(\'arms\',[])] !=\n                [\'ridge_transport\',\'direct_delta18_at14\',\'source_norm_matched_noise\']\n                or transport.get(\'lambda_relative\') != RIDGE_RELATIVE or len(transport.get(\'power_iterations\',[])) != 4\n                or transport.get(\'solver\',{}).get(\'max_iter\') != CG_MAX_ITER\n                or transport.get(\'solver\',{}).get(\'rtol\') != CG_RTOL):\n            raise ValueError(\'Transport solver/controls differ from the prespecified bounded rule\')\n    gens = record.get(\'generations\',[])\n    if [row.get(\'arm\') for row in gens] != [arm[\'name\'] for arm in GENERATION_ARMS]:\n        raise ValueError(\'Exactly four fixed secondary free generations required\')\n    for row,arm in zip(gens,GENERATION_ARMS,strict=True):\n        gen = row.get(\'record\',{})\n        selected, emitted, steps = gen.get(\'selected_token_ids\',[]),gen.get(\'generated_token_ids\',[]),gen.get(\'steps\',[])\n        eos = gen.get(\'stop_reason\') == \'eos\'\n        if (gen.get(\'arm\') != arm or gen.get(\'request_id\') != f\'r{request["index"]:03d}\'\n                or gen.get(\'seed\') != base.SEED+request[\'index\'] or gen.get(\'max_new_tokens\') != GENERATION_CAP\n                or not 1 <= len(selected) <= GENERATION_CAP or len(steps) != len(selected)\n                or selected != [step.get(\'selected_token_id\') for step in steps]\n                or emitted != (selected[:-1] if eos else selected)\n                or gen.get(\'decode_forward_count\') != len(selected)-1 or gen.get(\'prefill_forward_count\') != 1\n                or gen.get(\'stop_reason\') not in (\'eos\',\'max_new_tokens\')\n                or gen.get(\'truncated\') is not (not eos) or (not eos and len(selected)!=GENERATION_CAP)\n                or any(step.get(\'is_eos\') for step in steps[:-1])\n                or bool(steps[-1].get(\'is_eos\')) is not eos\n                or gen.get(\'prompt_length\') != len(request[\'prompt_ids\'])\n                or gen.get(\'final_cache_length\') != len(request[\'prompt_ids\'])+len(selected)-1\n                or gen.get(\'pulse_positions\') != request[\'pulse_positions\']\n                or gen.get(\'prefill_capture_positions\') != list(range(max(0,len(request[\'prompt_ids\'])-4),len(request[\'prompt_ids\'])))\n                or row.get(\'prefill_equals_primary_measurement\') is not True):\n            raise ValueError(\'Generation cap/EOS/cache/prefill/prompt seed contract violated\')\n\n\ndef validate_arrays(record, arrays):\n    import numpy as np\n    expected = array_shapes(record)\n    if set(arrays) != set(expected):\n        raise ValueError(\'Checkpoint has missing or unexpected activation arrays\')\n    for key,shape in expected.items():\n        value = np.asarray(arrays[key])\n        if value.dtype != np.float32 or value.shape != shape or not np.isfinite(value).all():\n            raise ValueError(\'Checkpoint array must have exact finite FP32 shape: \'+key)\n    for row in record[\'measurements\']:\n        if row.get(\'state\')==\'complete\':\n            digest=hashlib.sha256(np.asarray(arrays[f\'o{row["offset"]}_\'+row[\'arm\']+\'_logits\']).tobytes()).hexdigest()\n            if digest != row.get(\'logits_sha256\'):\n                raise ValueError(\'Full-vocabulary measurement hash disagrees with its record\')\n    for row in record[\'generations\']:\n        digest=hashlib.sha256(np.asarray(arrays[\'o0_\'+row[\'arm\']+\'_logits\']).tobytes()).hexdigest()\n        if row[\'record\'].get(\'first_step_logits_sha256\') != digest:\n            raise ValueError(\'Generation prefill hash differs from the saved primary full logits\')\n    for prefix,rows in ((\'patch18_\',record[\'b18_controls\']),(\'transport_\',record[\'transport\'].get(\'arms\',[]))):\n        for row in rows:\n            digest=hashlib.sha256(np.asarray(arrays[prefix+row[\'name\']+\'_logits\']).tobytes()).hexdigest()\n            if any(ref.get(\'logits_sha256\') != digest for ref in row[\'logit_references\'].values()):\n                raise ValueError(\'Control/transport full logits disagree with the recorded comparator\')\n\n\ndef should_pause(elapsed, measured, remaining):\n    return bool(remaining and (elapsed>=REQUEST_LIMIT_SECONDS\n                              or elapsed+max(measured[-3:])>REQUEST_LIMIT_SECONDS))\n\n\ndef write_checkpoint(output, record, arrays, plan_hash, hardware):\n    import numpy as np\n    record = {**record,\'schema\':SCHEMA,\'state\':\'complete\',\'plan_sha256\':plan_hash,\'hardware\':hardware,\n        \'technical_checks\':{\'state\':\'passed\',\'plan_sha256\':plan_hash,\'checks\':record[\'checks\']},\n        \'completed_utc\':dt.datetime.now(dt.UTC).isoformat()}\n    validate_record(record,plan_hash)\n    validate_arrays(record,arrays)\n    destination = output/\'requests\'/record[\'request\'][\'request_id\']\n    if destination.exists():\n        raise ValueError(\'Refusing to overwrite a completed or conflicting input checkpoint\')\n    destination.parent.mkdir(parents=True,exist_ok=True)\n    temporary = Path(tempfile.mkdtemp(prefix=\'.\'+destination.name+\'_\',dir=destination.parent))\n    try:\n        np.savez_compressed(temporary/\'arrays.npz\',**arrays)\n        with (temporary/\'arrays.npz\').open(\'rb\') as stream:\n            os.fsync(stream.fileno())\n        record[\'npz_sha256\'] = base.sha_file(temporary/\'arrays.npz\')\n        base.atomic_json(temporary/\'result.json\',record)\n        os.rename(temporary,destination)\n    finally:\n        if temporary.exists():\n            shutil.rmtree(temporary)\n    return record\n\n\ndef verified(output, index, plan_hash, item):\n    import numpy as np\n    folder = Path(output)/\'requests\'/f\'b14b18_{index:03d}\'\n    if not folder.exists():\n        return None\n    if not (folder/\'result.json\').is_file() or not (folder/\'arrays.npz\').is_file():\n        raise ValueError(\'Incomplete/conflicting input checkpoint; no automatic deletion\')\n    record = json.loads((folder/\'result.json\').read_text())\n    if base.sha_file(folder/\'arrays.npz\') != record.get(\'npz_sha256\'):\n        raise ValueError(\'Input checkpoint arrays SHA mismatch\')\n    validate_record(record,plan_hash,item)\n    with np.load(folder/\'arrays.npz\',allow_pickle=False) as arrays:\n        validate_arrays(record,arrays)\n    return record\n\n\ndef restore_resume(resume_zip, output, plan_hash, items):\n    if not resume_zip:\n        return\n    with zipfile.ZipFile(resume_zip) as bundle:\n        plan = json.loads(bundle.read(\'run_plan.json\'))\n        if hashlib.sha256(base.canonical(plan)).hexdigest() != plan_hash:\n            raise ValueError(\'Resume ZIP belongs to another exact source/plan/reference binding\')\n        if len(bundle.namelist()) != len(set(bundle.namelist())) or bundle.testzip() is not None:\n            raise ValueError(\'Corrupt/duplicate-member resume ZIP\')\n        for index,item in enumerate(items):\n            folder = f\'requests/b14b18_{index:03d}\'\n            names = [folder+\'/\'+name for name in (\'result.json\',\'arrays.npz\')]\n            present = [name in bundle.namelist() for name in names]\n            if not any(present):\n                continue\n            if not all(present) or any(bundle.getinfo(name).file_size > 64*(1<<20) for name in names):\n                raise ValueError(\'Incomplete or oversized resume checkpoint\')\n            existing = verified(output,index,plan_hash,item)\n            target = output/folder\n            if existing is not None:\n                if any((target/Path(name).name).read_bytes()!=bundle.read(name) for name in names):\n                    raise ValueError(\'Resume conflicts with existing complete input; no overwrite\')\n                continue\n            target.parent.mkdir(parents=True,exist_ok=True)\n            temp = Path(tempfile.mkdtemp(prefix=\'.\'+target.name,dir=target.parent))\n            try:\n                for name in names:\n                    (temp/Path(name).name).write_bytes(bundle.read(name))\n                # Validate before publish using the same reader under a private root.\n                staging = temp/\'requests\'/target.name\n                staging.mkdir(parents=True)\n                for name in (\'result.json\',\'arrays.npz\'):\n                    os.rename(temp/name,staging/name)\n                verified(temp,index,plan_hash,item)\n                os.rename(staging,target)\n            finally:\n                shutil.rmtree(temp)\n\n\ndef archive(output):\n    target = output/\'b14_b18_results.zip\'\n    temporary = output/\'.b14_b18_results.zip.tmp\'\n    with zipfile.ZipFile(temporary,\'w\',zipfile.ZIP_DEFLATED) as bundle:\n        for path in sorted(output.rglob(\'*\')):\n            if path.is_file() and path not in (target,temporary) and not any(part.startswith(\'.\') for part in path.relative_to(output).parts):\n                bundle.write(path,path.relative_to(output))\n    os.replace(temporary,target)\n    return target\n\n\ndef initialize_engine(reference, cache, on_loaded):\n    from importlib.metadata import version\n\n    import torch\n    import transformers\n    if transformers.__version__ != base.TRANSFORMERS or version(\'accelerate\') != base.ACCELERATE:\n        raise ValueError(\'Exact original transformers5.22.2 and accelerate1.16.0 required\')\n    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False\n    torch.set_float32_matmul_precision(\'highest\')\n    torch.manual_seed(base.SEED)\n    transformers.utils.logging.disable_progress_bar()\n    hardware = base.hardware_preflight(torch,cache)\n    os.environ[\'HF_DEACTIVATE_ASYNC_LOAD\'] = \'1\'\n    print(\'Stage model: Original Qwen3-1.7B FP32 laden und Dateien prüfen.\',flush=True)\n    tokenizer,model = base.load_model(reference,cache)\n    hardware.update(torch=torch.__version__,cuda=torch.version.cuda,python=platform.python_version(),\n        transformers=transformers.__version__,attention=\'sdpa/EFFICIENT_ATTENTION + explicit derivative MATH\',\n        dtype=\'float32\',tf32=False,compute_capability=list(torch.cuda.get_device_capability()))\n    on_loaded(hardware)\n    groups = defaultdict(list)\n    for layer,neuron in reference.coordinates:\n        groups[layer].append(neuron)\n    with torch.no_grad():\n        norms = {layer:model.model.layers[layer].mlp.down_proj.weight.float().norm(dim=0)[columns]\n                 for layer,columns in groups.items()}\n    return {\'model\':model,\'tokenizer\':tokenizer,\'hardware\':hardware,\'weight_norms\':norms}\n\n\ndef run(hn,policy,refw,items_path,plan_path,out_dir,cache_dir=None,resume_zip=None,offline_plan_only=False):\n    output = Path(out_dir); output.mkdir(parents=True,exist_ok=True)\n    nonce = os.environ.get(\'H286_WRAPPER_ATTEMPT\')\n    status = {\'state\':\'running\',\'stage\':\'reference\',\'completed_inputs\':0,\'scope\':SCHEMA,\n              \'confirmatory_eligible\':False,\'wrapper_attempt_id\':nonce}\n    metrics = {\'weights_loaded\':False,\'pretrained_model_run\':False,\'no_training\':True,\n               \'wrapper_attempt_id\':nonce,\'generation_count\':0}\n    started = time.monotonic()\n    def save():\n        base.atomic_json(output/\'status.json\',status); base.atomic_json(output/\'metrics.json\',metrics)\n    save()\n    try:\n        reference,stats,items,scientific_plan = load_inputs(hn,policy,refw,items_path,plan_path)\n        plan = plan_document(reference,stats,items,scientific_plan)\n        plan_hash = hashlib.sha256(base.canonical(plan)).hexdigest()\n        existing = output/\'run_plan.json\'\n        if existing.exists() and existing.read_bytes() != base.canonical(plan)+b\'\\n\':\n            raise ValueError(\'Output directory belongs to another exact plan; no overwrite\')\n        if not existing.exists():\n            base.atomic_json(existing,plan)\n        if offline_plan_only:\n            status.update(state=\'complete\',stage=\'offline_plan_only\',plan_sha256=plan_hash,no_model_executed=True)\n            return status\n        restore_resume(resume_zip,output,plan_hash,items)\n        done = [verified(output,index,plan_hash,item) for index,item in enumerate(items)]\n        status[\'completed_inputs\'] = sum(row is not None for row in done)\n        metrics[\'generation_count\'] = 4*status[\'completed_inputs\']\n        if all(row is not None for row in done):\n            status.update(state=\'complete\',stage=\'complete\',resumed_without_gpu=True)\n            metrics[\'retained_technical_checks\'] = [row[\'technical_checks\'] for row in done]\n            return status\n        status[\'stage\'] = \'pinned_model_load\'; save()\n        def loaded(hardware):\n            metrics.update(weights_loaded=True,hardware=hardware); save()\n        engine = initialize_engine(reference,Path(cache_dir or \'/content/h286_model_cache\'),loaded)\n        engine[\'diagnostics_dir\'] = output/\'technical\'\n        compute_start = time.monotonic(); measured = []\n        import torch\n        with torch.no_grad(),attention_context(torch,\'EFFICIENT_ATTENTION\'):\n            ids = torch.tensor([items[next(i for i,row in enumerate(done) if row is None)][\'prompt_ids\']],\n                               dtype=torch.long,device=next(engine[\'model\'].parameters()).device)\n            warm = engine[\'model\'](input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=False)\n            if not bool(torch.isfinite(warm.logits).all()):\n                raise ValueError(\'Nonfinite original-model warmup logits\')\n        metrics.update(pretrained_model_run=True,warmup_forward_count=1); save()\n        metrics[\'soft_budget_seconds_after_load\'] = REQUEST_LIMIT_SECONDS\n        for index,item in enumerate(items):\n            if done[index] is not None:\n                continue\n            status.update(stage=\'request\',current_input=f\'b14b18_{index:03d}\'); save()\n            print(f\'Request {index+1}/16: sourcegebundene Präfixe, lokaler Jacobian, vier freie Antworten.\',flush=True)\n            request = prepare_request(engine[\'tokenizer\'],index,item,engine[\'model\'].config.vocab_size)\n            t0 = time.monotonic()\n            result,arrays = compute_request(engine,reference,stats,request)\n            metrics[\'pretrained_model_run\'] = True\n            done[index] = write_checkpoint(output,result,arrays,plan_hash,engine[\'hardware\'])\n            measured.append(time.monotonic()-t0)\n            status[\'completed_inputs\'] = sum(row is not None for row in done)\n            metrics[\'generation_count\'] = 4*status[\'completed_inputs\']\n            remaining = len(items)-status[\'completed_inputs\']\n            eta = sum(measured)/len(measured)*remaining\n            elapsed = time.monotonic()-compute_start\n            status.update(estimated_remaining_seconds=eta,compute_elapsed_seconds=elapsed)\n            save()\n            print(f\'Gespeichert {status["completed_inputs"]}/16; Restschätzung {eta/60:.1f} min.\',flush=True)\n            if should_pause(elapsed,measured,remaining):\n                status.update(state=\'paused\',stage=\'safe_input_checkpoint\',\n                    reason=\'Soft 45-minute compute budget/ETA reached; complete inputs preserved in resume ZIP\')\n                break\n        else:\n            status.update(state=\'complete\',stage=\'complete\')\n        metrics[\'compute_elapsed_seconds\'] = time.monotonic()-compute_start\n    except BaseException as exc:  # noqa: BLE001 — preserve diagnostics on interruption/OOM.\n        status.update(state=\'failed\',error_type=type(exc).__name__,error=str(exc))\n        (output/\'traceback.txt\').write_text(traceback.format_exc())\n        print(\'HARD FAILURE: \'+str(exc),flush=True)\n    finally:\n        metrics[\'elapsed_seconds\'] = time.monotonic()-started\n        save(); archive(output)\n    return status\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--hn-set\',required=True)\n    parser.add_argument(\'--policy\',required=True)\n    parser.add_argument(\'--refw\',required=True)\n    parser.add_argument(\'--items\',required=True)\n    parser.add_argument(\'--plan\',required=True)\n    parser.add_argument(\'--out-dir\',required=True)\n    parser.add_argument(\'--cache-dir\')\n    parser.add_argument(\'--resume-zip\')\n    parser.add_argument(\'--offline-plan-only\',action=\'store_true\')\n    args = parser.parse_args()\n    state = run(args.hn_set,args.policy,args.refw,args.items,args.plan,args.out_dir,\n                args.cache_dir,args.resume_zip,args.offline_plan_only)\n    print(json.dumps(state),flush=True)\n    raise SystemExit(0 if state[\'state\'] in (\'complete\',\'paused\') else 1)\n', 'qwen3_b14_b18_refine_runtime.py': '"""Prespecified local B14→B18 inverse refinement; no free generations.\n\nAll four same-lambda stages precede the two fixed damping ablations. Original\nweights, H286, source tokens and the old runtime remain unchanged. Checkpoints\nbind one case and one damping factor; a complete run requires all twelve.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport datetime as dt\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport tempfile\nimport time\nimport traceback\nimport zipfile\nfrom pathlib import Path\n\nimport qwen3_b14_b18_runtime as old\n\nbase = old.base\nSCHEMA = \'qwen3_b14_b18_refinement_v1\'\nINDICES = (0, 1, 12, 13)\nFACTORS = (1., .3, .1)\nMAX_ITER = 512\nRTOL = 1e-4\nNOISE_COUNT = 8\nNOISE_SEED_BASE = base.SEED + 14000\nBUDGET_SECONDS = 1200\nANCHOR_RTOL = 1e-3\nANCHOR_ATOL = 1e-5\nANCHORS_SHA256 = \'d56873915a388bcf9c42e606a64f512d02098bf1747e45306196938e9284b2c7\'\nREFINE_PLAN_SHA256 = \'ff4cd84d3ff1e01e87074d5c2938d79224c83634fa441a095a70c7d1d9b78afb\'\nOLD_RUNTIME_SHA256 = \'c04586073a54fed7a257db7e27fed48209abab8b151914a74439d154e1fad6af\'\nSOURCE_NAMES = (\'qwen3_b14_b18_runtime.py\', *old.SOURCE_NAMES,\n                \'attention_probe/transport_refinement.py\')\nVECTOR_KEYS = (\'baseline_source\', \'baseline_target\', \'source_delta\',\n               \'target_delta\', \'jvp\', \'old_delta_source\')\nPROOF_KEYS = (\'native_sham_exact\', \'mapped_baseline\', \'mapped_pulse\',\n              \'anchor_continuity\', \'derivative_calibration\', \'frozen_parameters\',\n              \'global_rng\', \'hook_cleanup\', \'finite_arrays\', \'unchanged_positions\',\n              \'upstream_causal\')\n\n\ndef factor_name(factor):\n    if type(factor) not in (int, float) or factor not in FACTORS:\n        raise ValueError(\'Only the three prespecified damping factors are allowed\')\n    return {1.: \'f1\', .3: \'f0p3\', .1: \'f0p1\'}[factor]\n\n\ndef stage_order():\n    return [(index, factor) for factor in FACTORS for index in INDICES]\n\n\ndef arm_names(factor):\n    factor_name(factor)\n    return ([\'recorded_old_ridge\', \'raw_target_copy\'] if factor == 1. else []) + [\n        \'refined_ridge\', \'norm_matched_copy\', *[f\'norm_matched_noise_{i}\' for i in range(NOISE_COUNT)]]\n\n\ndef _sha(path):\n    return base.sha_file(path)\n\n\ndef _json_file(path, expected):\n    raw = Path(path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != expected:\n        raise ValueError(\'Exact pinned refinement input bytes required: \' + Path(path).name)\n    return json.loads(raw)\n\n\ndef load_anchors(path, items, *, expected_sha256=None, expected_width=2048):\n    document = _json_file(path, expected_sha256 or ANCHORS_SHA256)\n    cases = document.get(\'cases\', [])\n    if (document.get(\'schema\') != \'b14_b18_refine_anchors_v1\'\n            or document.get(\'source_runtime_sha256\') != OLD_RUNTIME_SHA256\n            or [case.get(\'index\') for case in cases] != list(INDICES)):\n        raise ValueError(\'Anchors must bind the four original cases and original runtime\')\n    for case in cases:\n        index = case[\'index\']\n        if (case.get(\'request_id\') != f\'b14b18_{index:03d}\'\n                or case.get(\'item_sha256\') != hashlib.sha256(base.canonical(items[index])).hexdigest()\n                or type(case.get(\'lambda\')) not in (float, int)\n                or not math.isfinite(case[\'lambda\']) or case[\'lambda\'] <= 0):\n            raise ValueError(\'Anchor context, item or original damping changed\')\n        for key in VECTOR_KEYS:\n            values = case.get(key)\n            if (not isinstance(values, list) or len(values) != expected_width\n                    or any(type(value) not in (float, int) or not math.isfinite(value)\n                           or abs(value) > 3.4028234663852886e38 for value in values)):\n                raise ValueError(\'Anchor vectors must have finite native hidden width\')\n    return document\n\n\ndef load_inputs(hn, policy, refw, items_path, plan_path, refine_plan_path, anchors_path):\n    reference, stats, items, original_plan = old.load_inputs(hn, policy, refw, items_path, plan_path)\n    refinement = _json_file(refine_plan_path, REFINE_PLAN_SHA256)\n    expected = {\'schema\': \'b14_b18_refine_plan_v1\', \'indices\': list(INDICES),\n                \'lambda_factors\': list(FACTORS), \'max_iter\': MAX_ITER, \'rtol\': RTOL,\n                \'noise_count\': NOISE_COUNT, \'noise_seed_base\': NOISE_SEED_BASE,\n                \'true_residual_interval\': 16,\n                \'seconds_after_model_load\': BUDGET_SECONDS,\n                \'anchor_rtol\': ANCHOR_RTOL, \'anchor_atol\': ANCHOR_ATOL}\n    if any(refinement.get(key) != value for key, value in expected.items()):\n        raise ValueError(\'Refinement plan changes prespecified numerical rules\')\n    anchors = load_anchors(anchors_path, items)\n    if _sha(old.__file__) != OLD_RUNTIME_SHA256:\n        raise ValueError(\'The frozen original runtime source has changed\')\n    parent = old.plan_document(reference, stats, items, original_plan)\n    parent_raw = base.canonical(parent)\n    if (hashlib.sha256(parent_raw).hexdigest() != anchors.get(\'source_plan_sha256\')\n            or hashlib.sha256(parent_raw + b\'\\n\').hexdigest() != anchors.get(\'source_plan_file_sha256\')):\n        raise ValueError(\'Original parent plan/source binding cannot be reconstructed\')\n    return reference, stats, items, original_plan, refinement, anchors\n\n\ndef plan_document(reference, stats, items, original_plan, refinement, anchors):\n    return {\'schema\': SCHEMA, \'helper_sha256\': _sha(__file__),\n        \'source_sha256\': {name: _sha(old.source_path(name)) for name in SOURCE_NAMES},\n        \'parent_plan_sha256\': anchors[\'source_plan_sha256\'],\n        \'parent_plan_file_sha256\': anchors[\'source_plan_file_sha256\'],\n        \'source_zip_sha256\': anchors[\'source_zip_sha256\'],\n        \'refine_plan_sha256\': REFINE_PLAN_SHA256, \'anchors_sha256\': ANCHORS_SHA256,\n        \'refinement\': refinement, \'original_plan\': original_plan,\n        \'items_sha256\': old.ITEMS_SHA256, \'original_input_plan_sha256\': old.INPUT_PLAN_SHA256,\n        \'items\': items, \'indices\': list(INDICES), \'stage_order\': [list(row) for row in stage_order()],\n        \'model\': reference.model.to_dict(), \'coordinates\': [list(c) for c in reference.coordinates],\n        \'hn_sha256\': reference.hn_sha256, \'policy_sha256\': reference.policy_sha256,\n        \'refw_sha256\': base.REFW_SHA256, \'refw_source_sha256\': stats[\'source_hn_meta_sha256\'],\n        \'seed\': base.SEED, \'dtype\': \'float32\', \'tf32\': False,\n        \'transformers\': base.TRANSFORMERS, \'accelerate\': base.ACCELERATE,\n        \'attention\': \'sdpa/MATH; native local derivatives and nonlinear endpoint probes\',\n        \'max_iter\': MAX_ITER, \'rtol\': RTOL, \'lambda_factors\': list(FACTORS),\n        \'lambda_rule\': \'recorded original lambda times fixed factor; no gain re-estimation or outcome selection\',\n        \'controls\': {\'noise_seeds\': {str(i): [NOISE_SEED_BASE+i+k for k in range(NOISE_COUNT)] for i in INDICES},\n                     \'copy\': \'target_delta direction normalized to the refined source norm after native FP32 rounding\',\n                     \'paired_noise\': \'same eight fixed directions across damping factors\',\n                     \'old_baselines\': \'recorded sixteen-CG delta and original unmatched raw target-copy once per case\'},\n        \'anchor_gate\': {\'metric\': \'L2(current-reference) <= atol + rtol*L2(reference)\',\n                        \'rtol\': ANCHOR_RTOL, \'atol\': ANCHOR_ATOL,\n                        \'logit_SHA\': \'recorded evidence, not a cross-hardware exact gate\'},\n        \'budget\': {\'compute_seconds\': BUDGET_SECONDS, \'boundary\': \'before stages, solver iterations and endpoint arms\',\n                   \'cooperative\': \'an in-flight GPU call cannot be preempted; solver may finish its final true-residual evidence\',\n                   \'partial\': \'partial stages retained separately; only full stage checkpoints are resumable\'},\n        \'free_generations\': 0, \'claim\': \'exploratory numerical local inverse refinement; no training or global inverse proof\'}\n\n\ndef anchor_comparison(current, values):\n    import torch\n    reference = torch.tensor(values, dtype=current.dtype, device=current.device)\n    if (reference.shape != current.shape or not bool(torch.isfinite(current).all())\n            or not bool(torch.isfinite(reference).all())):\n        raise ValueError(\'Current anchor is nonfinite or has changed shape\')\n    norm = float(reference.double().norm())\n    error = float((current.double()-reference.double()).norm())\n    limit = ANCHOR_ATOL + ANCHOR_RTOL*norm\n    return {\'reference_norm\': norm, \'current_norm\': float(current.double().norm()),\n            \'error_norm\': error, \'relative_error\': error/norm if norm else None,\n            \'limit\': limit, \'passed\': error <= limit, \'bitwise_equal\': bool(torch.equal(current, reference))}\n\n\ndef control_vectors(target, solution, index):\n    """Fixed directions; local CPU generators do not alter global RNG state."""\n    import torch\n    norm, target_norm = float(solution.double().norm()), float(target.double().norm())\n    if target_norm == 0:\n        copy = torch.zeros_like(target)\n    else:\n        copy = (target.double()*(norm/target_norm)).to(dtype=target.dtype)\n    return [(\'refined_ridge\', solution), (\'norm_matched_copy\', copy)] + [\n        (f\'norm_matched_noise_{k}\', old.fixed_noise_like(solution, NOISE_SEED_BASE+index+k))\n        for k in range(NOISE_COUNT)]\n\n\ndef endpoint_probe(model, request, coordinates, stats, arm, *, weight_norms=None, patch=None):\n    """Observe all earlier prompt positions at the six fixed residual layers."""\n    earlier = {}\n    observers = []\n    calls = 0\n    def remove_observers(*_args):\n        while observers:\n            observers.pop().remove()\n    def install_observers(_model, _args):\n        nonlocal calls\n        calls += 1\n        if calls != 1:\n            raise ValueError(\'Endpoint observer expected exactly one native full forward\')\n        # old.forward_probe has now installed its patch/capture hooks. Appending\n        # these output observers here places them after the actual source patch.\n        for block in old.RESIDUAL_BLOCKS:\n            def observe(_module, _args, output, block=block):\n                value = output[0] if isinstance(output, tuple) else output\n                earlier[str(block)] = old.digest_tensor(value[:, :-1, :])\n            observers.append(model.model.layers[block].register_forward_hook(observe))\n    with contextlib.ExitStack() as contexts:\n        contexts.callback(remove_observers)\n        contexts.callback(model.register_forward_pre_hook(install_observers).remove)\n        contexts.callback(model.register_forward_hook(remove_observers, always_call=True).remove)\n        result = old.forward_probe(model, request, 0, coordinates, stats, arm,\n                                   backend=\'MATH\', weight_norms=weight_norms, patch=patch)\n    if calls != 1 or set(earlier) != {str(block) for block in old.RESIDUAL_BLOCKS}:\n        raise ValueError(\'Missing native post-patch earlier-position observers\')\n    result[\'earlier_prompt_residual_sha256\'] = earlier\n    return result\n\n\ndef _calibrate(mapping, source_delta, index, arrays):\n    import torch\n    predicted = mapping.jvp(source_delta)\n    differences = []\n    for step in old.FD_STEPS:\n        finite = mapping.central_fd(source_delta, step)\n        row = old.vector_metrics(finite, predicted)\n        floor = max(1e-7, 64*torch.finfo(source_delta.dtype).eps*float(mapping.baseline_target.double().norm())/step)\n        row.update(step=step, estimated_FP32_noise_floor=floor,\n                   informative=row[\'actual_norm\'] > floor)\n        row[\'passed\'] = bool(row[\'informative\'] and row[\'relative_error\'] <= .05\n                             and row[\'cosine\'] is not None and row[\'cosine\'] >= .99)\n        differences.append(row)\n        arrays[f\'calibration_fd_{str(step).replace(".", "_")}\'] = finite.cpu().numpy()\n    direction = old.fixed_noise_like(torch.ones_like(source_delta), base.SEED+index+51000)\n    cotangent = old.fixed_noise_like(torch.ones_like(source_delta), base.SEED+index+52000)\n    image, transpose = mapping.jvp(direction), mapping.vjp(cotangent)\n    left = float(torch.dot(image.double(), cotangent.double()))\n    right = float(torch.dot(direction.double(), transpose.double()))\n    scale = float(image.double().norm()*cotangent.double().norm()+direction.double().norm()*transpose.double().norm())\n    error = abs(left-right)\n    adjoint = {\'left\': left, \'right\': right, \'absolute_error\': error,\n               \'normalized_error\': error/scale if scale else 0., \'tolerance\': 1e-5,\n               \'passed\': error <= 1e-5*scale+1e-8}\n    passed = any(row[\'passed\'] for row in differences) and adjoint[\'passed\']\n    if not passed:\n        raise ValueError(\'Native finite-difference/adjoint gate failed before inverse refinement: \'\n                         +json.dumps({\'finite_differences\': differences, \'adjoint\': adjoint}, allow_nan=False))\n    return {\'state\': \'passed\', \'finite_differences\': differences, \'adjoint\': adjoint}\n\n\ndef compute_stage(engine, reference, stats, request, anchor, factor, *, stop_requested=None):\n    import numpy as np\n    import torch\n\n    from attention_probe.native_jacobian import build_residual_map\n    from attention_probe.transport_refinement import refine_transport\n\n    factor_name(factor)\n    stop = stop_requested or (lambda: False)\n    model = engine[\'model\']\n    device = next(model.parameters()).device\n    started = time.monotonic()\n    before_parameters, before_hooks = base.parameter_evidence(model), old.hook_count(model)\n    cpu_rng = torch.random.get_rng_state().clone()\n    cuda_rng = torch.cuda.get_rng_state().clone() if device.type == \'cuda\' else None\n    arrays = {}\n    record = {\'request\': request, \'lambda_factor\': factor, \'original_lambda\': anchor[\'lambda\'],\n              \'lambda\': anchor[\'lambda\']*factor, \'state\': \'partial\', \'arms\': [],\n              \'array_contract\': {\'coordinates\': len(reference.coordinates),\n                                 \'coordinate_layers\': [block for block, _ in reference.coordinates],\n                                 \'hidden_size\': model.config.hidden_size,\n                                 \'vocab_size\': model.config.vocab_size},\n              \'timing\': {}, \'checks\': {}}\n    if stop():\n        record.update(stop_reason=\'budget_exhausted_before_stage\')\n        return record, arrays\n    clean = endpoint_probe(model, request, reference.coordinates, stats, old.ARMS[0],\n                           weight_norms=engine.get(\'weight_norms\'))\n    if engine.get(\'on_model_run\') is not None:\n        engine[\'on_model_run\']()\n    pulse = endpoint_probe(model, request, reference.coordinates, stats, old.ARMS[4],\n                           weight_norms=engine.get(\'weight_norms\'))\n    for name, probe in ((\'math_clean\', clean), (\'math_pulse\', pulse)):\n        old.save_probe(arrays, name, probe)\n        arrays[name+\'_logits\'] = probe[\'logits\'].numpy()\n    ids = torch.tensor([request[\'prompt_ids\']], dtype=torch.long, device=device)\n    with torch.no_grad(), old.attention_context(torch, \'MATH\'):\n        native = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False).logits[0, -1].detach().float().cpu()\n    native_exact = bool(torch.equal(native, clean[\'logits\']))\n    if not native_exact:\n        raise ValueError(\'Uninstrumented MATH clean differs from passive-capture sham\')\n    mapping = build_residual_map(model, ids, 14, 18, attention_mask=torch.ones_like(ids))\n    if (not torch.equal(mapping.baseline_source, old.tensor_residual(clean, 14, device))\n            or not torch.equal(mapping.baseline_target, old.tensor_residual(clean, 18, device))):\n        raise ValueError(\'Native suffix baseline differs from the captured MATH clean source/target\')\n    source_delta = old.tensor_residual(pulse, 14, device)-mapping.baseline_source\n    target_delta = old.tensor_residual(pulse, 18, device)-mapping.baseline_target\n    prediction = mapping.jvp(source_delta)\n    vectors = {\'baseline_source\': mapping.baseline_source, \'baseline_target\': mapping.baseline_target,\n               \'source_delta\': source_delta, \'target_delta\': target_delta, \'jvp\': prediction}\n    comparisons = {key: anchor_comparison(value, anchor[key]) for key, value in vectors.items()}\n    if not all(row[\'passed\'] for row in comparisons.values()):\n        raise ValueError(\'Prior native-state/JVP continuity gate failed: \'+str(comparisons))\n    recovered = mapping(mapping.baseline_source).detach()\n    actual = mapping(old.tensor_residual(pulse, 14, device)).detach()-mapping.baseline_target\n    baseline_ok = bool(torch.allclose(recovered, mapping.baseline_target, rtol=1e-5, atol=1e-5))\n    pulse_ok = bool(torch.allclose(actual, target_delta, rtol=1e-4, atol=1e-5))\n    if not baseline_ok or not pulse_ok:\n        raise ValueError(\'Native suffix primal continuity failed\')\n    arrays.update({f\'jac_{key}\': value.detach().cpu().numpy() for key, value in vectors.items()})\n    old_delta = torch.tensor(anchor[\'old_delta_source\'], dtype=torch.float32, device=device)\n    arrays[\'recorded_old_delta_source\'] = old_delta.cpu().numpy()\n    record[\'anchor_continuity\'] = {\'vectors\': comparisons,\n        \'earlier_prompt_residual_sha256\': clean[\'earlier_prompt_residual_sha256\'],\n        \'pulse_earlier_positions_equal_clean\': pulse[\'earlier_prompt_residual_sha256\'] == clean[\'earlier_prompt_residual_sha256\'],\n        \'pulse_upstream_raw_z_unchanged\': bool(np.array_equal(\n            pulse[\'arrays\'][\'raw_z\'][..., [i for i, (block, _) in enumerate(reference.coordinates) if block < 14]],\n            clean[\'arrays\'][\'raw_z\'][..., [i for i, (block, _) in enumerate(reference.coordinates) if block < 14]])),\n        \'pulse_upstream_resid13_unchanged\': bool(np.array_equal(pulse[\'arrays\'][\'residual_13\'], clean[\'arrays\'][\'residual_13\'])),\n        \'math_clean_logits_sha256\': old.digest_tensor(clean[\'logits\']),\n        \'original_math_clean_logits_sha256\': anchor[\'math_clean_logits_sha256\'],\n        \'math_pulse_logits_sha256\': old.digest_tensor(pulse[\'logits\']),\n        \'original_math_pulse_logits_sha256\': anchor[\'math_pulse_logits_sha256\'],\n        \'claim\': \'bounded vector continuity, not proof of an exactly identical operator across hardware\'}\n    record[\'calibration\'] = (_calibrate(mapping, source_delta, request[\'index\'], arrays) if factor == 1.\n                              else {\'state\': \'not_run\', \'reason\': \'all four primary f1 stages perform the fixed gates\'})\n    record[\'timing\'][\'setup_seconds\'] = time.monotonic()-started\n    solve_started = time.monotonic()\n    solved = refine_transport(mapping.jvp, mapping.vjp, target_delta, record[\'lambda\'],\n                              max_iter=MAX_ITER, rtol=RTOL, stop_requested=stop,\n                              true_residual_interval=16)\n    record[\'solver\'] = solved.diagnostics\n    record[\'timing\'][\'solver_seconds\'] = time.monotonic()-solve_started\n    arrays[\'solver_delta_source\'] = solved.delta_source.detach().cpu().numpy()\n    arrays[\'solver_dual_solution\'] = solved.dual_solution.detach().cpu().numpy()\n    if solved.diagnostics.get(\'stop_reason\') == \'budget_exhausted\' or stop():\n        record.update(stop_reason=\'budget_exhausted_during_solver\')\n        record[\'timing\'][\'total_seconds\'] = time.monotonic()-started\n        return record, arrays\n    controls = control_vectors(target_delta, solved.delta_source, request[\'index\'])\n    if factor == 1.:\n        original_target = torch.tensor(anchor[\'target_delta\'], dtype=torch.float32, device=device)\n        controls = [(\'recorded_old_ridge\', old_delta), (\'raw_target_copy\', original_target), *controls]\n    evaluation_started = time.monotonic()\n    for name, delta in controls:\n        if stop():\n            record.update(stop_reason=\'budget_exhausted_between_endpoint_arms\')\n            break\n        arm_started = time.monotonic()\n        probe = endpoint_probe(model, request, reference.coordinates, stats, old.ARMS[0],\n                               weight_norms=engine.get(\'weight_norms\'),\n                               patch={\'block\': 14, \'delta\': delta})\n        achieved = old.tensor_residual(probe, 18, device)-mapping.baseline_target\n        realized = old.tensor_residual(probe, 14, device)-mapping.baseline_source\n        linear = mapping.jvp(delta)\n        prefix = \'endpoint_\'+name\n        old.save_probe(arrays, prefix, probe)\n        arrays[prefix+\'_logits\'] = probe[\'logits\'].numpy()\n        arrays[prefix+\'_requested_delta\'] = delta.detach().cpu().numpy()\n        arrays[prefix+\'_realized_source_delta\'] = realized.detach().cpu().numpy()\n        arrays[prefix+\'_target_actual\'] = achieved.detach().cpu().numpy()\n        arrays[prefix+\'_target_predicted\'] = linear.detach().cpu().numpy()\n        source_norm = float(delta.double().norm())\n        refined_norm = float(solved.delta_source.double().norm())\n        row = {\'name\': name, **probe[\'evidence\'],\n               \'earlier_prompt_positions_unchanged\': probe[\'earlier_prompt_residual_sha256\'] == clean[\'earlier_prompt_residual_sha256\'],\n               \'earlier_prompt_residual_sha256\': probe[\'earlier_prompt_residual_sha256\'],\n               \'upstream_raw_z_unchanged\': bool(np.array_equal(\n                   probe[\'arrays\'][\'raw_z\'][..., [i for i, (block, _) in enumerate(reference.coordinates) if block <= 14]],\n                   clean[\'arrays\'][\'raw_z\'][..., [i for i, (block, _) in enumerate(reference.coordinates) if block <= 14]])),\n               \'upstream_resid13_unchanged\': bool(np.array_equal(probe[\'arrays\'][\'residual_13\'], clean[\'arrays\'][\'residual_13\'])),\n               \'requested_source_norm\': source_norm, \'realized_source_norm\': float(realized.double().norm()),\n               \'source_rounding_error_norm\': float((realized.double()-delta.double()).norm()),\n               \'refined_source_norm\': refined_norm,\n               \'norm_ratio_to_refined\': source_norm/refined_norm if refined_norm else None,\n               \'norm_match_error\': abs(source_norm-refined_norm),\n               \'noise_seed\': NOISE_SEED_BASE+request[\'index\']+int(name.rsplit(\'_\', 1)[-1]) if name.startswith(\'norm_matched_noise_\') else None,\n               \'linear_target_prediction\': old.vector_metrics(target_delta, linear),\n               \'nonlinear_target_prediction\': old.vector_metrics(target_delta, achieved),\n               \'linear_vs_nonlinear\': old.vector_metrics(achieved, linear),\n               \'logit_references\': {\'math_clean\': old.logits_metrics(probe[\'logits\'], clean[\'logits\'], request[\'continuation_ids\'][0]),\n                                    \'math_pulse\': old.logits_metrics(probe[\'logits\'], pulse[\'logits\'], request[\'continuation_ids\'][0])},\n               \'elapsed_seconds\': time.monotonic()-arm_started}\n        if name.startswith(\'norm_matched_\') and abs(source_norm-refined_norm) > 2e-6*max(1., refined_norm):\n            raise ValueError(\'Native FP32 control norm matching failed\')\n        record[\'arms\'].append(row)\n    record[\'timing\'][\'endpoint_seconds\'] = time.monotonic()-evaluation_started\n    record[\'timing\'][\'total_seconds\'] = time.monotonic()-started\n    if len(record[\'arms\']) != len(arm_names(factor)):\n        return record, arrays\n    checks = {\'native_sham_exact\': native_exact, \'mapped_baseline\': baseline_ok,\n              \'mapped_pulse\': pulse_ok, \'anchor_continuity\': all(row[\'passed\'] for row in comparisons.values()),\n              \'derivative_calibration\': record[\'calibration\'][\'state\'] in (\'passed\', \'not_run\'),\n              \'frozen_parameters\': before_parameters == base.parameter_evidence(model),\n              \'global_rng\': bool(torch.equal(cpu_rng, torch.random.get_rng_state()) and\n                                 (cuda_rng is None or torch.equal(cuda_rng, torch.cuda.get_rng_state()))),\n              \'hook_cleanup\': old.hook_count(model) == before_hooks,\n              \'unchanged_positions\': bool(record[\'anchor_continuity\'][\'pulse_earlier_positions_equal_clean\'] and\n                                           all(row[\'earlier_prompt_positions_unchanged\'] for row in record[\'arms\'])),\n              \'upstream_causal\': bool(record[\'anchor_continuity\'][\'pulse_upstream_raw_z_unchanged\'] and\n                                      record[\'anchor_continuity\'][\'pulse_upstream_resid13_unchanged\'] and\n                                      all(row[\'upstream_raw_z_unchanged\'] and row[\'upstream_resid13_unchanged\'] for row in record[\'arms\'])),\n              \'finite_arrays\': all(value.dtype == np.float32 and np.isfinite(value).all() for value in arrays.values())}\n    if not all(checks.values()):\n        raise ValueError(\'Native/frozen/RNG/cleanup evidence failed: \'+str(checks))\n    record.update(state=\'complete\', checks=checks)\n    return record, arrays\n\n\ndef array_shapes(record):\n    width = record[\'array_contract\'][\'hidden_size\']\n    n = record[\'array_contract\'][\'coordinates\']\n    vocab = record[\'array_contract\'][\'vocab_size\']\n    shapes = {}\n    def probe(prefix):\n        shapes.update({prefix+\'_\'+key: (1, 1, n) for key in (\'raw_z\', \'cett\', \'zcett\', \'down_output_norm_raw\')})\n        shapes.update({prefix+f\'_residual_{block}\': (1, 1, width) for block in old.RESIDUAL_BLOCKS})\n        shapes[prefix+\'_logits\'] = (vocab,)\n    probe(\'math_clean\')\n    probe(\'math_pulse\')\n    shapes.update({f\'jac_{key}\': (width,) for key in VECTOR_KEYS if key != \'old_delta_source\'})\n    shapes.update({key: (width,) for key in (\'recorded_old_delta_source\', \'solver_delta_source\', \'solver_dual_solution\')})\n    if record[\'lambda_factor\'] == 1.:\n        shapes.update({f\'calibration_fd_{str(step).replace(".", "_")}\': (width,) for step in old.FD_STEPS})\n    for name in arm_names(record[\'lambda_factor\']):\n        prefix = \'endpoint_\'+name\n        probe(prefix)\n        shapes.update({prefix+\'_\'+key: (width,) for key in (\'requested_delta\', \'realized_source_delta\', \'target_actual\', \'target_predicted\')})\n    return shapes\n\n\ndef validate_record(record, plan_hash, item=None, factor=None):\n    request = record.get(\'request\', {})\n    current_factor = record.get(\'lambda_factor\')\n    if (record.get(\'schema\') != SCHEMA or record.get(\'state\') != \'complete\'\n            or record.get(\'plan_sha256\') != plan_hash or type(request.get(\'index\')) is not int\n            or request.get(\'index\') not in INDICES\n            or request.get(\'request_id\') != f\'b14b18_{request["index"]:03d}\'\n            or (item is not None and request.get(\'item\') != item)\n            or type(current_factor) not in (int, float) or current_factor not in FACTORS\n            or (factor is not None and current_factor != factor)\n            or record.get(\'lambda\') != record.get(\'original_lambda\', -1)*current_factor\n            or not math.isfinite(record[\'lambda\']) or record[\'lambda\'] <= 0):\n        raise ValueError(\'Complete stage changes its bound schema/context/plan/damping\')\n    if (request.get(\'prompt_ids\') != request[\'item\'][\'prompt_ids\']\n            or request.get(\'pulse_positions\') != [len(request[\'prompt_ids\'])-1]):\n        raise ValueError(\'Stage changed original source tokens or pulse position\')\n    checks = record.get(\'checks\', {})\n    proof = record.get(\'technical_checks\', {})\n    if (set(checks) != set(PROOF_KEYS) or not all(value is True for value in checks.values())\n            or proof != {\'state\': \'passed\', \'schema\': SCHEMA, \'plan_sha256\': plan_hash, \'checks\': checks}):\n        raise ValueError(\'Complete stage requires its own full technical proof\')\n    if not all(row.get(\'passed\') is True for row in record.get(\'anchor_continuity\', {}).get(\'vectors\', {}).values()) or set(record.get(\'anchor_continuity\', {}).get(\'vectors\', {})) != set(VECTOR_KEYS)-{\'old_delta_source\'}:\n        raise ValueError(\'Native prior-anchor continuity proof missing\')\n    calibration = record.get(\'calibration\', {})\n    if (current_factor == 1. and calibration.get(\'state\') != \'passed\') or (current_factor != 1. and calibration.get(\'state\') != \'not_run\'):\n        raise ValueError(\'Fixed derivative-calibration phase changed\')\n    solver = record.get(\'solver\', {})\n    if (solver.get(\'max_iter\') != MAX_ITER or solver.get(\'rtol\') != RTOL\n            or solver.get(\'true_residual_interval\') != 16\n            or solver.get(\'lambda\') != record[\'lambda\']\n            or solver.get(\'stop_reason\') == \'budget_exhausted\'\n            or not math.isfinite(solver.get(\'true_dual_relative_residual\', math.nan))\n            or type(solver.get(\'iterations\')) is not int or not 0 <= solver[\'iterations\'] <= MAX_ITER):\n        raise ValueError(\'Missing bounded finite solver evidence\')\n    relative = solver[\'true_dual_relative_residual\']\n    breakdown = solver.get(\'breakdown\')\n    expected_converged = relative <= RTOL and breakdown is None\n    expected_status = \'breakdown\' if breakdown else \'converged\' if expected_converged else \'not_converged\'\n    if (relative < 0 or type(solver.get(\'converged\')) is not bool\n            or solver[\'converged\'] != expected_converged or solver.get(\'status\') != expected_status):\n        raise ValueError(\'Solver convergence/status contradict its measured true residual\')\n    if [row.get(\'name\') for row in record.get(\'arms\', [])] != arm_names(current_factor):\n        raise ValueError(\'Missing/changed fixed endpoint controls\')\n    for row in record[\'arms\']:\n        if (row.get(\'pulse_fired\') != 1 or row.get(\'capture_fired\') != 1\n                or row.get(\'backend\') != \'MATH\' or row.get(\'pulse_positions\') != request[\'pulse_positions\']\n                or row.get(\'capture_positions\') != request[\'pulse_positions\']\n                or row.get(\'patch\', {}).get(\'fired\') != 1):\n            raise ValueError(\'Native endpoint patch/capture evidence missing\')\n        if not all(row.get(key) is True for key in (\'earlier_prompt_positions_unchanged\', \'upstream_raw_z_unchanged\', \'upstream_resid13_unchanged\')):\n            raise ValueError(\'Earlier-position/upstream causal invariance evidence failed\')\n        earlier = record[\'anchor_continuity\'].get(\'earlier_prompt_residual_sha256\', {})\n        if (set(earlier) != {str(block) for block in old.RESIDUAL_BLOCKS}\n                or row.get(\'earlier_prompt_residual_sha256\') != earlier):\n            raise ValueError(\'Stored earlier-position hashes contradict the causal invariance claim\')\n        if row[\'name\'].startswith(\'norm_matched_\') and row.get(\'norm_match_error\', math.inf) > 2e-6*max(1., row[\'refined_source_norm\']):\n            raise ValueError(\'Control source norms no longer match\')\n        if row[\'name\'].startswith(\'norm_matched_noise_\') and row.get(\'noise_seed\') != NOISE_SEED_BASE+request[\'index\']+int(row[\'name\'].rsplit(\'_\', 1)[-1]):\n            raise ValueError(\'Fixed paired noise seed changed\')\n        for comparison in (\'linear_target_prediction\', \'nonlinear_target_prediction\'):\n            if not math.isfinite(row.get(comparison, {}).get(\'error_norm\', math.nan)):\n                raise ValueError(\'Missing finite linear/nonlinear endpoint result\')\n\n\ndef validate_arrays(record, arrays):\n    import numpy as np\n    import torch\n    expected = array_shapes(record)\n    if set(arrays) != set(expected):\n        raise ValueError(\'Stage arrays have missing or unrecognized keys\')\n    for name, shape in expected.items():\n        value = arrays[name]\n        if value.dtype != np.float32 or value.shape != shape or not np.isfinite(value).all():\n            raise ValueError(\'Stage array changed shape/dtype/finiteness: \'+name)\n    clean = torch.from_numpy(arrays[\'math_clean_logits\'])\n    pulse = torch.from_numpy(arrays[\'math_pulse_logits\'])\n    layers = record[\'array_contract\'][\'coordinate_layers\']\n    if len(layers) != record[\'array_contract\'][\'coordinates\'] or any(type(block) is not int or not 0 <= block < 28 for block in layers):\n        raise ValueError(\'Invalid fixed coordinate-layer layout\')\n    before_source = [i for i, block in enumerate(layers) if block < 14]\n    through_source = [i for i, block in enumerate(layers) if block <= 14]\n    if (not np.array_equal(arrays[\'math_clean_raw_z\'][..., before_source], arrays[\'math_pulse_raw_z\'][..., before_source])\n            or not np.array_equal(arrays[\'math_clean_residual_13\'], arrays[\'math_pulse_residual_13\'])):\n        raise ValueError(\'Saved pulse changed upstream measurements\')\n    for row in record[\'arms\']:\n        prefix = \'endpoint_\'+row[\'name\']\n        if (not np.array_equal(arrays[prefix+\'_raw_z\'][..., through_source], arrays[\'math_clean_raw_z\'][..., through_source])\n                or not np.array_equal(arrays[prefix+\'_residual_13\'], arrays[\'math_clean_residual_13\'])):\n            raise ValueError(\'Saved endpoint changed upstream measurements\')\n        logits = torch.from_numpy(arrays[prefix+\'_logits\'])\n        for label, baseline in ((\'math_clean\', clean), (\'math_pulse\', pulse)):\n            if old.logits_metrics(logits, baseline, record[\'request\'][\'continuation_ids\'][0]) != row[\'logit_references\'][label]:\n                raise ValueError(\'Full-logit comparator/hash mismatch\')\n        for field, key in ((\'nonlinear_target_prediction\', \'target_actual\'), (\'linear_target_prediction\', \'target_predicted\')):\n            if old.vector_metrics(torch.from_numpy(arrays[\'jac_target_delta\']), torch.from_numpy(arrays[prefix+\'_\'+key])) != row[field]:\n                raise ValueError(\'Saved endpoint vector no longer agrees with numerical metrics\')\n        norm = float(np.linalg.norm(arrays[prefix+\'_requested_delta\'].astype(np.float64)))\n        realized_norm = float(np.linalg.norm(arrays[prefix+\'_realized_source_delta\'].astype(np.float64)))\n        refined_norm = float(np.linalg.norm(arrays[\'solver_delta_source\'].astype(np.float64)))\n        if (not math.isclose(norm, row[\'requested_source_norm\'], rel_tol=1e-12, abs_tol=1e-12)\n                or not math.isclose(realized_norm, row[\'realized_source_norm\'], rel_tol=1e-12, abs_tol=1e-12)\n                or not math.isclose(refined_norm, row[\'refined_source_norm\'], rel_tol=1e-12, abs_tol=1e-12)\n                or not math.isclose(abs(norm-refined_norm), row[\'norm_match_error\'], rel_tol=1e-10, abs_tol=1e-12)):\n            raise ValueError(\'Saved intervention vectors disagree with measured source norms\')\n    reconstruction = arrays[\'endpoint_refined_ridge_target_predicted\'].astype(np.float64)\n    damping = record[\'solver\'][\'lambda_squared_native\']\n    target = arrays[\'jac_target_delta\'].astype(np.float64)\n    residual = target-reconstruction-damping*arrays[\'solver_dual_solution\'].astype(np.float64)\n    norm = float(np.linalg.norm(target))\n    relative = float(np.linalg.norm(residual))/norm if norm else 0.\n    if not math.isclose(relative, record[\'solver\'][\'true_dual_relative_residual\'], rel_tol=1e-7, abs_tol=1e-11):\n        raise ValueError(\'Returned native dual/source do not reproduce the saved true residual\')\n    if not np.array_equal(arrays[\'solver_delta_source\'], arrays[\'endpoint_refined_ridge_requested_delta\']):\n        raise ValueError(\'Endpoint did not use the exact returned refined source vector\')\n\n\ndef stage_path(output, index, factor):\n    return Path(output)/\'requests\'/f\'b14b18_{index:03d}\'/factor_name(factor)\n\n\ndef write_checkpoint(output, record, arrays, plan_hash, hardware):\n    import numpy as np\n    target = stage_path(output, record[\'request\'][\'index\'], record[\'lambda_factor\'])\n    if target.exists():\n        raise ValueError(\'Complete stage already exists; no overwrite\')\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = Path(tempfile.mkdtemp(prefix=\'.\'+target.name+\'-\', dir=target.parent))\n    try:\n        np.savez_compressed(temporary/\'arrays.npz\', **arrays)\n        record = dict(record, schema=SCHEMA, state=\'complete\', plan_sha256=plan_hash,\n                      hardware=hardware, completed_utc=dt.datetime.now(dt.UTC).isoformat(),\n                      npz_sha256=_sha(temporary/\'arrays.npz\'))\n        record[\'technical_checks\'] = {\'state\': \'passed\', \'schema\': SCHEMA,\n                                     \'plan_sha256\': plan_hash, \'checks\': record[\'checks\']}\n        validate_record(record, plan_hash)\n        validate_arrays(record, arrays)\n        base.atomic_json(temporary/\'result.json\', record)\n        os.rename(temporary, target)\n    finally:\n        if temporary.exists():\n            shutil.rmtree(temporary)\n    return record\n\n\ndef verified(output, index, factor, plan_hash, item):\n    import numpy as np\n    folder = stage_path(output, index, factor)\n    if not folder.exists():\n        return None\n    record = json.loads((folder/\'result.json\').read_text())\n    validate_record(record, plan_hash, item, factor)\n    if _sha(folder/\'arrays.npz\') != record.get(\'npz_sha256\'):\n        raise ValueError(\'Checkpoint NPZ hash mismatch\')\n    with np.load(folder/\'arrays.npz\', allow_pickle=False) as saved:\n        validate_arrays(record, {key: saved[key] for key in saved.files})\n    return record\n\n\ndef write_partial(output, record, arrays, plan_hash):\n    import numpy as np\n    parent = Path(output)/\'partial\'/record[\'request\'][\'request_id\']/factor_name(record[\'lambda_factor\'])\n    parent.mkdir(parents=True, exist_ok=True)\n    target = parent/str(time.time_ns())\n    temporary = Path(tempfile.mkdtemp(prefix=\'.partial-\', dir=parent))\n    try:\n        np.savez_compressed(temporary/\'arrays.npz\', **arrays)\n        base.atomic_json(temporary/\'result.json\', dict(record, schema=SCHEMA, state=\'partial\',\n                         plan_sha256=plan_hash, npz_sha256=_sha(temporary/\'arrays.npz\'),\n                         resume_eligible=False, interpretation=\'Incomplete stage; not full evidence or a complete case\'))\n        os.rename(temporary, target)\n    finally:\n        if temporary.exists():\n            shutil.rmtree(temporary)\n    return target\n\n\ndef restore_resume(resume_zip, output, plan_hash, items):\n    if not resume_zip:\n        return\n    with zipfile.ZipFile(resume_zip) as bundle:\n        if bundle.testzip() is not None:\n            raise ValueError(\'Resume CRC failure\')\n        if (\'COMPACT_PACKAGE.json\' in bundle.namelist()\n                or json.loads(bundle.read(\'run_plan.json\')).get(\'schema\') != SCHEMA):\n            raise ValueError(\'Only a full matching refinement archive can resume\')\n        if hashlib.sha256(base.canonical(json.loads(bundle.read(\'run_plan.json\')))).hexdigest() != plan_hash:\n            raise ValueError(\'Resume plan differs\')\n        allowed = {f\'requests/b14b18_{index:03d}/{factor_name(factor)}/{name}\'\n                   for index, factor in stage_order() for name in (\'result.json\', \'arrays.npz\')}\n        request_names = [name for name in bundle.namelist() if name.startswith(\'requests/\') and not name.endswith(\'/\')]\n        if len(request_names) != len(set(request_names)) or set(request_names)-allowed:\n            raise ValueError(\'Unrecognized resume request path\')\n        for index, factor in stage_order():\n            folder = f\'requests/b14b18_{index:03d}/{factor_name(factor)}\'\n            names = [folder+\'/result.json\', folder+\'/arrays.npz\']\n            present = [name in request_names for name in names]\n            if not any(present):\n                continue\n            if not all(present):\n                raise ValueError(\'Partial file pair inside complete resume namespace\')\n            existing = verified(output, index, factor, plan_hash, items[index])\n            if existing is not None:\n                if any((Path(output)/name).read_bytes() != bundle.read(name) for name in names):\n                    raise ValueError(\'Resume conflicts with current complete checkpoint\')\n                continue\n            with tempfile.TemporaryDirectory(prefix=\'refine-resume-\') as directory:\n                staging = Path(directory)\n                target = stage_path(staging, index, factor)\n                target.mkdir(parents=True)\n                for name in names:\n                    (target/Path(name).name).write_bytes(bundle.read(name))\n                verified(staging, index, factor, plan_hash, items[index])\n                destination = stage_path(output, index, factor)\n                destination.parent.mkdir(parents=True, exist_ok=True)\n                copying = Path(tempfile.mkdtemp(prefix=\'.\'+destination.name+\'-resume-\', dir=destination.parent))\n                try:\n                    shutil.copytree(target, copying, dirs_exist_ok=True)\n                    os.rename(copying, destination)\n                finally:\n                    if copying.exists():\n                        shutil.rmtree(copying)\n\n\ndef archive(output):\n    output = Path(output)\n    target, temporary = output/\'b14_b18_refine_results.zip\', output/\'.b14_b18_refine_results.zip.tmp\'\n    with zipfile.ZipFile(temporary, \'w\', zipfile.ZIP_DEFLATED) as bundle:\n        for path in sorted(output.rglob(\'*\')):\n            if path.is_file() and path not in (target, temporary) and not any(part.startswith(\'.\') for part in path.relative_to(output).parts):\n                bundle.write(path, path.relative_to(output))\n    with zipfile.ZipFile(temporary) as bundle:\n        if bundle.testzip() is not None:\n            raise ValueError(\'Final archive CRC failure\')\n    os.replace(temporary, target)\n    return target\n\n\ndef run(hn, policy, refw, items_path, plan_path, refine_plan_path, anchors_path,\n        out_dir, cache_dir=None, resume_zip=None, offline_plan_only=False):\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    nonce = os.environ.get(\'H286_WRAPPER_ATTEMPT\')\n    status = {\'state\': \'running\', \'stage\': \'reference\', \'schema\': SCHEMA, \'completed_inputs\': 0,\n              \'completed_stages\': 0, \'wrapper_attempt_id\': nonce, \'confirmatory_eligible\': False}\n    metrics = {\'weights_loaded\': False, \'pretrained_model_run\': False, \'no_training\': True,\n               \'wrapper_attempt_id\': nonce, \'generation_count\': 0}\n    started = time.monotonic()\n    def save():\n        base.atomic_json(output/\'status.json\', status)\n        base.atomic_json(output/\'metrics.json\', metrics)\n    save()\n    try:\n        reference, stats, items, original, refinement, anchors = load_inputs(\n            hn, policy, refw, items_path, plan_path, refine_plan_path, anchors_path)\n        plan = plan_document(reference, stats, items, original, refinement, anchors)\n        plan_hash = hashlib.sha256(base.canonical(plan)).hexdigest()\n        existing = output/\'run_plan.json\'\n        if existing.exists() and existing.read_bytes() != base.canonical(plan)+b\'\\n\':\n            raise ValueError(\'Different bound refinement plan; no output overwrite\')\n        if not existing.exists():\n            base.atomic_json(existing, plan)\n        if offline_plan_only:\n            status.update(state=\'complete\', stage=\'offline_plan_only\', plan_sha256=plan_hash, no_model_executed=True)\n            return status\n        restore_resume(resume_zip, output, plan_hash, items)\n        done = {(index, factor): verified(output, index, factor, plan_hash, items[index])\n                for index, factor in stage_order()}\n        def progress():\n            status[\'completed_stages\'] = sum(value is not None for value in done.values())\n            status[\'completed_inputs\'] = sum(all(done[index, factor] is not None for factor in FACTORS) for index in INDICES)\n            status[\'completed_primary_stages\'] = sum(done[index, 1.] is not None for index in INDICES)\n            retained = [record for record in done.values() if record is not None]\n            metrics[\'completed_stages\'] = len(retained)\n            metrics[\'completed_inputs\'] = status[\'completed_inputs\']\n            metrics[\'solver_count\'] = len(retained)\n            metrics[\'converged_solver_count\'] = sum(record[\'solver\'][\'converged\'] for record in retained)\n            metrics[\'all_solvers_converged\'] = len(retained) == 12 and metrics[\'converged_solver_count\'] == 12\n        progress()\n        if all(value is not None for value in done.values()):\n            status.update(state=\'complete\', stage=\'complete\', resumed_without_gpu=True)\n            metrics[\'retained_pretrained_model_run\'] = True\n            metrics[\'retained_technical_checks\'] = [record[\'technical_checks\'] for record in done.values()]\n            return status\n        status[\'stage\'] = \'pinned_model_load\'\n        save()\n        def loaded(hardware):\n            hardware = dict(hardware, attention=\'sdpa/MATH native local derivatives and endpoint measurements\')\n            metrics.update(weights_loaded=True, hardware=hardware)\n            save()\n        engine = old.initialize_engine(reference, Path(cache_dir or \'/content/h286_model_cache\'), loaded)\n        engine[\'hardware\'] = dict(engine[\'hardware\'], attention=\'sdpa/MATH native local derivatives and endpoint measurements\')\n        def model_ran():\n            metrics[\'pretrained_model_run\'] = True\n            save()\n        engine[\'on_model_run\'] = model_ran\n        compute_started = time.monotonic()\n        deadline = compute_started+BUDGET_SECONDS\n        stop = lambda: time.monotonic() >= deadline\n        anchor_by_index = {case[\'index\']: case for case in anchors[\'cases\']}\n        measured = []\n        for index, factor in stage_order():\n            if done[index, factor] is not None:\n                continue\n            if stop():\n                status.update(state=\'paused\', stage=\'safe_stage_checkpoint\', reason=\'20-minute soft compute budget reached\')\n                break\n            status.update(stage=\'same_lambda_convergence\' if factor == 1. else \'fixed_damping_ablation\',\n                          current_input=f\'b14b18_{index:03d}\', current_factor=factor)\n            save()\n            print(f\'Stage {status["completed_stages"]+1}/12: Fall {index}, Lambda-Faktor {factor:g}; feste Kontrollen.\', flush=True)\n            request = old.prepare_request(engine[\'tokenizer\'], index, items[index], engine[\'model\'].config.vocab_size)\n            metrics[\'model_forward_attempted\'] = True\n            save()\n            record, arrays = compute_stage(engine, reference, stats, request, anchor_by_index[index], factor, stop_requested=stop)\n            metrics[\'pretrained_model_run\'] = True\n            if record[\'state\'] != \'complete\':\n                partial = write_partial(output, record, arrays, plan_hash)\n                status.update(state=\'paused\', stage=\'partial_stage_checkpoint\',\n                              reason=record.get(\'stop_reason\', \'incomplete stage\'), partial_path=str(partial.relative_to(output)))\n                break\n            done[index, factor] = write_checkpoint(output, record, arrays, plan_hash, engine[\'hardware\'])\n            measured.append(record[\'timing\'][\'total_seconds\'])\n            progress()\n            remaining = 12-status[\'completed_stages\']\n            status.update(compute_elapsed_seconds=time.monotonic()-compute_started,\n                          estimated_remaining_seconds=sum(measured)/len(measured)*remaining)\n            save()\n            print(f\'Gespeichert {status["completed_stages"]}/12 Stufen; {status["completed_inputs"]}/4 Fälle vollständig; \'\n                  f\'letzte Rechnung {measured[-1]:.2f}s; Restschätzung {status["estimated_remaining_seconds"]/60:.1f}min.\', flush=True)\n        else:\n            status.update(state=\'complete\', stage=\'complete\')\n        metrics[\'compute_elapsed_seconds\'] = time.monotonic()-compute_started\n        metrics[\'completed_stages\'] = status[\'completed_stages\']\n        metrics[\'completed_inputs\'] = status[\'completed_inputs\']\n    except BaseException as exc:  # noqa: BLE001 — preserve interruption/OOM diagnostics.\n        status.update(state=\'failed\', error_type=type(exc).__name__, error=str(exc))\n        (output/\'traceback.txt\').write_text(traceback.format_exc())\n        print(\'HARD FAILURE: \'+str(exc), flush=True)\n    finally:\n        metrics[\'elapsed_seconds\'] = time.monotonic()-started\n        save()\n        archive(output)\n    return status\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    for flag in (\'hn-set\', \'policy\', \'refw\', \'items\', \'plan\', \'refine-plan\', \'anchors\', \'out-dir\'):\n        parser.add_argument(\'--\'+flag, required=True)\n    parser.add_argument(\'--cache-dir\')\n    parser.add_argument(\'--resume-zip\')\n    parser.add_argument(\'--offline-plan-only\', action=\'store_true\')\n    arguments = parser.parse_args()\n    terminal = run(arguments.hn_set, arguments.policy, arguments.refw, arguments.items,\n                   arguments.plan, arguments.refine_plan, arguments.anchors, arguments.out_dir,\n                   arguments.cache_dir, arguments.resume_zip, arguments.offline_plan_only)\n    raise SystemExit(0 if terminal[\'state\'] in (\'complete\', \'paused\') else 1)\n', 'attention_probe/transport_refinement.py': '"""Bounded refinement of a fixed, damped, matrix-free local transport solve.\n\nSolve (J J.T + lambda**2 I) y = b, then return x = J.T y.  This module\nnever changes damping, forms a Jacobian, or selects an answer-dependent arm.\nConvergence concerns the damped dual equation, not exact reconstruction of b.\n\nThe callbacks must describe one fixed linear map and its adjoint. Vectors use\nnative FP32/FP64; reductions and the final residual diagnostic use FP64. A\nrecursive CG tolerance crossing is only a candidate: the true residual of the\nnative, rescaled iterate must confirm it. Reliable residual replacement restarts\nCG if recurrence drift exceeds 10% or a candidate convergence is false.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom collections.abc import Callable\nfrom dataclasses import dataclass\nfrom numbers import Integral\nfrom typing import Any\n\nimport torch\n\nfrom .local_transport import _norm64, _positive_real\n\nMAX_REFINEMENT_ITERATIONS = 512\n\n\n@dataclass(frozen=True)\nclass TransportRefinementResult:\n    """Detached native source/dual vectors and JSON-safe measured diagnostics."""\n\n    delta_source: torch.Tensor\n    dual_solution: torch.Tensor\n    diagnostics: dict[str, Any]\n\n\ndef refine_transport(\n    jvp: Callable[[torch.Tensor], torch.Tensor],\n    vjp: Callable[[torch.Tensor], torch.Tensor],\n    target: torch.Tensor,\n    lambda_: float,\n    *,\n    max_iter: int = MAX_REFINEMENT_ITERATIONS,\n    rtol: float = 1e-4,\n    stop_requested: Callable[[], bool] | None = None,\n    true_residual_interval: int = 16,\n) -> TransportRefinementResult:\n    """Refine a fixed Tikhonov problem without silently accepting false CG stops.\n\n    ``stop_requested`` is a cooperative deadline/cancellation callback, checked\n    before each iteration. It cannot preempt an executing operator. Finalizing\n    a stopped iterate may require one VJP and one JVP after that callback fires;\n    this bounded accounting cost supplies the actual final residual. The result\n    records ``budget_exhausted`` even if that last residual happens to pass.\n\n    Internally the RHS has unit norm. At each true check the dual iterate is\n    rescaled and rounded to native dtype, x=VJP(y) and JVP(x) are evaluated, and\n    b-JVP(x)-lambda_native_squared*y is reduced in FP64. Returned y and x are\n    exactly these checked native tensors. No unrounded surrogate is certified.\n    If needed, a restart uses this true residual divided by the original RHS\n    norm. This handles the original-RHS rounding introduced by normalization.\n\n    Invalid/nonfinite callback tensors raise. Finite iteration limits, requested\n    stops and nonpositive curvature return a flagged last finite iterate. There\n    are at most max_iter CG products plus max_iter+1 true residual evaluations.\n    """\n    if not callable(jvp) or not callable(vjp):\n        raise TypeError("jvp and vjp must be callables for one fixed local map")\n    if not isinstance(target, torch.Tensor):\n        raise TypeError("target must be a torch.Tensor")\n    if target.dtype not in (torch.float32, torch.float64):\n        raise ValueError("target must use native FP32 or FP64")\n    if target.numel() == 0 or not bool(torch.isfinite(target).all().item()):\n        raise ValueError("target must be nonempty and finite")\n    for value, name in (\n        (max_iter, "max_iter"),\n        (true_residual_interval, "true_residual_interval"),\n    ):\n        if isinstance(value, bool) or not isinstance(value, Integral):\n            raise TypeError(f"{name} must be an integer")\n        if not 1 <= value <= MAX_REFINEMENT_ITERATIONS:\n            raise ValueError(f"{name} must be between 1 and 512")\n    if stop_requested is not None and not callable(stop_requested):\n        raise TypeError("stop_requested must be callable or None")\n    lam = _positive_real(lambda_, "lambda_")\n    tolerance = _positive_real(rtol, "rtol", below_one=True)\n    squared = lam * lam\n    if not math.isfinite(squared):\n        raise ValueError("lambda_^2 must be finite")\n    b = target.detach().clone()\n    damping = torch.tensor(squared, device=b.device, dtype=b.dtype)\n    damping_value = float(damping.item())\n    if not math.isfinite(damping_value) or damping_value <= 0:\n        raise ValueError("lambda_^2 overflows or underflows native dtype")\n    target_norm = _norm64(b)\n    source_shape: tuple[int, ...] | None = None\n    counts = {\n        "jvp_calls": 0,\n        "vjp_calls": 0,\n        "cg_operator_calls": 0,\n        "true_residual_evaluations": 0,\n        "zero_jvp_shortcuts": 0,\n        "stop_callback_checks": 0,\n    }\n\n    def checked(value, name, shape=None):\n        if not isinstance(value, torch.Tensor):\n            raise TypeError(f"{name} must return a torch.Tensor")\n        if value.dtype != b.dtype or value.device != b.device:\n            raise ValueError(f"{name} must preserve native dtype/device")\n        if value.numel() == 0 or (shape is not None and tuple(value.shape) != shape):\n            raise ValueError(f"{name} returned an unexpected shape")\n        if not bool(torch.isfinite(value).all().item()):\n            raise ValueError(f"{name} returned nonfinite values")\n        return value.detach().clone()\n\n    def transpose(value):\n        nonlocal source_shape\n        counts["vjp_calls"] += 1\n        out = checked(vjp(value.detach().clone()), "vjp", source_shape)\n        if source_shape is None:\n            source_shape = tuple(out.shape)\n        return out\n\n    def forward(value):\n        if not bool(torch.count_nonzero(value).item()):\n            counts["zero_jvp_shortcuts"] += 1\n            return torch.zeros_like(b)\n        counts["jvp_calls"] += 1\n        return checked(jvp(value.detach().clone()), "jvp", tuple(b.shape))\n\n    def dual_product(value):\n        counts["cg_operator_calls"] += 1\n        return checked(\n            forward(transpose(value)) + damping * value, "dual operator", tuple(b.shape)\n        )\n\n    def inner(left, right):\n        value = float(\n            torch.dot(left.reshape(-1).double(), right.reshape(-1).double()).item()\n        )\n        if not math.isfinite(value):\n            raise FloatingPointError("Nonfinite FP64 CG reduction")\n        return value\n\n    true_history: list[dict[str, Any]] = []\n    restart_history: list[dict[str, Any]] = []\n    recursive_history = [0.0 if target_norm == 0 else 1.0]\n    iterations = 0\n    false_convergence_count = 0\n    solution = torch.zeros_like(b)\n    cached = None\n\n    def evaluate(solution, iteration, reason, recursive_relative):\n        counts["true_residual_evaluations"] += 1\n        dual = checked(\n            (solution.double() * target_norm).to(b.dtype),\n            "rescaled dual",\n            tuple(b.shape),\n        )\n        source = transpose(dual)\n        if not bool(torch.count_nonzero(dual).item()) and bool(\n            torch.count_nonzero(source).item()\n        ):\n            raise ValueError("vjp(0) must be zero for a fixed linearized map")\n        reconstruction = forward(source)\n        residual64 = (\n            b.double() - reconstruction.double() - damping_value * dual.double()\n        )\n        residual_norm = _norm64(residual64)\n        relative = residual_norm / target_norm if target_norm else 0.0\n        if not math.isfinite(relative):\n            raise FloatingPointError("Nonfinite true relative residual")\n        true_history.append(\n            {\n                "iteration": iteration,\n                "reason": reason,\n                "recursive_relative_residual": recursive_relative,\n                "true_relative_residual": relative,\n                "true_residual_norm": residual_norm,\n            }\n        )\n        return {\n            "dual": dual,\n            "source": source,\n            "reconstruction": reconstruction,\n            "residual64": residual64,\n            "relative": relative,\n            "residual_norm": residual_norm,\n        }\n\n    stop_reason = "zero_target" if target_norm == 0 else "max_iter"\n    if target_norm:\n        rhs = (b.double() / target_norm).to(b.dtype)\n        rhs_norm = _norm64(rhs)\n        residual = rhs.clone()\n        direction = residual.clone()\n        rr = inner(residual, residual)\n        for _ in range(int(max_iter)):\n            if stop_requested is not None:\n                counts["stop_callback_checks"] += 1\n                requested = stop_requested()\n                if not isinstance(requested, bool):\n                    raise TypeError("stop_requested must return a bool")\n                if requested:\n                    stop_reason = "budget_exhausted"\n                    break\n            product = dual_product(direction)\n            curvature = inner(direction, product)\n            if curvature <= 0:\n                stop_reason = "nonpositive_curvature"\n                break\n            alpha = rr / curvature\n            if not math.isfinite(alpha):\n                stop_reason = "nonfinite_step"\n                break\n            next_solution = solution + alpha * direction\n            next_residual = residual - alpha * product\n            if not bool(torch.isfinite(next_solution).all().item()) or not bool(\n                torch.isfinite(next_residual).all().item()\n            ):\n                stop_reason = "nonfinite_iterate"\n                break\n            next_rr = inner(next_residual, next_residual)\n            solution, residual = next_solution, next_residual\n            cached = None\n            iterations += 1\n            relative = math.sqrt(next_rr) / rhs_norm\n            recursive_history.append(relative)\n            candidate = relative <= tolerance\n            if candidate or iterations % int(true_residual_interval) == 0:\n                cached = evaluate(\n                    solution,\n                    iterations,\n                    "recursive_candidate" if candidate else "periodic",\n                    relative,\n                )\n                if cached["relative"] <= tolerance:\n                    stop_reason = "converged"\n                    break\n                true_normalized = (cached["residual64"] / target_norm).to(b.dtype)\n                drift = _norm64(true_normalized - residual)\n                unreliable = drift > 0.1 * _norm64(true_normalized)\n                if candidate or unreliable:\n                    if candidate:\n                        false_convergence_count += 1\n                    restart_history.append(\n                        {\n                            "iteration": iterations,\n                            "reason": "false_recursive_convergence"\n                            if candidate\n                            else "residual_drift",\n                            "residual_drift_norm": drift,\n                            "true_relative_residual": cached["relative"],\n                        }\n                    )\n                    residual = true_normalized.detach().clone()\n                    direction = residual.clone()\n                    rr = inner(residual, residual)\n                    continue\n            beta = next_rr / rr\n            if not math.isfinite(beta):\n                stop_reason = "nonfinite_step"\n                break\n            next_direction = residual + beta * direction\n            if not bool(torch.isfinite(next_direction).all().item()):\n                stop_reason = "nonfinite_iterate"\n                break\n            direction, rr = next_direction, next_rr\n\n    if cached is None:\n        cached = evaluate(solution, iterations, "final", recursive_history[-1])\n    source = cached["source"]\n    dual = cached["dual"]\n    reconstruction_error = _norm64(cached["reconstruction"].double() - b.double())\n    reconstruction_relative = reconstruction_error / target_norm if target_norm else 0.0\n    source_norm = _norm64(source)\n    objective = (\n        reconstruction_relative**2 + damping_value * (source_norm / target_norm) ** 2\n        if target_norm\n        else 0.0\n    )\n    if not all(math.isfinite(value) for value in (reconstruction_relative, objective)):\n        raise FloatingPointError("Nonfinite final transport diagnostics")\n    breakdown = (\n        stop_reason\n        if stop_reason\n        in ("nonpositive_curvature", "nonfinite_step", "nonfinite_iterate")\n        else None\n    )\n    converged = cached["relative"] <= tolerance and breakdown is None\n    if stop_reason == "budget_exhausted":\n        status = "stopped"\n    elif breakdown:\n        status = "breakdown"\n    elif converged:\n        status = "converged"\n        if stop_reason != "zero_target":\n            stop_reason = "converged"\n    else:\n        status = "not_converged"\n    diagnostics = dict(\n        schema="local_transport_refinement_v1",\n        method="reliable_matrix_free_dual_cg",\n        status=status,\n        converged=bool(converged),\n        stop_reason=stop_reason,\n        termination=stop_reason,\n        breakdown=breakdown,\n        iterations=iterations,\n        max_iter=int(max_iter),\n        rtol=tolerance,\n        true_residual_interval=int(true_residual_interval),\n        true_dual_relative_residual=cached["relative"],\n        dual_relative_residual=cached["relative"],\n        true_dual_residual_norm=cached["residual_norm"],\n        dual_residual_norm=cached["residual_norm"],\n        target_norm=target_norm,\n        source_norm=source_norm,\n        dual_solution_norm=_norm64(dual),\n        linear_reconstruction_error_norm=reconstruction_error,\n        linear_reconstruction_relative_error=reconstruction_relative,\n        normalized_tikhonov_objective=objective,\n        recursive_relative_residual_history=recursive_history,\n        true_residual_history=true_history,\n        restart_history=restart_history,\n        restart_count=len(restart_history),\n        false_convergence_count=false_convergence_count,\n        rhs_normalization="unit L2 internally; native rescaled dual/source checked against original RHS",\n        final_residual_scope="FP64 reduction of b - JVP(returned source) - native_lambda_squared * returned dual",\n        finalization_budget="At most one VJP and one JVP after a stop request; executing operators cannot be preempted",\n        zero_target=target_norm == 0,\n        target_shape=list(b.shape),\n        source_shape=list(source.shape),\n        dtype=str(b.dtype),\n        device=str(b.device),\n        full_jacobian_formed=False,\n        singular_values_computed=False,\n        convergence_scope="damped local linear dual equation; not undamped reconstruction or nonlinear replay",\n        **counts,\n    )\n    diagnostics["lambda"] = lam\n    diagnostics["lambda_squared_native"] = damping_value\n    return TransportRefinementResult(source.detach(), dual.detach(), diagnostics)\n'}
MODULE_SHA256 = {'attention_probe/__init__.py': 'de626db34130b6d517d0dc776df8eff3b794bec5d7c9ba70157abe016f422705', 'attention_probe/reference.py': 'e89bd465d0e962e840ba589280ae89b84dd74f6fda1a459f4c76826ef4bd51e3', 'attention_probe/fixed_refw.py': 'b5cb362889f473f8b902454b4e22f4a615a54ed0a2bac0d7b0cc4c80e3c778e3', 'attention_probe/interventions.py': '1c7afb046108e2d7a214e1849198574128acf2c3e300888cdb6cc47e0e086561', 'attention_probe/pilot_items.json': '12a3cba862185b9f25ca39ea3ea49bd3bb6fc5f248c12dbbe02bcc8836663fb3', 'qwen3_h286_runtime.py': 'e30b9da80bd9370eb5de04245ad43b8c0ac0e292611ba32d1a8721b28e1e4312', 'attention_probe/answer_validation.py': '9bac92b5e286388d2f1e8cbf71b9fc181840de8b60f0988360eb9e5692a1ba3c', 'attention_probe/native_jacobian.py': 'e9eda255debe4f234041c7a378b47277994d4392bb2dccc847488867cf422e8e', 'attention_probe/local_transport.py': 'e6c59988ffaa314ddf8af8f58977998ae4e5baaaa30847b644b69ef7cbef1808', 'attention_probe/result_bundle.py': '1860c056df18124b26a990f0787df0b56be799a6d05fde319c1bded78d142774', 'qwen3_b14_b18_runtime.py': 'c04586073a54fed7a257db7e27fed48209abab8b151914a74439d154e1fad6af', 'qwen3_b14_b18_refine_runtime.py': '6eb29e8c54236c7389b6de2432a25791577705a5b653acae2d4bb9795e0d60ea', 'attention_probe/transport_refinement.py': '6ffd521e3d7f9b7236e65663245999a82faf5a30ddd9536e0828b0b6c51ffb23'}


In [ ]:
import base64
from collections import deque
import datetime
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import traceback
import zipfile
import uuid

wrapper_attempt_id = uuid.uuid4().hex
base = Path('/content') if Path('/content').is_dir() else Path.cwd()
binding_fingerprint = hashlib.sha256(json.dumps({'modules': MODULE_SHA256, 'references': REFERENCE_SHA256},
                                                sort_keys=True).encode()).hexdigest()[:12]
work = (Path(RUN_DIRECTORY).expanduser() if RUN_DIRECTORY
        else base / ('b14_b18_refinement_' + binding_fingerprint))
output = work / 'output'
output.mkdir(parents=True, exist_ok=True)
failure = None
helper_started = False
helper_status = None
return_code = None
status = {'state': 'running', 'stage': 'gpu_preflight', 'purpose': 'original_qwen3_b14_b18_refinement',
          'pretrained_weights_loaded': False, 'inference': 'exploratory (post hoc)'}

def write_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + chr(10))

def run_command(command, logname, env=None):
    process = None
    tail = deque(maxlen=10)
    shown = 0
    try:
        with (output / logname).open('a', encoding='utf-8') as log:
            process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                log.write(line); log.flush()
                short = line.strip().replace(chr(13), ' ')[:500]
                tail.append(short)
                if logname == 'runtime.log' and shown < 120 and short.startswith(
                        ('Request ', 'Gespeichert ', 'Stage ', 'Status:', 'Status ', 'CHECK ', 'HARD FAILURE:')):
                    print(short, flush=True); shown += 1
            code = process.wait()
        if code:
            print('Kindprozess fehlgeschlagen; vollständiges Log im ZIP. Letzte Zeilen:', flush=True)
            for short in tail:
                print(short, flush=True)
            raise subprocess.CalledProcessError(code, command)
        return code
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
        raise

try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Keine CUDA-GPU. A100, L4 oder B200 auswählen; kein Warten.')
    capability = tuple(torch.cuda.get_device_capability(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    if capability[0] < 8 or free_bytes < MIN_FREE_VRAM_GIB * 1024**3:
        raise RuntimeError('Ampere-oder-neuere CUDA-GPU mit mindestens 12 GiB freiem Speicher erforderlich.')
    torch_before = importlib.metadata.version('torch')
    environment = {'python': sys.version, 'torch': torch_before, 'torch_cuda': torch.version.cuda,
                   'gpu': torch.cuda.get_device_name(0), 'capability': list(capability),
                   'free_vram_gib': free_bytes / 1024**3, 'total_vram_gib': total_bytes / 1024**3,
                   'transformers_required': TRANSFORMERS_VERSION, 'accelerate_required': ACCELERATE_VERSION,
                   'execution_dtype': 'float32', 'tf32': False}
    write_json(output / 'environment_before.json', environment)
    print('GPU:', environment['gpu'], '| frei:', round(environment['free_vram_gib'], 2), 'GiB')
    print('Torch/CUDA bleibt erhalten:', torch_before, torch.version.cuda)
    status['stage'] = 'reference_validation'
    inputs = work / 'inputs'; inputs.mkdir(exist_ok=True)
    for name, expected in REFERENCE_SHA256.items():
        if name in REFERENCE_FILES_B64:
            raw = base64.b64decode(REFERENCE_FILES_B64[name], validate=True)
        elif REFERENCE_DIRECTORY:
            raw = (Path(REFERENCE_DIRECTORY).expanduser() / name).read_bytes()
        else:
            raise RuntimeError('Öffentliches Template ohne private Referenz. Die bereitgestellte private Nutzdatei verwenden.')
        if hashlib.sha256(raw).hexdigest() != expected:
            raise ValueError('Originalreferenz-Prüfsumme weicht ab: ' + name)
        target = inputs / name
        if target.exists() and target.read_bytes() != raw:
            raise ValueError('Vorhandene Referenz gehört zu einer anderen Auswahl; Ergebnisse nicht mischen.')
        target.write_bytes(raw)
    binding = {'sha256': MODULE_SHA256, 'reference_sha256': REFERENCE_SHA256,
               'model': 'Qwen/Qwen3-1.7B', 'revision': '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'}
    binding_path = output / 'embedded_source.json'
    if binding_path.exists() and json.loads(binding_path.read_text()) != binding:
        raise ValueError('Vorhandener Lauf gehört zu anderem Quellcode oder Referenzen; Ergebnisse nicht mischen.')
    code = work / 'code'; code.mkdir(exist_ok=True)
    for name, source in MODULE_SOURCES.items():
        if hashlib.sha256(source.encode()).hexdigest() != MODULE_SHA256[name]:
            raise ValueError('Eingebetteter Quellcode weicht ab: ' + name)
        path = code / name; path.parent.mkdir(parents=True, exist_ok=True); path.write_text(source)
    write_json(binding_path, binding)
    status['stage'] = 'dependencies'
    constraint = work / 'preserve_torch.txt'
    constraint.write_text('torch==' + torch_before + chr(10))
    print('Exakte Transformers-/Accelerate-Version installieren; GPU-Torch bleibt festgelegt.', flush=True)
    child_env = dict(os.environ, HF_HUB_DISABLE_PROGRESS_BARS='1', TQDM_DISABLE='1',
                     TOKENIZERS_PARALLELISM='false')
    run_command([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--upgrade-strategy', 'only-if-needed',
                 '--constraint', str(constraint), 'transformers==' + TRANSFORMERS_VERSION,
                 'accelerate==' + ACCELERATE_VERSION], 'install.log', child_env)
    installed = {name: importlib.metadata.version(name) for name in ('torch', 'transformers', 'accelerate')}
    if installed != {'torch': torch_before, 'transformers': TRANSFORMERS_VERSION, 'accelerate': ACCELERATE_VERSION}:
        raise RuntimeError('Paketversionen weichen ab; keine Modellgewichte angefordert.')
    write_json(output / 'environment_installed.json', installed)
    status['stage'] = 'runtime'
    helper_started = True
    status['pretrained_weights_loaded'] = False
    env = dict(child_env, PYTHONPATH=str(code), H286_WRAPPER_ATTEMPT=wrapper_attempt_id)
    command = [sys.executable, str(code / 'qwen3_b14_b18_refine_runtime.py'), '--hn-set', str(inputs / 'hn_set.json'),
               '--policy', str(inputs / 'htq_policy.json'), '--refw', str(inputs / 'refw_h286.json'),
               '--items', str(inputs / 'b14_b18_items.json'),
               '--plan', str(inputs / 'b14_b18_plan.json'),
               '--refine-plan', str(inputs / 'b14_b18_refine_plan.json'),
               '--anchors', str(inputs / 'b14_b18_refine_anchors.json'),
               '--out-dir', str(output), '--cache-dir', str(base / 'h286_model_cache')]
    if RESUME_ZIP:
        command += ['--resume-zip', str(Path(RESUME_ZIP).expanduser())]
    print('Frischer Python-Prozess: Original-FP32-Modell, feste H286/REF-W, B14→B18: vier feste Fälle, Konvergenz und Dämpfung.', flush=True)
    return_code = run_command(command, 'runtime.log', env)
    helper_status = json.loads((output / 'status.json').read_text())
    if helper_status.get('state') not in ('complete', 'paused'):
        raise RuntimeError('Runtime hat keinen vollständigen oder bewusst pausierten Abschluss gemeldet.')
    status.update(helper_status, helper_return_code=return_code, wrapper_verified=True)
except BaseException as error:
    failure = error
    status.update(state='failed', error_type=type(error).__name__, error=str(error), wrapper_verified=False)
    (output / 'wrapper_diagnostic.txt').write_text(traceback.format_exc())
finally:
    # Keep the child's original status for diagnosis, then publish one authoritative final status.
    child_path = output / 'status.json'
    if helper_started and child_path.exists():
        try:
            helper_status = json.loads(child_path.read_text())
            write_json(output / 'helper_status.json', helper_status)
            if failure is not None:
                status['helper_state'] = helper_status.get('state')
                status['pretrained_weights_loaded'] = helper_status.get('pretrained_weights_loaded')
        except Exception:
            status['helper_state'] = 'unreadable'
    # Only this attempt's metrics can establish whether its model weights loaded.
    # Stable work folders may contain metrics from an earlier successful attempt.
    if helper_started:
        status['pretrained_weights_loaded'] = None
        status['weights_loaded_evidence'] = 'missing_unreadable_or_stale_child_metrics'
        try:
            child_metrics = json.loads((output / 'metrics.json').read_text())
            loaded = child_metrics.get('weights_loaded')
            if (child_metrics.get('wrapper_attempt_id') == wrapper_attempt_id
                    and type(loaded) is bool):
                status['pretrained_weights_loaded'] = loaded
                status['weights_loaded_evidence'] = 'fresh_child_metrics'
        except (OSError, ValueError, TypeError, AttributeError):
            pass
    status['wrapper_attempt_id'] = wrapper_attempt_id
    status['written_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    write_json(output / 'status.json', status)
    members = {str(path.relative_to(output)): hashlib.sha256(path.read_bytes()).hexdigest()
               for path in sorted(output.rglob('*')) if path.is_file()
               and path.name not in ('WRAPPER_MANIFEST.json',) and path.suffix not in ('.zip', '.tmp')
               and not any(part.startswith('.') for part in path.relative_to(output).parts)}
    write_json(output / 'WRAPPER_MANIFEST.json', members)
    target = base / ('b14_b18_refine_diagnostic.zip' if failure is not None else 'b14_b18_refine_results.zip')
    temporary = target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(output.rglob('*')):
            if (path.is_file() and path.suffix not in ('.zip', '.tmp')
                    and not any(part.startswith('.') for part in path.relative_to(output).parts)):
                archive.write(path, str(path.relative_to(output)))
    with zipfile.ZipFile(temporary) as archive:
        if archive.testzip() is not None:
            raise RuntimeError('Ergebnis-ZIP ist beschädigt.')
    os.replace(temporary, target)
    raw_target = None
    if helper_started:
        # Compact analysis retains every state/derivative array. Full logit
        # vectors and exact resume checkpoints stay in a separate RAW archive.
        try:
            import importlib.util
            bundle_path = code / 'attention_probe/result_bundle.py'
            expected_bundle = MODULE_SHA256['attention_probe/result_bundle.py']
            if hashlib.sha256(bundle_path.read_bytes()).hexdigest() != expected_bundle:
                raise ValueError('Result-export source SHA mismatch')
            bundle_spec = importlib.util.spec_from_file_location('b14_b18_result_export', bundle_path)
            bundle_module = importlib.util.module_from_spec(bundle_spec)
            bundle_spec.loader.exec_module(bundle_module)
            raw_target = target.with_name(target.stem + '_RAW.zip')
            os.replace(target, raw_target)
            compact_info = bundle_module.build_compact_result(raw_target, target)
            print('Kompakte Auswertung:', target, '| Rohdaten ebenfalls sichern:', raw_target, flush=True)
        except Exception as compact_error:
            print('Kompaktexport nicht verfügbar; vollständiges Original-ZIP verwenden:', str(compact_error)[:300], flush=True)
            if raw_target is not None and raw_target.is_file():
                os.replace(raw_target, target)
            raw_target = None
    print('Status:', status['state'], '| ZIP:', target, flush=True)
    try:
        from google.colab import files
        files.download(str(target))
        if raw_target is not None:
            files.download(str(raw_target))
    except Exception:
        try:
            from IPython.display import FileLink, display
            display(FileLink(os.path.relpath(target, Path.cwd())))
            if raw_target is not None:
                display(FileLink(os.path.relpath(raw_target, Path.cwd())))
        except Exception:
            print('ZIP aus diesem Verzeichnis herunterladen:', target)
if failure is not None:
    raise RuntimeError('Versuch angehalten; das Diagnose-ZIP enthält den tatsächlichen Fehler.') from None
if status['state'] == 'paused':
    print('Bewusst pausierter Checkpoint; kein abgeschlossener Versuch und kein PASS.')
else:
    print('B14→B18-Folgelauf abgeschlossen. Konvergenz und Rekonstruktion getrennt auswerten.')


Kompakte Auswertung und vollständige Rohdaten sichern; Solver-Konvergenz, Dämpfung und Rekonstruktionsfehler getrennt beurteilen.
